# N.E.U.R.O.N. (Neuroimaging and Event-based Unified Risk Outcomes Network)
## Forecasting the Cognitive Clock: Survival Modeling for Alzheimer's Progression

---

## Overview

This notebook implements a multimodal survival analysis pipeline to predict Alzheimer's disease
progression using longitudinal clinical data from the Alzheimer's Disease Neuroimaging Initiative (ADNI).
Rather than framing progression as a binary classification problem, we model **time-to-event** outcomes
directly using survival analysis methods, preserving the censoring structure of the data, and producing
interpretable risk rankings over continuous time.

**Two cohorts are modeled:**
- **MCI → AD** (primary): subjects diagnosed with Mild Cognitive Impairment at baseline
- **CN → MCI or AD** (secondary): cognitively normal subjects at baseline

**Four model families are trained and compared:**

| Model | Family | Key Characteristic |
|-------|--------|--------------------|
| Cox PH (elastic net) | Linear | Interpretable log-hazard ratios; L1/L2 regularization |
| GBSA | Tree-based | Non-linear interactions; no feature scaling required |
| Weibull AFT | Parametric | Accelerated failure time; predicts time directly |
| DeepSurv | Neural | Non-linear Cox via MLP; captures complex feature interactions |


**Primary evaluation metric:** IPCW Antolini time-dependent concordance index (C-td), which
extends Harrell's C to handle time-varying predictions and adjusts for censoring bias via
Inverse Probability of Censoring Weighting. A C-td of 0.5 is random; 1.0 is perfect ranking.

**Pipeline stages:**

| # | Stage | Description |
|---|-------|-------------|
| 1 | Setup | Install dependencies, configure paths and constants |
| 2 | Data Preparation | Load ADNIMERGE CSV, harmonize diagnoses, construct survival labels |
| 3 | Harmonization & Imputation | ComBat MRI correction; three-tier imputation strategy |
| 4 | Feature Engineering | ICV-normalized MRI, composites |
| 5 | Evaluation Framework | C-td, horizon AUC, CSF imputation |
| 6 | Survival Models | Cox PH, GBSA, Weibull AFT, DeepSurv |
| 7 | Results | Model comparison table, KM risk-quartile plots |
| 8 | Key Findings | Cross-model summary and recommendations |
| 9 | Conformal Prediction | Fixed-width CP, inversion sensitivity, within-interval bias |
| 10 | CSD | Adaptive-width intervals via Conformalized Survival Distribution |
| 11 | Split Ratio Comparison | 70/20/10 vs. 80/10/10: q_hat, width, and coverage |
| 12 | Longitudinal CP | Landmark analysis: covariates up to prediction time, visit-count sensitivity |
| 13 | IPCW Calibration | Censoring-informativeness diagnostic, IPCW-weighted CP, interval-shape check |

---

> **Reproducibility:** Set `RETRAIN = True` to train all models from scratch.  
> Set `RETRAIN = False` to load saved checkpoints from `checkpoints/` and skip training.


**Paper version (Nov 2026).** Preprocessing is fit on training data only: the calibration and evaluation subjects are fixed in Section 3.0 before any preprocessing, and ComBat (3.2), the z-score constants (4.1), and MICE (4.2) are fit on the MCI training subjects (`FIT_SET = 'mci_train'`) and then applied to everyone. The subjects in each split are the same as before. New cells for the paper: 4.4 (Table 1, baseline characteristics), 10.8 (Table 2 numbers and Figure 4), 10.9 (Figure 5), 10.10 (re-centered fixed width, IPCW-weighted pi, grid-maximum check, widths, full CiPOT), 12.6 (landmark table), 13.6 (Table 3 with CiPOT), and 14.1b (collects the new numbers).

## Run configuration (read before committing)

Flags that control how long this notebook takes. Defaults are set for a full "Save & Run All" commit:

| Flag | Cell | Default | Effect |
|---|---|---|---|
| `RETRAIN` | 1.3 config | `True` | Retrains all four models (needed in a fresh commit) |
| `FIT_SET` | 3.0 | `'mci_train'` | Subjects used to fit ComBat, z-score constants, and MICE |
| `RUN_MODEL_COMPARISON_BOOTSTRAP` | Section 7 | `False` | Set `True` for the 100-resample cross-model bootstrap (about 7 hours extra) |
| `RUN_KFOLD_BOOTSTRAP` | Section 12.5 | `True` | Landmark k-fold + bootstrap CIs (fast, Cox only) |
| `RUN_INTERVAL_SHAPE_KFOLD` | Section 13.6 | `True` | Refits all four models x 5 folds for Table 3 (slow) |
| `RETRAIN_SPLIT_COMPARISON` | Section 11 | `False` | Dropped from the paper, leave off |

Kaggle settings for the commit: Internet **on**, accelerator **None** (CPU). Section 14 at the end prints every number the paper reports and saves the final figures at 300 dpi.


## 1. Setup

### 1.1 Clone Repository and Install Dependencies

Clones this repository and installs the packages in `requirements.txt`. Skip this cell if you are running from a local clone.


In [ ]:
REPO_URL = 'https://github.com/savannahnix/adni-conformal-timing.git'
!git clone {REPO_URL}
%cd adni-conformal-timing
!pip install -r requirements.txt -q


### 1.2 Data

This notebook reads the ADNIMERGE table (one row per subject visit). ADNI data cannot be redistributed, so it is not included in this repository. Request access at https://adni.loni.usc.edu, download `ADNIMERGE.csv`, and see `data/README.md` for where to put it.

On Kaggle, upload `ADNIMERGE.csv` as a **private** dataset (never public) and point `DATA_PATH` below at it.


In [ ]:
import os
from pathlib import Path

# ── Set your paths here ───────────────────────────────────────────────────────
# REPO_DIR: the root of this repository.
# DATA_PATH: your local copy of ADNIMERGE.csv (see data/README.md).
#   Kaggle example: Path('/kaggle/input/<your-private-dataset>/ADNIMERGE.csv')
REPO_DIR  = Path.cwd() if (Path.cwd() / 'pipeline').exists() else Path.cwd().parent
DATA_PATH = REPO_DIR / 'data' / 'ADNIMERGE.csv'
# ─────────────────────────────────────────────────────────────────────────────
os.chdir(REPO_DIR)
assert DATA_PATH.exists(), f"ADNIMERGE.csv not found at {DATA_PATH}. See data/README.md."
print(f'REPO_DIR  : {REPO_DIR}')
print(f'DATA_PATH : {DATA_PATH}')


### 1.3 Imports and Global Configuration

Imports all required libraries and sets global constants used throughout the pipeline.

| Constant | Value | Purpose |
|----------|-------|---------|
| `RETRAIN` | `True` / `False` | Retrain from scratch vs. load checkpoints |
| `RANDOM_SEED` | `42` | Fixed across all models, CV splitters, and imputers |
| `N_FOLDS` | `5` | Stratified K-fold CV folds used in all HPO objectives |
| `HORIZONS` | `[3, 5]` | Prediction horizons (years) for binary AUC evaluation |

Feature name lists (`COG_FEATURES`, `MRI_FEATURES`, `CSF_FEATURES`, etc.) define the
canonical feature vocabulary for each modality. Module-level constants in `modeling.py`,
`preprocessing.py`, and `postprocessing.py` are synchronized to these values so all
modules share a single source of truth.


In [ ]:
%cd pipeline


In [ ]:
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats
import torch
import optuna
from pycox.evaluation import EvalSurv
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

print(f"GPU available : {torch.cuda.is_available()}")
print(f"Device        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU -- check runtime type'}")

# ── Paths (relative to the pipeline/ working directory) ───
BASELINE_DIR   = Path.cwd()
CHECKPOINT_DIR = BASELINE_DIR / 'checkpoints'
FIG_DIR        = BASELINE_DIR / 'figures'
OUT_DIR        = BASELINE_DIR / 'outputs'

# Directories are committed to repo with .gitkeep, no mkdir needed
assert CHECKPOINT_DIR.exists(), f"checkpoints/ not found. Is the repo cloned correctly?"
assert FIG_DIR.exists(),        f"figures/ not found. Is the repo cloned correctly?"
assert OUT_DIR.exists(),        f"outputs/ not found. Is the repo cloned correctly?"

# ── Global constants ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_FOLDS     = 5
HORIZONS    = [3, 5]
np.random.seed(RANDOM_SEED)

DX_ORDER   = ['CN', 'MCI', 'AD']
DX_PALETTE = {'CN': '#2ecc71', 'MCI': '#f39c12', 'AD': '#e74c3c'}

RETRAIN = True

# ── Feature groups ────────────────────────────────────────────────────────────
COG_FEATURES = [
    'MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
    'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
    'RAVLT_perc_forgetting', 'FAQ', 'MOCA', 'mPACCdigit', 'mPACCtrailsB',
    'EcogPtTotal', 'EcogSPTotal',
]
CSF_FEATURES  = ['ABETA', 'TAU', 'PTAU', 'FDG', 'AV45', 'FBB']
MRI_FEATURES  = ['Hippocampus', 'Entorhinal', 'Ventricles',
                  'WholeBrain', 'Fusiform', 'MidTemp', 'ICV']
DEMO_FEATURES = ['AGE', 'PTEDUCAT', 'APOE4']

SLOPE_FEATURES = [
    'MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
    'RAVLT_immediate', 'FAQ',
    'Hippocampus', 'Entorhinal', 'Ventricles', 'WholeBrain', 'Fusiform',
    'EcogPtTotal', 'EcogSPTotal', 'mPACCdigit',
]

MISSINGNESS_FLAGS = ['ABETA', 'TAU', 'PTAU', 'FDG', 'AV45', 'FBB',
                     'Hippocampus', 'mPACCdigit', 'EcogSPTotal']
MRI_HARMONIZE_COLS = ['Hippocampus', 'Entorhinal', 'Ventricles',
                       'Fusiform', 'MidTemp', 'WholeBrain']

print('Config ready.')
print(f'Baseline dir : {BASELINE_DIR}')
print(f'Data path    : {DATA_PATH}')
print(f'Figures      : {FIG_DIR}')
print(f'Outputs      : {OUT_DIR}')
print(f'Checkpoints  : {CHECKPOINT_DIR}')


# ── Human-readable feature label map ─────────────────────────────────────────
# Used by all importance plots, KM charts, and survival curve titles so that
# a general audience never sees raw variable names on any visual.
FEATURE_LABELS = {
    # Demographics
    'AGE':                   'Age at Baseline',
    'PTGENDER_num':          'Sex (Male=1)',
    'PTEDUCAT':              'Years of Education',
    'age_edu_interaction':   'Age × Education',
    'APOE4':                 'APOE ε4 Allele Count',
    # Cognitive tests
    'MMSE':                  'MMSE (Global Cognition)',
    'CDRSB':                 'CDR Sum of Boxes (Severity)',
    'ADAS11':                'ADAS-Cog 11 (Memory)',
    'ADAS13':                'ADAS-Cog 13 (Extended)',
    'LDELTOTAL':             'Logical Memory II (Delayed Recall)',
    'RAVLT_immediate':       'RAVLT Immediate Recall',
    'RAVLT_learning':        'RAVLT Learning Rate',
    'RAVLT_forgetting':      'RAVLT Forgetting Rate',
    'RAVLT_perc_forgetting': 'RAVLT % Forgetting',
    'FAQ':                   'FAQ (Functional Activities)',
    'MOCA':                  'MoCA (Global Cognition)',
    'mPACCdigit':            'mPACC Digit Span',
    'mPACCtrailsB':          'mPACC Trails B',
    'EcogPtTotal':           'ECog Patient-Reported Total',
    'EcogSPTotal':           'ECog Study Partner Total',
    # MRI volumetrics (ICV-normalized)
    'Hippocampus_ICV':       'Hippocampal Volume (ICV-adj)',
    'Entorhinal_ICV':        'Entorhinal Cortex Volume (ICV-adj)',
    'Ventricles_ICV':        'Ventricular Volume (ICV-adj)',
    'Fusiform_ICV':          'Fusiform Gyrus Volume (ICV-adj)',
    'MidTemp_ICV':           'Middle Temporal Gyrus Volume (ICV-adj)',
    'WholeBrain_ICV':        'Whole Brain Volume (ICV-adj)',
    'hippo_entorhinal_ratio':'Hippocampus / Entorhinal Ratio',
    'midtemp_fusiform_ratio':'MidTemporal / Fusiform Ratio',
    # CSF / PET biomarkers
    'ABETA':                 'CSF Amyloid-β (pg/mL)',
    'TAU':                   'CSF Total Tau (pg/mL)',
    'PTAU':                  'CSF Phospho-Tau (pg/mL)',
    'FDG':                   'FDG-PET (Glucose Metabolism)',
    'AV45':                  'AV45-PET (Amyloid Burden)',
    'FBB':                   'Florbetaben-PET (Amyloid Burden)',
    # APOE4 interaction terms
    'APOE4_x_amyloid':       'APOE ε4 × Amyloid Load',
    'APOE4_x_tau':           'APOE ε4 × Tau Burden',
    'APOE4_x_hippo_ICV':     'APOE ε4 × Hippocampal Volume',
    'APOE4_amyloid_pos':     'APOE ε4 × Amyloid Positivity',
    # Engineered / composite features
    'amyloid_composite':          'Amyloid Composite (CSF + PET)',
    'neurodegeneration_composite':'Neurodegeneration Composite (Tau + FDG)',
    'ATN_composite':              'ATN Biology Score',
    'amyloid_positive':           'Amyloid Positive (binary)',
    'cog_composite_pos':          'Cognitive Ability Composite',
    'severity_composite':         'Cognitive Severity Composite',
    'ecog_sp_pt_ratio':           'ECog Study-Partner / Patient Ratio',
    'n_modalities':               'Number of Available Modalities',
}

# Category membership: drives color-coding on importance plots
FEATURE_CATEGORIES = {
    'Demographics':        ['AGE','PTGENDER_num','PTEDUCAT','age_edu_interaction'],
    'Cognitive Tests':     ['MMSE','CDRSB','ADAS11','ADAS13','LDELTOTAL',
                            'RAVLT_immediate','RAVLT_learning','RAVLT_forgetting',
                            'RAVLT_perc_forgetting','FAQ','MOCA',
                            'mPACCdigit','mPACCtrailsB','EcogPtTotal','EcogSPTotal'],
    'MRI Volumes':         ['Hippocampus_ICV','Entorhinal_ICV','Ventricles_ICV',
                            'Fusiform_ICV','MidTemp_ICV','WholeBrain_ICV',
                            'hippo_entorhinal_ratio','midtemp_fusiform_ratio'],
    'CSF / PET Biomarkers':['ABETA','TAU','PTAU','FDG','AV45','FBB'],
    'APOE4 Interactions':  ['APOE4','APOE4_x_amyloid','APOE4_x_tau',
                            'APOE4_x_hippo_ICV','APOE4_amyloid_pos'],
    'Engineered Features': ['amyloid_composite','neurodegeneration_composite',
                            'ATN_composite','amyloid_positive',
                            'cog_composite_pos','severity_composite',
                            'ecog_sp_pt_ratio','n_modalities'],
}
CATEGORY_COLORS = {
    'Demographics':         '#5b7fce',
    'Cognitive Tests':      '#e74c3c',
    'MRI Volumes':          '#27ae60',
    'CSF / PET Biomarkers': '#e67e22',
    'APOE4 Interactions':   '#9b59b6',
    'Engineered Features':  '#16a085',
    'Other':                '#95a5a6',
}

def feat_label(col):
    """Return human-readable label for a feature column name."""
    return FEATURE_LABELS.get(col, col)

def feature_color(col):
    """Return category color for a feature column name."""
    for cat, feats in FEATURE_CATEGORIES.items():
        if col in feats:
            return CATEGORY_COLORS[cat]
    return CATEGORY_COLORS['Other']

def category_legend_patches():
    """Return matplotlib patches for a category color legend."""
    import matplotlib.patches as mpatches
    return [mpatches.Patch(color=c, label=k)
            for k, c in CATEGORY_COLORS.items()
            if k != 'Other']

def plot_importance(imp_series, title, xlabel, save_path, n=20, sign_series=None):
    """
    Plot a horizontal bar chart of feature importances with:
      - Human-readable labels on y-axis
      - Category color-coding
      - Legend identifying categories
      - Optional per-bar sign override (red=harmful, blue=protective)
        when sign_series is provided (used for Cox/AFT coefficients).
    """
    top = imp_series.abs().sort_values(ascending=False).head(n)
    labels_mapped = [feat_label(f) for f in top.index]
    if sign_series is not None:
        colors = ['#e74c3c' if sign_series.get(f, 1) > 0 else '#3498db' for f in top.index]
        legend_patches = [
            __import__('matplotlib.patches', fromlist=['Patch']).Patch(color='#e74c3c', label='Increases risk / shortens survival'),
            __import__('matplotlib.patches', fromlist=['Patch']).Patch(color='#3498db', label='Decreases risk / extends survival'),
        ]
    else:
        colors = [feature_color(f) for f in top.index]
        legend_patches = category_legend_patches()

    fig, ax = plt.subplots(figsize=(10, max(6, n * 0.38)))
    ax.barh(range(len(top)), top.values[::-1] if False else list(reversed(top.values)),
            color=list(reversed(colors)), alpha=0.88, edgecolor='white')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(list(reversed(labels_mapped)), fontsize=10)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=13, pad=12)
    ax.legend(handles=legend_patches, fontsize=8,
              loc='lower right', framealpha=0.9)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
import modeling
modeling.CHECKPOINT_DIR = CHECKPOINT_DIR
modeling.RANDOM_SEED = RANDOM_SEED
modeling.N_FOLDS = N_FOLDS
modeling.HORIZONS = HORIZONS
modeling.FIG_DIR = FIG_DIR
modeling.OUT_DIR = OUT_DIR

from modeling import save_checkpoint, load_checkpoint

import preprocessing
preprocessing.MRI_HARMONIZE_COLS = MRI_HARMONIZE_COLS
preprocessing.RANDOM_SEED = RANDOM_SEED
preprocessing.FIG_DIR = FIG_DIR

import postprocessing
postprocessing.FIG_DIR = FIG_DIR
postprocessing.OUT_DIR = OUT_DIR

## 2. Data Preparation

### 2.1 Data Loading and Diagnosis Harmonization

Loads the ADNIMERGE CSV and standardizes diagnosis labels across ADNI study phases.
ADNI uses different coding conventions across phases (ADNI1, ADNI-GO, ADNI2, ADNI3, ADNI4),
so raw `DX` values are remapped to a consistent three-state vocabulary: `CN`, `MCI`, `AD`.

**Input:** `ADNIMERGE.csv` from ADNI (see `data/README.md`)  
**Output:** `df`: harmonized longitudinal DataFrame, one row per subject-visit

A baseline-only view (`df_bl`) is extracted by filtering to `VISCODE == 'bl'` for
downstream feature engineering and cohort assembly.


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {df.shape}  |  Subjects: {df["RID"].nunique()}')

# ── Remap all diagnosis variants to 3 states ───────────────────────
dx_remap = {
    'CN': 'CN',   'SMC': 'CN',
    'MCI': 'MCI', 'EMCI': 'MCI', 'LMCI': 'MCI',
    'AD': 'AD', 'Dementia': 'AD', 'Dem': 'AD',
}
df['DX']    = df['DX'].map(dx_remap)
df['DX_bl'] = df['DX_bl'].map(dx_remap)

unmapped = df['DX'].isna().sum()
print(f'Rows with no recorded diagnosis (expected in ADNI): {unmapped}')

# ── Parse CSF strings that contain '>' or '<' ────────────────────────────────
for col in ['ABETA', 'TAU', 'PTAU', 'ABETA_bl', 'TAU_bl', 'PTAU_bl']:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace('>', '').str.replace('<', '').str.strip(),
        errors='coerce'
    )

# ── Coerce all numeric columns ────────────────────────────────────────────────
numeric_cols = (
    DEMO_FEATURES + COG_FEATURES + CSF_FEATURES + MRI_FEATURES +
    ['Years_bl', 'Month_bl', 'mPACCdigit', 'mPACCtrailsB']
)
for c_ in numeric_cols:
    if c_ in df.columns:
        df[c_] = pd.to_numeric(df[c_], errors='coerce')

df = df.sort_values(['RID', 'Years_bl']).reset_index(drop=True)

df_bl = df[df['VISCODE'] == 'bl'].copy()
print(f'\nBaseline subjects: {len(df_bl)}')
print('Baseline DX breakdown:')
print(df_bl['DX_bl'].value_counts())

### 2.2 Reversion Removal

Subjects who show a backward MCI → CN transition are excluded from the MCI cohort.
Per sponsor guidance, these reversions reflect transient factors (sleep deprivation, mood,
medication effects) rather than genuine biological recovery and would introduce noise
into event-time labels.

Subjects are classified into four trajectory groups using `classify_reverters()`:

| Group | Definition | Action |
|-------|-----------|--------|
| `transient_noise` | Single reversion, immediately returns to MCI | Exclude |
| `sustained_recovery` | ≥3 trailing CN visits | Exclude |
| `bouncer` | Alternating MCI/CN pattern | Exclude |
| `progressor` | Reverted to CN but later progressed to AD | Exclude |

All four groups are removed from the MCI → AD cohort before survival label construction.


In [ ]:
from preprocessing import classify_reverters

reversion_groups = classify_reverters(df)
for g, rids in reversion_groups.items():
    print(f'  {g:<22}: {len(rids)} subjects')

# Drop noise + bouncers; keep progressors; flag sustained for sensitivity
REVERSION_EXCLUSIONS = (reversion_groups['transient_noise'] |
                        reversion_groups['bouncer'])
SENSITIVITY_EXCLUSIONS = REVERSION_EXCLUSIONS | reversion_groups['sustained_recovery']

print(f'\nMain analysis exclusions:  {len(REVERSION_EXCLUSIONS)} subjects dropped')
print(f'Sensitivity exclusions:    {len(SENSITIVITY_EXCLUSIONS)} subjects dropped')
print(f'Progressors kept:          {len(reversion_groups["progressor"])} subjects')

### 2.3 Survival Label Construction

Builds time-to-event labels for each cohort transition using `build_survival_labels()`.
For each subject:

| Field | Definition |
|-------|-----------|
| `event = 1` | Target diagnosis observed at a post-baseline visit |
| `event = 0` | Censored: subject never reached target state or was lost to follow-up |
| `duration` | Years from baseline to first conversion (events) or last visit (censored) |
| `cutoff` | Temporal boundary used downstream to enforce leakage-free slope computation |

Subjects with `duration ≤ 0` are dropped as degenerate records.

Two label sets are constructed: **MCI → AD** (`surv_mci`) and **CN → MCI or AD** (`surv_cn`).
Event rates and median follow-up time are printed to screen for inspection.


In [ ]:
from preprocessing import build_survival_labels

# ── MCI -> AD (primary cohort) ─────────────────────────────────────────
surv_mci = build_survival_labels(df, df_bl, 'MCI', 'AD',
                                  exclusion_rids=REVERSION_EXCLUSIONS)
print(f'MCI -> AD:  {surv_mci["event"].sum()} events / {len(surv_mci)} '
      f'({surv_mci["event"].mean()*100:.1f}%)')

# ── CN -> MCI ────────────────────────────────────────────────────────────────
surv_cn_mci = build_survival_labels(df, df_bl, 'CN', 'MCI')

# ── CN -> AD (direct skip) ─────────────────────────────────────────────
surv_cn_ad = build_survival_labels(df, df_bl, 'CN', 'AD')

# ── CN -> MCI or AD (competing risks combined) ──────────────────────────────
surv_cn = surv_cn_mci.copy()
for rid in surv_cn.index:
    if rid in surv_cn_ad.index and surv_cn_ad.loc[rid, 'event'] == 1:
        if surv_cn.loc[rid, 'event'] == 0:
            surv_cn.loc[rid, ['event','duration','cutoff']] = [
                1, surv_cn_ad.loc[rid,'duration'], surv_cn_ad.loc[rid,'cutoff']
            ]
        else:
            earlier = min(surv_cn.loc[rid,'duration'], surv_cn_ad.loc[rid,'duration'])
            surv_cn.loc[rid, ['duration','cutoff']] = [earlier, earlier]

print(f'CN  -> MCI or AD: {surv_cn["event"].sum()} events / {len(surv_cn)} '
      f'({surv_cn["event"].mean()*100:.1f}%)')

# ── Kaplan-Meier overview plot ────────────────────────────────────────────────
from lifelines import KaplanMeierFitter
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, surv, label, color in [
    (axes[0], surv_mci, 'MCI -> AD', '#e74c3c'),
    (axes[1], surv_cn,  'CN -> MCI or AD', '#f39c12'),
]:
    kmf = KaplanMeierFitter()
    kmf.fit(surv['duration'], event_observed=surv['event'], label=label)
    kmf.plot_survival_function(ax=ax, color=color, ci_show=True, ci_alpha=0.15)
    ax.set(xlabel='Years from Baseline', ylabel='Survival Probability')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.suptitle('Kaplan-Meier Curves by Cohort', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'km_cohort_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Harmonization and Imputation

### 3.0 Held-Out Split Fixed Before Preprocessing (version 2)

In [ ]:
# ── 3.0  Fix the held-out split before any preprocessing (version 2) ─────────
# Preprocessing (ComBat in 3.2, z-score constants in 4.1, MICE in 4.2) is fit only on
# subjects outside the held-out calibration and evaluation sets, then applied to everyone.
# The split uses the same calls and seed as Section 4.2, so each set holds the same subjects
# as version 1; Section 4.2 asserts this.
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from neuroCombat import neuroCombat
from preprocessing import assemble_cohort

FIT_SET = 'mci_train'   # 'mci_train' (paper) or 'all_non_heldout'

def _split_rids(surv_labels):
    slopes = pd.DataFrame({'RID': df_bl['RID'].unique()})
    _, ev, _, rids = assemble_cohort(df_bl, surv_labels, slopes, ['AGE'], [])
    idx = np.arange(len(rids))
    i_tr, i_tmp, _, ev_tmp = train_test_split(idx, ev, test_size=0.30,
                                              random_state=RANDOM_SEED, stratify=ev)
    i_cal, i_ev = train_test_split(i_tmp, test_size=0.333,
                                   random_state=RANDOM_SEED, stratify=ev_tmp)
    rids = np.asarray(rids)
    return {'train': set(rids[i_tr]), 'cal': set(rids[i_cal]), 'eval': set(rids[i_ev])}

MCI_SPLIT = _split_rids(surv_mci)
CN_SPLIT = _split_rids(surv_cn)
HELDOUT_RIDS = MCI_SPLIT['cal'] | MCI_SPLIT['eval'] | CN_SPLIT['cal'] | CN_SPLIT['eval']
FIT_RIDS = set(MCI_SPLIT['train']) if FIT_SET == 'mci_train' else set(df_bl['RID']) - HELDOUT_RIDS
print(f"MCI split: train={len(MCI_SPLIT['train'])}, cal={len(MCI_SPLIT['cal'])}, eval={len(MCI_SPLIT['eval'])}")
print(f"Held-out subjects (MCI + CN cal/eval): {len(HELDOUT_RIDS)}")
print(f"Preprocessing fit set ({FIT_SET}): {len(FIT_RIDS)} of {df_bl['RID'].nunique()} baseline subjects")

def mice_fit_transform(train_df, other_dfs, max_iter=10, seed=RANDOM_SEED):
    """MICE fit on the training rows only, then applied to the other sets."""
    imp = IterativeImputer(max_iter=max_iter, random_state=seed, keep_empty_features=True)
    tr = pd.DataFrame(imp.fit_transform(train_df), columns=train_df.columns, index=train_df.index)
    others = [pd.DataFrame(imp.transform(d), columns=d.columns, index=d.index) for d in other_dfs]
    return tr, others

def run_combat_train_only(df_baseline, fit_rids, cols, neuroCombat):
    """ComBat (same settings as preprocessing.run_combat: batch = field strength, no
    covariate columns passed) with estimates fit on `fit_rids` only, then applied to every
    subject. NaNs are never filled in the output; they are median-filled (training medians)
    only inside the fit, exactly as the original function does."""
    has = df_baseline[cols[0]].notna() & df_baseline['FLDSTRENG'].notna()
    for col in cols:
        df_baseline[f'{col}_raw'] = df_baseline[col].copy()
    batch = df_baseline['FLDSTRENG'].map({'1.5 Tesla MRI': 1, '3 Tesla MRI': 2})
    fit = has & df_baseline['RID'].isin(fit_rids)
    data_fit = df_baseline.loc[fit, cols].astype(float)
    data_fit = data_fit.fillna(data_fit.median())
    res = neuroCombat(dat=data_fit.T.values, covars=pd.DataFrame({'batch': batch[fit].values}), batch_col='batch')
    est = res['estimates']
    grand = np.asarray(est['stand.mean'])[:, 0]
    sd_pool = np.sqrt(np.asarray(est['var.pooled'])[:, 0])
    gamma, delta = np.asarray(est['gamma.star']), np.asarray(est['delta.star'])
    levels = np.sort(np.unique(batch[fit].values))          # neuroCombat recodes batches in sorted order
    X = df_baseline.loc[has, cols].values.astype(float)     # NaN stays NaN
    b = batch[has].values
    out = np.full_like(X, np.nan)
    for k, lvl in enumerate(levels):
        r = b == lvl
        out[r] = ((X[r] - grand) / sd_pool - gamma[k]) / np.sqrt(delta[k]) * sd_pool + grand
    df_baseline.loc[has, cols] = out
    print(f'  ComBat fit on {int(fit.sum())} subjects, applied to {int(has.sum())}')
    return df_baseline, res, fit


### 3.1 MRI Batch Effect Audit

Different ADNI phases used different MRI scanners: 1.5T in ADNI1/GO and 3T in ADNI2/3/4.
This creates systematic batch effects in volumetric MRI features that can bias survival
models if left uncorrected. A model trained on pooled data would partially learn
scanner differences rather than biology.

This cell audits the magnitude of the effect by visualizing raw feature distributions
stratified by both field strength and ADNI protocol phase for three regions:
Hippocampus, Entorhinal cortex, and Ventricles.

**What to look for:** Visible distributional shift between 1.5T and 3T groups in any
of the three features indicates that ComBat harmonization is warranted before modeling.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
mri_check = ['Hippocampus', 'Entorhinal', 'Ventricles']

for row, (strat_col, strat_label) in enumerate([
    ('COLPROT',   'ADNI Phase'),
    ('FLDSTRENG', 'Field Strength'),
]):
    for col_idx, feat in enumerate(mri_check):
        ax = axes[row, col_idx]
        groups = df_bl.dropna(subset=[feat, 'ICV', strat_col]).copy()
        groups[f'{feat}_ICV'] = groups[feat] / groups['ICV']
        for name, grp in groups.groupby(strat_col):
            ax.hist(grp[f'{feat}_ICV'], bins=40, alpha=0.5, label=str(name))
        ax.set(xlabel=f'{feat} / ICV', title=f'{feat} by {strat_label}')
        ax.legend(fontsize=8)

plt.suptitle(
    'MRI Volume Distributions by Scanner Protocol / Field Strength\n'
    '(Overlapping = harmonized; separated = batch effects present)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'batch_effects_audit.png', dpi=150, bbox_inches='tight')
plt.show()

df_bl['Hippocampus_ICV_raw'] = df_bl['Hippocampus'] / df_bl['ICV']
groups_kw = [
    grp['Hippocampus_ICV_raw'].dropna()
    for _, grp in df_bl.dropna(subset=['Hippocampus_ICV_raw']).groupby('COLPROT')
    if len(grp) > 5
]
if len(groups_kw) >= 2:
    stat, p = stats.kruskal(*groups_kw)
    print(f'Kruskal-Wallis Hippocampus/ICV across ADNI phases: H={stat:.2f}, p={p:.4f}')
    if p < 0.05:
        print('Significant batch effect -- proceed to Cell 6b (ComBat)')
    else:
        print('No significant batch effect -- skip 6b and 6c')

### 3.2 ComBat Harmonization

Applies ComBat batch effect correction (via `neuroCombat`) to six MRI volumetric features:
Hippocampus, Entorhinal, Ventricles, Fusiform, MidTemp, and WholeBrain.

ComBat estimates and removes additive and multiplicative scanner effects (parameterized
per-batch) while preserving biological variance associated with protected covariates:
diagnosis (`DX_bl`), age, and sex. These protected covariates are explicitly passed to
`neuroCombat` so their signal is not inadvertently absorbed into the batch correction.

**Implementation notes:**
- NaN cells are filled with per-feature medians before ComBat (which cannot handle missing values)
  and restored to NaN afterward. No new data is fabricated
- Original pre-harmonization values are saved in `_raw` columns for validation
- Writeback uses `iloc` integer indexing to avoid index alignment issues

A before/after histogram and `harmonization_report()` are generated to confirm the
1.5T vs. 3T gap is meaningfully reduced while cross-phase biological differences remain.


In [ ]:
from preprocessing import run_combat

# ── Infer missing FLDSTRENG from COLPROT ─────────────────────────────────────
protocol_to_field = {
    'ADNI1':  '1.5 Tesla MRI', 'ADNIGO': '1.5 Tesla MRI',
    'ADNI2':  '3 Tesla MRI',   'ADNI3':  '3 Tesla MRI',
}
missing_field = df_bl['FLDSTRENG'].isna()
df_bl.loc[missing_field, 'FLDSTRENG'] = df_bl.loc[missing_field, 'COLPROT'].map(protocol_to_field)
print(f"Inferred FLDSTRENG for {missing_field.sum()} subjects from COLPROT.")
print(df_bl['FLDSTRENG'].value_counts())

# ── Reset index so iloc positional writes are unambiguous ─────────────────────
df_bl = df_bl.reset_index(drop=True)
print(f"\ndf_bl index reset. Shape: {df_bl.shape}")


print("\nRunning ComBat...")
df_bl, combat_result, combat_fit_mask = run_combat_train_only(df_bl, FIT_RIDS, MRI_HARMONIZE_COLS, neuroCombat)

df_bl['ICV'] = pd.to_numeric(df_bl['ICV'], errors='coerce')
for region in MRI_HARMONIZE_COLS:
    df_bl[f'{region}_ICV'] = df_bl[region] / df_bl['ICV']

print("\nICV-normalized columns updated.")
print(f"ICV NaN count       : {df_bl['ICV'].isna().sum()}")
print(f"Hippocampus NaN     : {df_bl['Hippocampus'].isna().sum()} (should be ~349)")
print("Hippocampus_ICV by COLPROT:")
print(df_bl.groupby('COLPROT')['Hippocampus_ICV'].mean().round(6))

In [ ]:
from preprocessing import harmonization_report, plot_before_after


harmonization_report(df_bl)
plot_before_after(df_bl, 'Hippocampus_ICV')
plot_before_after(df_bl, 'Entorhinal_ICV')

### 3.3 Tiered Imputation

Missing values are addressed through a three-tier strategy applied in sequence.
Each tier is applied only to values that remain missing after the previous tier.

**Tier 1: Longitudinal nearest-neighbor fill** (`longitudinal_fill`)  
For each subject, missing values at a given visit are filled from the nearest available
observation within a ±1 year window, respecting the longitudinal structure of the data
and avoiding cross-subject contamination. This is the most principled imputation for
longitudinal clinical data because it borrows from the same subject near the same timepoint.

**Tier 2: MICE** (`mice_impute`)  
Remaining gaps after Tier 1 are imputed using `IterativeImputer` (sklearn's implementation
of MICE: Multivariate Imputation by Chained Equations), which models each feature as a
function of all others iteratively until convergence. Default 10 iterations.

**Tier 3: Two-stage CSF prediction** (Section 5.2 below)  
Missing CSF ABETA values are predicted from PET and MRI features using a LightGBM
regressor trained on subjects with known measurements. Applied after Tiers 1 and 2.

**Missingness flags** (`miss_*` columns) are preserved as binary model features throughout
all tiers, allowing models to learn whether the absence of a measurement is itself informative.


In [ ]:
from preprocessing import longitudinal_fill

print('Applying longitudinal nearest-neighbor fill (Tier 1)...')
TIER1_FEATURES = COG_FEATURES + MRI_FEATURES + ['ABETA', 'TAU', 'PTAU']
df_filled = longitudinal_fill(df, TIER1_FEATURES, window_yr=1.0)
print(f'  Fill rate improvement for MMSE: '
      f'{df["MMSE"].notna().mean()*100:.1f}% -> '
      f'{df_filled["MMSE"].notna().mean()*100:.1f}%')
print(f'  Fill rate improvement for Hippocampus: '
      f'{df["Hippocampus"].notna().mean()*100:.1f}% -> '
      f'{df_filled["Hippocampus"].notna().mean()*100:.1f}%')

# Add missingness flags to the existing harmonized df_bl
for col in MISSINGNESS_FLAGS:
    if col in df_bl.columns:
        df_bl[f'miss_{col}'] = df_bl[col].isna().astype(int)

print('Missingness flags added.')
print('Imputation framework ready: MICE will be applied per cohort in the cohort assembly step.')

## 4. Feature Engineering

### 4.1 Engineered Features

Constructs derived features on the baseline DataFrame before cohort assembly:

| Feature Group | Construction | Rationale |
|--------------|-------------|-----------|
| ICV-normalized MRI volumes | Raw volume / intracranial volume | Corrects for head size variation across subjects |
| Amyloid composite | Scaled average of ABETA (inverted), AV45, FBB | Combines CSF and PET amyloid into a single biomarker |
| Cognitive composite: ability | Standardized MMSE + LDELTOTAL | Summarizes memory and orientation into one score |
| Cognitive composite: severity | Standardized CDRSB + FAQ | Summarizes functional impairment |
| APOE4 interaction terms | APOE4 × hippocampal/amyloid features | Captures gene-biomarker interactions |
| Missingness flags | Binary indicator per frequently-missing feature | Encodes whether absence of measurement is informative |
| Visit metadata | `n_modalities`, protocol dummies, scanner dummies | Controls for data collection heterogeneity |


In [ ]:
_FIT = df_bl['RID'].isin(FIT_RIDS)   # version 2: constants from non-held-out subjects only

# ── ICV-normalized MRI volumes ───────────────────────────────────────────────
for region in ['Hippocampus', 'Entorhinal', 'Ventricles',
               'Fusiform', 'MidTemp', 'WholeBrain']:
    if f'{region}_ICV' not in df_bl.columns:
        df_bl[f'{region}_ICV'] = df_bl[region] / df_bl['ICV']

# ── Amyloid composite (ABETA CSF + AV45 PET combined) ────────────────────────
for feat, col in [('ABETA','ABETA_z'), ('AV45','AV45_z')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['ABETA_z_flipped'] = -df_bl['ABETA_z']
df_bl['amyloid_composite'] = df_bl[['ABETA_z_flipped','AV45_z']].mean(axis=1, skipna=True)
df_bl['amyloid_positive'] = (df_bl['ABETA'] < 880).astype(float)
df_bl.loc[df_bl['ABETA'].isna() & df_bl['AV45'].notna(), 'amyloid_positive'] = (
    df_bl.loc[df_bl['ABETA'].isna() & df_bl['AV45'].notna(), 'AV45'] > 1.11
).astype(float)

# ── Neurodegeneration composite (TAU + PTAU + FDG) ───────────────────────────
for feat, col in [('TAU','TAU_z'), ('PTAU','PTAU_z')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
mu, sd = df_bl.loc[_FIT, 'FDG'].mean(), df_bl.loc[_FIT, 'FDG'].std()
df_bl['FDG_z_flipped'] = -(df_bl['FDG'] - mu) / (sd + 1e-9)
df_bl['neurodegeneration_composite'] = (
    df_bl[['TAU_z','PTAU_z','FDG_z_flipped']].mean(axis=1, skipna=True)
)

# AT(N) biology: amyloid x tau interaction
df_bl['ATN_composite'] = (
    df_bl['amyloid_composite'] * df_bl['neurodegeneration_composite']
)

# ── Cognitive composites ──────────────────────────────────────────────────────
for feat in ['MMSE', 'LDELTOTAL', 'RAVLT_immediate']:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[f'{feat}_z'] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['cog_composite_pos'] = (
    df_bl[['MMSE_z','LDELTOTAL_z','RAVLT_immediate_z']].mean(axis=1, skipna=True)
)

for feat, col in [('CDRSB','CDRSB_z_sev'),('FAQ','FAQ_z_sev'),('ADAS13','ADAS13_z_sev')]:
    mu, sd = df_bl.loc[_FIT, feat].mean(), df_bl.loc[_FIT, feat].std()
    df_bl[col] = (df_bl[feat] - mu) / (sd + 1e-9)
df_bl['severity_composite'] = (
    df_bl[['CDRSB_z_sev','FAQ_z_sev','ADAS13_z_sev']].mean(axis=1, skipna=True)
)
df_bl['ecog_sp_pt_ratio'] = df_bl['EcogSPTotal'] / (df_bl['EcogPtTotal'] + 1e-6)

# ── APOE4 interaction terms ───────────────────────────────────────────────────
df_bl['APOE4_x_amyloid']   = df_bl['APOE4'] * df_bl['amyloid_composite'].fillna(0)
df_bl['APOE4_x_tau']       = df_bl['APOE4'] * df_bl['TAU_z'].fillna(0)
df_bl['APOE4_x_hippo_ICV'] = df_bl['APOE4'] * df_bl['Hippocampus_ICV'].fillna(
    df_bl.loc[_FIT, 'Hippocampus_ICV'].median())
df_bl['APOE4_amyloid_pos'] = df_bl['APOE4'] * df_bl['amyloid_positive'].fillna(0)

# ── Multimodal completeness score ────────────────────────────────────────────
df_bl['n_modalities'] = (
    df_bl['MMSE'].notna().astype(int)
    + df_bl['ABETA'].notna().astype(int)
    + df_bl['AV45'].notna().astype(int)
    + df_bl['FDG'].notna().astype(int)
    + df_bl['Hippocampus'].notna().astype(int)
    + df_bl['APOE4'].notna().astype(int)
)

# ── Demographics ─────────────────────────────────────────────────────────────
df_bl['PTGENDER_num']        = (df_bl['PTGENDER'] == 'Male').astype(int)
df_bl['age_edu_interaction'] = df_bl['AGE'] * df_bl['PTEDUCAT']

# ── Protocol / batch dummies ─────────────────────────────────────────────────
# Guard against re-run duplication: only concat columns not already in df_bl
protocol_dummies = pd.get_dummies(df_bl['COLPROT'], prefix='prot', drop_first=True)
field_dummies    = pd.get_dummies(df_bl['FLDSTRENG'], prefix='field', drop_first=True)

new_prot  = [c for c in protocol_dummies.columns if c not in df_bl.columns]
new_field = [c for c in field_dummies.columns    if c not in df_bl.columns]

if new_prot:
    df_bl = pd.concat([df_bl, protocol_dummies[new_prot]], axis=1)
if new_field:
    df_bl = pd.concat([df_bl, field_dummies[new_field]], axis=1)

# Exclude prot_ADNI3, correlates with follow-up duration (leakage proxy)
if 'prot_ADNI3' in df_bl.columns:
    df_bl = df_bl.drop(columns=['prot_ADNI3'])

print('Feature engineering complete.')
print(f'df_bl shape: {df_bl.shape}')
print(f'Protocol dummies present: {[c for c in df_bl.columns if c.startswith("prot_")]}')
print(f'Field dummies present:    {[c for c in df_bl.columns if c.startswith("field_")]}')

### 4.2 Cohort Assembly and Final Imputation

Joins baseline features and survival labels into a single model-ready DataFrame
per cohort using `assemble_cohort()`. A final MICE pass fills residual NaNs.

The assembled cohort is split 70/30 into development and held-out test sets using
a stratified split on the event indicator. The test set is not touched until Section 7.

**Feature policy:** Only true clinical or engineered biological features are included.
Missingness flags (`miss_*`), protocol dummies (`prot_*`, `field_*`), and
`n_modalities` (which encodes data collection patterns rather than patient biology)
are excluded. A hard guard assertion confirms this after assembly.

**Outputs per cohort:**

```
X_mci_train, X_mci_test          -- feature matrices
y_ev_mci_train, y_ev_mci_test    -- binary event indicators
y_dur_mci_train, y_dur_mci_test  -- time to event / censoring (years)
```


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from preprocessing import mice_impute, assemble_cohort
from sklearn.model_selection import train_test_split

# True clinical/biological features only.
# Excluded by design:
#   - miss_* flags       : encode data collection patterns, not biology
#   - field_* / prot_*   : scanner/protocol metadata, not patient features
#   - n_modalities       : encodes missingness pattern (proxy for data quality,
#                          not a clinical predictor)
#   - composites that are linear combinations of included raw features
#     (cog_composite_pos, severity_composite, amyloid_composite,
#      neurodegeneration_composite, ATN_composite, amyloid_positive,
#      ecog_sp_pt_ratio), re-add post-analysis if ablation warrants it
core_features = [
    # Demographics
    'AGE', 'PTGENDER_num', 'PTEDUCAT', 'APOE4',
    # Cognitive tests, raw scores
    'MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
    'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
    'FAQ', 'MOCA', 'EcogPtTotal', 'EcogSPTotal',
    'mPACCdigit', 'mPACCtrailsB',
    # MRI volumetrics, ICV-normalized
    'Hippocampus_ICV', 'Entorhinal_ICV', 'Ventricles_ICV',
    'Fusiform_ICV', 'MidTemp_ICV', 'WholeBrain_ICV',
    'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio',
    # CSF / PET biomarkers: raw measurements
    'ABETA', 'TAU', 'PTAU', 'FDG', 'AV45',
    # APOE4 interaction terms
    'APOE4_x_amyloid', 'APOE4_x_tau', 'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos',
]

slopes_empty = pd.DataFrame({'RID': df_bl['RID'].unique()})  # slopes disabled

X_mci_raw, y_ev_mci, y_dur_mci, rids_mci = assemble_cohort(
    df_bl, surv_mci, slopes_empty, core_features, [])
X_cn_raw,  y_ev_cn,  y_dur_cn,  rids_cn  = assemble_cohort(
    df_bl, surv_cn,  slopes_empty, core_features, [])

print(f'MCI cohort: {X_mci_raw.shape[0]} subjects x {X_mci_raw.shape[1]} features | '
      f'events={y_ev_mci.sum()} ({y_ev_mci.mean()*100:.1f}%)')
print(f'CN cohort:  {X_cn_raw.shape[0]}  subjects x {X_cn_raw.shape[1]} features | '
      f'events={y_ev_cn.sum()} ({y_ev_cn.mean()*100:.1f}%)')

X_mci_raw = X_mci_raw.loc[:, ~X_mci_raw.columns.duplicated()]
X_cn_raw  = X_cn_raw.loc[:,  ~X_cn_raw.columns.duplicated()]

# Guard: none of these columns should survive into the feature matrix
forbidden = (
    [c for c in X_mci_raw.columns if c.startswith('miss_')]
  + [c for c in X_mci_raw.columns if c.startswith('prot_')]
  + [c for c in X_mci_raw.columns if c.startswith('field_')]
  + ['n_modalities', 'pre_conversion_span_yr', 'n_visits_used',
     'visit_regularity', 'prot_ADNI3']
)
for f in forbidden:
    assert f not in X_mci_raw.columns, f'Forbidden feature in X_mci_raw: {f}'
    assert f not in X_cn_raw.columns,  f'Forbidden feature in X_cn_raw:  {f}'
print('Feature guard passed. No missingness/protocol columns present.')

# ── 70 / 20 / 10 stratified split (train / calibration / evaluation) ─────────
# Version 2: split the raw matrices first (same calls and seed, so the subjects in each set
# are unchanged), then fit MICE on the training rows only.
X_mci_train, X_mci_tmp, y_ev_mci_train, y_ev_mci_tmp, y_dur_mci_train, y_dur_mci_tmp = (
    train_test_split(X_mci_raw, y_ev_mci, y_dur_mci,
                     test_size=0.30, random_state=RANDOM_SEED, stratify=y_ev_mci))
X_cn_train, X_cn_tmp, y_ev_cn_train, y_ev_cn_tmp, y_dur_cn_train, y_dur_cn_tmp = (
    train_test_split(X_cn_raw, y_ev_cn, y_dur_cn,
                     test_size=0.30, random_state=RANDOM_SEED, stratify=y_ev_cn))
X_mci_cal, X_mci_eval, y_ev_mci_cal, y_ev_mci_eval, y_dur_mci_cal, y_dur_mci_eval = (
    train_test_split(X_mci_tmp, y_ev_mci_tmp, y_dur_mci_tmp,
                     test_size=0.333, random_state=RANDOM_SEED, stratify=y_ev_mci_tmp))
X_cn_cal, X_cn_eval, y_ev_cn_cal, y_ev_cn_eval, y_dur_cn_cal, y_dur_cn_eval = (
    train_test_split(X_cn_tmp, y_ev_cn_tmp, y_dur_cn_tmp,
                     test_size=0.333, random_state=RANDOM_SEED, stratify=y_ev_cn_tmp))

# Preprocessing in Sections 3.2 and 4.1 was fit around the split fixed in Section 3.0.
for _name, _X, _ref in [('MCI train', X_mci_train, MCI_SPLIT['train']), ('MCI cal', X_mci_cal, MCI_SPLIT['cal']),
                        ('MCI eval', X_mci_eval, MCI_SPLIT['eval']), ('CN train', X_cn_train, CN_SPLIT['train']),
                        ('CN cal', X_cn_cal, CN_SPLIT['cal']), ('CN eval', X_cn_eval, CN_SPLIT['eval'])]:
    assert set(_X.index) == _ref, f'{_name} differs from the split fixed in Section 3.0'
print('Split matches Section 3.0.')

print('Applying MICE imputation (fit on training rows only)...')
X_mci_train, (X_mci_cal, X_mci_eval) = mice_fit_transform(X_mci_train, [X_mci_cal, X_mci_eval])
X_cn_train, (X_cn_cal, X_cn_eval) = mice_fit_transform(X_cn_train, [X_cn_cal, X_cn_eval])
X_mci_imp = pd.concat([X_mci_train, X_mci_cal, X_mci_eval]).loc[X_mci_raw.index]
X_cn_imp = pd.concat([X_cn_train, X_cn_cal, X_cn_eval]).loc[X_cn_raw.index]

# Backward-compat alias: X_mci_test still refers to the cal set for earlier model sections
X_mci_test     = X_mci_cal
y_ev_mci_test  = y_ev_mci_cal
y_dur_mci_test = y_dur_mci_cal
X_cn_test      = X_cn_cal
y_ev_cn_test   = y_ev_cn_cal
y_dur_cn_test  = y_dur_cn_cal

print(f'MCI  train : n={len(X_mci_train)}, events={y_ev_mci_train.sum()} ({y_ev_mci_train.mean()*100:.1f}%)')
print(f'MCI  cal   : n={len(X_mci_cal)},   events={y_ev_mci_cal.sum()}   ({y_ev_mci_cal.mean()*100:.1f}%)')
print(f'MCI  eval  : n={len(X_mci_eval)},  events={y_ev_mci_eval.sum()}  ({y_ev_mci_eval.mean()*100:.1f}%)')

assert X_mci_imp.isna().sum().sum() == 0
assert X_cn_imp.isna().sum().sum()  == 0
print('Imputation complete. Zero NaNs.')

feature_names_mci = list(X_mci_imp.columns)
feature_names_cn  = list(X_cn_imp.columns)
print(f'Final: MCI={len(feature_names_mci)} features | CN={len(feature_names_cn)} features')
print(f'MCI train: {y_ev_mci_train.mean()*100:.1f}% events | test: {y_ev_mci_test.mean()*100:.1f}%')
print(f'CN  train: {y_ev_cn_train.mean()*100:.1f}%  events | test: {y_ev_cn_test.mean()*100:.1f}%')

### 4.4 Table 1: Baseline Characteristics
Raw ADNIMERGE baseline values for the MCI cohort, converters vs non-converters. Prints LaTeX rows for Table 1.


In [ ]:
# ── 4.4  Table 1: baseline characteristics, converters vs non-converters ─────
# Paste after Section 4.2 (cohort assembly). Uses RAW ADNIMERGE baseline values
# from `df` (not the harmonized / z-scored / imputed features), for the 958
# subjects in the MCI cohort (rids_mci) and their event labels (y_ev_mci).
# Continuous: mean (SD), Mann-Whitney U p-value. Categorical: n (%), chi-square p.
# Prints LaTeX rows; missing counts are printed so you can footnote them.
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

base = df.copy()
if 'VISCODE' in base.columns and (base['VISCODE'] == 'bl').any():
    base = base[base['VISCODE'] == 'bl']
else:   # fall back to each subject's earliest visit
    sort_col = 'EXAMDATE' if 'EXAMDATE' in base.columns else base.columns[0]
    base = base.sort_values(['RID', sort_col])
base = base.drop_duplicates('RID').set_index('RID')

cohort = pd.DataFrame({'event': np.asarray(y_ev_mci, int)}, index=np.asarray(rids_mci))
tab = base.reindex(cohort.index)
tab['event'] = cohort['event']

def _num(s):  # ADNIMERGE stores some CSF values as strings like '>1700'
    return pd.to_numeric(s.astype(str).str.replace(r'[<>]', '', regex=True), errors='coerce')

if 'Hippocampus' in tab and 'ICV' in tab:
    tab['Hippo_ICV_x1000'] = _num(tab['Hippocampus']) / _num(tab['ICV']) * 1000

CONT = [('Age, years', 'AGE'), ('Education, years', 'PTEDUCAT'),
        ('MMSE', 'MMSE'), ('CDR-SB', 'CDRSB'), ('ADAS-Cog 13', 'ADAS13'), ('FAQ', 'FAQ'),
        ('Hippocampus/ICV ($\\times 10^{3}$)', 'Hippo_ICV_x1000'),
        ('CSF amyloid-beta, pg/mL', 'ABETA'), ('CSF total tau, pg/mL', 'TAU'),
        ('CSF p-tau, pg/mL', 'PTAU'), ('FDG-PET', 'FDG'), ('AV45-PET SUVR', 'AV45')]

conv, non = tab[tab.event == 1], tab[tab.event == 0]
print(f'n: all={len(tab)}, converters={len(conv)}, non-converters={len(non)}')
lines = [f'$n$ & {len(tab)} & {len(conv)} & {len(non)} & \\\\']

# Sex
if 'PTGENDER' in tab:
    def _female(d): return (d['PTGENDER'].astype(str).str.lower().str.startswith('f')).sum()
    ct = np.array([[_female(conv), len(conv) - _female(conv)], [_female(non), len(non) - _female(non)]])
    p = chi2_contingency(ct)[1]
    lines.append(f'Female, $n$ (\\%) & {_female(tab)} ({100*_female(tab)/len(tab):.1f}) & '
                 f'{_female(conv)} ({100*_female(conv)/len(conv):.1f}) & '
                 f'{_female(non)} ({100*_female(non)/len(non):.1f}) & {p:.3g} \\\\')
# APOE4 carriers
if 'APOE4' in tab:
    a = _num(tab['APOE4'])
    car = lambda idx: int((a.loc[idx] >= 1).sum())
    kn = lambda idx: int(a.loc[idx].notna().sum())
    ct = np.array([[car(conv.index), kn(conv.index) - car(conv.index)],
                   [car(non.index), kn(non.index) - car(non.index)]])
    p = chi2_contingency(ct)[1]
    lines.append(f'APOE4 carrier, $n$ (\\%) & {car(tab.index)} ({100*car(tab.index)/kn(tab.index):.1f}) & '
                 f'{car(conv.index)} ({100*car(conv.index)/kn(conv.index):.1f}) & '
                 f'{car(non.index)} ({100*car(non.index)/kn(non.index):.1f}) & {p:.3g} \\\\')

missing_notes = []
for label, col in CONT:
    if col not in tab:
        print(f'  (skipped {label}: column {col} not found)')
        continue
    x = _num(tab[col]); xc = x.loc[conv.index].dropna(); xn = x.loc[non.index].dropna()
    p = mannwhitneyu(xc, xn).pvalue if len(xc) and len(xn) else np.nan
    fmt = lambda v: f'{v.mean():.1f} ({v.std():.1f})' if abs(v.mean()) >= 10 else f'{v.mean():.2f} ({v.std():.2f})'
    p_str = '$<$0.001' if p < 0.001 else f'{p:.3f}'
    lines.append(f'{label} & {fmt(x.dropna())} & {fmt(xc)} & {fmt(xn)} & {p_str} \\\\')
    n_miss = int(x.isna().sum())
    if n_miss:
        missing_notes.append(f'{label}: {n_miss} missing ({100*n_miss/len(x):.0f}%)')

print('\n=== Table 1 rows (paste into LaTeX) ===')
print('\n'.join(lines))
print('\nMissing at baseline (for the table note):')
print('; '.join(missing_notes))
(OUT_DIR / 'table1_baseline_rows.tex').write_text('\n'.join(lines) + '\n% Missing: ' + '; '.join(missing_notes) + '\n')


### 4.3 Domain-Separated Feature Sets

Features are partitioned into modality domains using `get_domain_features()` to
quantify the independent predictive signal of each data type before combining them.
This answers the clinical question: *which modality drives conversion risk most?*

| Domain | Features Included |
|--------|------------------|
| `imaging` | ICV-normalized MRI volumes + longitudinal slopes for MRI features |
| `csf_pet` | CSF biomarkers (ABETA, TAU, PTAU), PET (FDG, AV45, FBB), amyloid/tau composites |
| `cognitive` | Cognitive scores (MMSE, CDRSB, ADAS, RAVLT, FAQ, MOCA), composites, EcogSP |
| `combined` | All features (no filtering) |

Demographics (age, sex, education, APOE4), missingness flags, and protocol/scanner
dummies are appended to every non-combined domain as a shared covariate base,
ensuring domain comparisons are not confounded by access to different demographic information.


In [ ]:
from preprocessing import get_domain_features

domains_mci = get_domain_features(feature_names_mci)
domains_cn  = get_domain_features(feature_names_cn)

print('Domain feature counts (MCI cohort):')
for d, feats in domains_mci.items():
    print(f'  {d:<12}: {len(feats)} features')

## 5. Evaluation Framework

### 5.1 Shared Evaluation Utilities

Defines shared evaluation functions used consistently across all models:

**`cv_cindex(X, y_event, y_duration, predict_fn)`**  
5-fold stratified cross-validation returning mean ± std Harrell C-index.
Used for fast model comparison during development; not the primary reported metric.

**`binary_horizon_dataset(y_event, y_duration, horizon_yr)`**  
Converts survival labels into binary outcomes at a fixed prediction horizon.
A subject is labeled 1 if they converted before `horizon_yr`, 0 if they had not
converted by `horizon_yr`. Censored subjects with follow-up shorter than the horizon
are excluded (their outcome is genuinely unknown).

**`horizon_aucs(X, y_event, y_duration, train_predict_fn)`**  
Computes cross-validated AUC-ROC at each horizon in `HORIZONS = [3, 5]` years.
Horizons with fewer than 15 events are skipped to avoid degenerate AUC estimates.

**`results` dict**  
Accumulates `(model_name, cohort) → (C-td, std)` entries throughout training.
Consumed by the final comparison table in Section 7.

> **Primary metric:** IPCW Antolini time-dependent concordance index (C-td) via
> `concordance_td(..., method='adj_antolini', ipcw=True)`. All final model
> comparisons use C-td for consistency across model families.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lifelines.utils import concordance_index
from modeling import cv_cindex, binary_horizon_dataset, horizon_aucs

results = {}

print('Evaluation framework ready.')

### 5.2 Two-Stage CSF Imputation

Predicts missing CSF ABETA values using `build_csf_imputer()`, which trains a LightGBM
regressor on subjects with known ABETA measurements. Predictor features include age, sex,
education, APOE4, ICV-normalized MRI volumes, cognitive scores, and PET measures
(AV45/FBB where available).

**Why two-stage?**  
PET amyloid (AV45/FBB) is available for many subjects who lack CSF data. Since PET
amyloid and CSF ABETA measure the same biological signal via different modalities,
a model trained on subjects with both measurements can generate soft ABETA predictions
for PET-only subjects, recovering predictive signal for approximately 27% of
subjects who would otherwise have a missing ABETA value.

**Evaluation:** Trained on 85% of subjects with known ABETA; validated on a 15%
holdout with RMSE and R² reported. The fitted model is then applied to all
subjects with missing ABETA who have sufficient predictor coverage.


In [ ]:
# How many of the 1205 subjects with missing ABETA actually have AV45?
missing_abeta = df_bl['ABETA'].isna()
has_av45 = df_bl['AV45'].notna()
print(f"Missing ABETA + have AV45: {(missing_abeta & has_av45).sum()}")
print(f"Missing ABETA + no AV45:   {(missing_abeta & ~has_av45).sum()}")

In [ ]:
from modeling import build_csf_imputer

print('Training two-stage CSF imputation model...')
csf_model, csf_predictor_cols = build_csf_imputer(df_bl, target_col='ABETA')

# Apply to subjects with missing ABETA
missing_mask = df_bl['ABETA'].isna()
if missing_mask.sum() > 0:
    X_missing = df_bl.loc[missing_mask, csf_predictor_cols].fillna(
        df_bl[csf_predictor_cols].median())
    df_bl.loc[missing_mask, 'ABETA_predicted'] = csf_model.predict(X_missing)
    df_bl.loc[~missing_mask, 'ABETA_predicted'] = df_bl.loc[~missing_mask, 'ABETA']
    df_bl['ABETA_imputation_flag'] = missing_mask.astype(int)
    print(f'  Predicted ABETA for {missing_mask.sum()} subjects with missing CSF')

# Visualize imputation quality
known = df_bl[df_bl['ABETA'].notna()]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(known['ABETA'], known['ABETA_predicted'], alpha=0.3, s=10)
lims = [known['ABETA'].min(), known['ABETA'].max()]
ax.plot(lims, lims, 'r--', linewidth=1)
ax.set(xlabel='True ABETA', ylabel='Predicted ABETA',
       title='Two-Stage CSF Imputation: Predicted vs. Actual')
plt.tight_layout()
plt.savefig(FIG_DIR / 'csf_imputation_quality.png', dpi=150, bbox_inches='tight')
plt.show()

---
#### CSF Imputation Findings

The 324/891 split confirms that AV45 PET is legitimately useful for recovering
ABETA signal in roughly **27% of subjects with missing CSF data**.

The scatter plot of predicted vs. true ABETA may appear tighter than the R² = 0.41
headline suggests, this is a scaling artifact. ABETA spans a wide range (~200–1700 pg/mL),
so predictions that are off by 300 units still appear visually close on that axis.
The RMSE of ~348 corresponds to roughly **23% relative error** across the full range,
which is acceptable for a soft imputation feature that supplements rather than replaces
direct measurement.

**Take-away:** The CSF imputer adds predictive signal for PET-positive subjects
but should not be treated as equivalent to a measured ABETA value. The `miss_ABETA`
flag is retained as a model feature throughout so downstream models can
learn to discount the imputed values appropriately.


## 6. Baseline Survival Models

### 6.1 Cox Proportional Hazards

The regularized Cox PH model is the interpretable linear baseline. It fits the
proportional hazards model:

$$h(t \mid x) = h_0(t) \cdot \exp(\boldsymbol{\beta}^\top x)$$

where $h_0(t)$ is a shared baseline hazard and $\boldsymbol{\beta}$ is estimated
by maximizing the Cox partial log-likelihood with an elastic net penalty. The linear
form makes coefficients directly interpretable as log-hazard ratios per unit feature change.

**Regularization:** Elastic net penalty controlled by `penalizer` (overall strength)
and `l1_ratio` (0 = pure ridge, 1 = pure LASSO, intermediate = elastic net).
The `l1_ratio` search is capped at 0.5 to prevent full coefficient collapse in
high-dimensional settings. LASSO shrinks uninformative coefficients to exactly zero,
providing implicit feature selection.

**Features are standardized** with `StandardScaler` before fitting so coefficient
magnitudes are directly comparable across features (a requirement for elastic net).

**Tuning:** Optuna, 30 trials, 5-fold stratified CV, Harrell C-index HPO objective.
Final evaluation uses IPCW adj_antolini C-td for consistency with other models.

**Overfitting diagnostics:** Per-fold training C is tracked alongside OOF C for
every Optuna trial. A train-vs-OOF gap > 0.06 in the best trial warrants stronger regularization.

| Output | Variable |
|--------|----------|
| Trained model | `cox_model_mci`, `cox_model_cn` |
| Feature scaler | `cox_scaler_mci`, `cox_scaler_cn` |
| Test survival matrix | `surv_cox_mci`, shape `(n_times × n_test_subjects)` |
| Test C-td | `c_cox_mci` |


In [ ]:
from modeling import run_cox_ph, calc_cox_ph_c, bootstrap_cindex_td, load_checkpoint, save_checkpoint

# Cox PH: drop near-collinear interaction columns
collinear_cols = [
    'age_edu_interaction', 'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio',
    'APOE4_x_amyloid', 'APOE4_x_tau', 'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos',
]
X_mci_train_cox = X_mci_train.drop(columns=[c for c in collinear_cols if c in X_mci_train.columns])
X_mci_test_cox  = X_mci_test.drop(columns=[c for c in collinear_cols if c in X_mci_test.columns])
X_mci_cal_cox   = X_mci_cal.drop(columns=[c for c in collinear_cols if c in X_mci_cal.columns])
X_mci_eval_cox  = X_mci_eval.drop(columns=[c for c in collinear_cols if c in X_mci_eval.columns])
X_cn_train_cox  = X_cn_train.drop(columns=[c for c in collinear_cols if c in X_cn_train.columns])
X_cn_test_cox   = X_cn_test.drop(columns=[c for c in collinear_cols if c in X_cn_test.columns])
print(f'Cox features: MCI={X_mci_train_cox.shape[1]}, CN={X_cn_train_cox.shape[1]}')

if RETRAIN:
    oof_c_cox_mci, train_c_cox_mci, cox_model_mci, cox_scaler_mci, _ = run_cox_ph(
        X_mci_train_cox, y_ev_mci_train, y_dur_mci_train, label='MCI->AD', n_trials=30)
    print(f'MCI OOF C: {oof_c_cox_mci:.4f} | train C: {train_c_cox_mci:.4f} | gap: {train_c_cox_mci-oof_c_cox_mci:+.4f}')
    save_checkpoint('cox_mci', (cox_model_mci, cox_scaler_mci))
    cox_cn_available = False
    try:
        oof_c_cox_cn, train_c_cox_cn, cox_model_cn, cox_scaler_cn, _ = run_cox_ph(
            X_cn_train_cox, y_ev_cn_train, y_dur_cn_train, label='CN->MCI or AD', n_trials=30)
        print(f'CN  OOF C: {oof_c_cox_cn:.4f} | train C: {train_c_cox_cn:.4f} | gap: {train_c_cox_cn-oof_c_cox_cn:+.4f}')
        save_checkpoint('cox_cn', (cox_model_cn, cox_scaler_cn))
        cox_cn_available = True
    except Exception as e:
        print(f'CoxPH CN training failed (expected for low-signal cohort): {e}')
        cox_cn_available = False
else:
    cox_model_mci, cox_scaler_mci = load_checkpoint('cox_mci')
    cox_cn_available = False
    try:
        _ck = load_checkpoint('cox_cn')
        if _ck: cox_model_cn, cox_scaler_cn = _ck; cox_cn_available = True
    except Exception: pass

# ── Evaluate on calibration set (20%) ────────────────────────────────────────
# surv_cox_mci is the survival matrix used for CP computed on X_mci_cal (via alias)
c_td_cox_mci, surv_cox_mci = calc_cox_ph_c(
    cox_model_mci, cox_scaler_mci, X_mci_cal_cox, y_ev_mci_cal, y_dur_mci_cal)
ci_cox_mci = bootstrap_cindex_td(
    y_ev_mci_cal, y_dur_mci_cal, surv_cox_mci, n_bootstrap=500, seed=RANDOM_SEED)
results[('CoxPH', 'MCI->AD')] = (c_td_cox_mci, ci_cox_mci['lower'], ci_cox_mci['upper'])
print(f'CoxPH MCI->AD:      {c_td_cox_mci:.4f}  95% CI [{ci_cox_mci["lower"]:.4f}, {ci_cox_mci["upper"]:.4f}]')

# ── Evaluate on held-out eval set (10%). Needed for Section 9.3/10.3 CP work ──
# Mirrors AFT's aft_surv_eval; without this, conformal calibration for Cox has
# no true held-out set to evaluate coverage on.
c_td_cox_mci_eval, surv_cox_mci_eval = calc_cox_ph_c(
    cox_model_mci, cox_scaler_mci, X_mci_eval_cox, y_ev_mci_eval, y_dur_mci_eval)
print(f'CoxPH MCI->AD (eval set, informational only): {c_td_cox_mci_eval:.4f}')

surv_cox_cn = None
if cox_cn_available:
    c_td_cox_cn, surv_cox_cn = calc_cox_ph_c(
        cox_model_cn, cox_scaler_cn, X_cn_test_cox, y_ev_cn_test, y_dur_cn_test)
    ci_cox_cn = bootstrap_cindex_td(
        y_ev_cn_test, y_dur_cn_test, surv_cox_cn, n_bootstrap=500, seed=RANDOM_SEED)
    results[('CoxPH', 'CN->MCI or AD')] = (c_td_cox_cn, ci_cox_cn['lower'], ci_cox_cn['upper'])
    print(f'CoxPH CN->MCI/AD:   {c_td_cox_cn:.4f}  95% CI [{ci_cox_cn["lower"]:.4f}, {ci_cox_cn["upper"]:.4f}]')
else:
    results[('CoxPH', 'CN->MCI or AD')] = (float('nan'), float('nan'), float('nan'))
    print('CoxPH CN->MCI/AD: training failed, and recorded as NaN in results.')

# ── Feature importance (MCI cohort) ──────────────────────────────────────────
_cox_params = cox_model_mci.params_.copy()
plot_importance(
    _cox_params.abs(),
    title='Cox PH: Feature Importance (MCI → AD)',
    xlabel='|log Hazard Ratio|',
    save_path=FIG_DIR / 'cox_feature_importance_mci.png',
    sign_series=dict(_cox_params),
)

# KM risk-quartile plot
from postprocessing import km_risk_quartile
cox_risk_mci = np.log(surv_cox_mci.values + 1e-9).mean(axis=0) * -1
km_risk_quartile(cox_risk_mci, y_ev_mci_cal, y_dur_mci_cal, 'CoxPH', 'MCI->AD')

if surv_cox_cn is not None:
    cox_risk_cn = np.log(surv_cox_cn.values + 1e-9).mean(axis=0) * -1
    km_risk_quartile(cox_risk_cn, y_ev_cn_test, y_dur_cn_test, 'CoxPH', 'CN->MCI or AD')
else:
    print('CoxPH CN KM: skipped (model unavailable).')


In [ ]:
from postprocessing import plot_individual_survival_curves

i = 56
curve_cox = surv_cox_mci.iloc[:, i]
plot_individual_survival_curves(
    curves=[curve_cox],
    duration=float(y_dur_mci_cal.iloc[i] if hasattr(y_dur_mci_cal, 'iloc') else y_dur_mci_cal[i]), event=int(y_ev_mci_cal[i]),
    labels=['CoxPH'])

In [ ]:
# ── SHAP Feature Importance: Cox PH ──────────────────────────────────────────
# NOTE: explains relative risk (partial hazard), not predicted time. Cox has no
# closed-form time prediction the way AFT does. Compare RANKINGS across models,
# not raw SHAP magnitudes (different models explain different target quantities).
import shap

def cox_model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train_cox.columns)
    X_scaled = pd.DataFrame(cox_scaler_mci.transform(X.values), columns=X.columns)
    return cox_model_mci.predict_partial_hazard(X_scaled).values

cox_explainer = shap.KernelExplainer(
    cox_model_predict, X_mci_train_cox.sample(100, random_state=RANDOM_SEED))
sample_size_cox = min(100, len(X_mci_cal_cox))
X_cal_sample_cox = X_mci_cal_cox.sample(sample_size_cox, random_state=RANDOM_SEED)
shap_values_cox = cox_explainer.shap_values(X_cal_sample_cox)

X_cal_labeled_cox = X_cal_sample_cox.rename(columns={c: feat_label(c) for c in X_cal_sample_cox.columns})

plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values_cox, X_cal_labeled_cox, plot_type='bar', show=False)
plt.title('SHAP Feature Importance: Cox PH Model (MCI → AD)\n(explains relative risk / partial hazard)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_cox_bar.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Cox SHAP expected value (baseline partial hazard): {cox_explainer.expected_value:.4f}')


---
#### Cox PH: What This Means

**Performance summary**

| Cohort | Test C-td | 95% CI | OOF gap |
|--------|-----------|--------|---------|
| MCI → AD | 0.8422 | [0.806, 0.884] | +0.006 |
| CN → MCI or AD | 0.7633 | [0.682, 0.849] | +0.023 |

The MCI → AD result is strong. A C-td of 0.843 means that for 84% of comparable subject pairs, the model correctly ranks the one who converts first as higher risk. The near-zero train/OOF gap (+0.004) confirms the elastic net regularization is working. The model is not memorizing training rank structure.

The CN → MCI or AD result (0.763) is better than expected for this cohort. CN subjects are much earlier in their disease trajectory and the biological signal is inherently weaker, so a gap of 0.08 C-td between cohorts is clinically reasonable rather than a modeling failure. The wider CI (spanning 0.167) reflects the smaller event count in this group and means individual CN predictions carry more uncertainty than MCI predictions.

**What the feature importance chart tells us**

The top features form a clinically coherent picture across three biological axes:

- **Cognitive reserve (blue/protective):** RAVLT Immediate Recall, MoCA, FDG-PET, and Logical Memory II all capture intact memory and metabolic function. Higher scores on these measures push the survival curve upward. Subjects with stronger cognitive reserve at baseline convert later or not at all.

- **Disease burden (red/risk-increasing):** CDR-SB, ADAS-Cog 11/13, FAQ, ECog Study Partner, and AV45-PET all reflect existing impairment or amyloid accumulation. The ECog Study Partner score is notable. Caregiver-reported functional decline is picking up real signal that objective tests sometimes miss.

- **Biology (mixed):** APOE ε4 (risk-increasing) is the strongest known genetic risk factor for late-onset AD, so its presence in the top 5 validates the model is capturing established biology. CSF Amyloid-β and Hippocampal Volume appear as protective (blue) due to measurement scale direction. Higher Amyloid-β pg/mL means less amyloid pathology (the protein is being cleared), and larger hippocampal volume means less atrophy.

**One counterintuitive finding**

Years of Education appears as risk-increasing. This is a known ADNI artifact: higher-education subjects tend to enroll later in their disease course due to cognitive reserve masking early symptoms, so education correlates with shorter observed time to conversion within the study window. It does not mean education causes dementia.

### 6.2 Gradient Boosting Survival Analysis (GBSA)

GBSA is a tree-based survival model from `scikit-survival` that fits an additive
ensemble of regression trees to the negative gradient of the Cox partial log-likelihood:

$$\mathcal{L}(\beta) = \sum_i \left[ x_i^\top \beta - \log\left( \sum_{j:\, t_j \geq t_i} \exp(x_j^\top \beta) \right) \right]$$

Each boosting iteration approximates the log-risk function with a shallow regression tree,
accumulating non-linear feature interactions that the linear Cox model cannot capture.
Unlike DeepSurv, GBSA is tree-based and does not require feature standardization or GPU
acceleration. Unlike Cox PH, it is invariant to monotonic feature transformations.

**HPO search space** (Optuna, 40 trials, 5-fold CV, Antolini C-td objective):

| Parameter | Range |
|-----------|-------|
| `learning_rate` | [0.005, 0.20] log-uniform |
| `n_estimators` | [100, 800] step 50 |
| `max_depth` | [1, 6] |
| `min_samples_split` | [2, 30] |
| `min_samples_leaf` | [1, 30] |
| `max_features` | [0.3, 1.0] |
| `subsample` | [0.5, 1.0] |

**Overfitting diagnostics:** OOF C-td and in-sample C-td are both reported.
A gap > 0.06–0.08 suggests the trees are memorizing training structure and warrants
lower `max_depth` or higher `min_samples_leaf`.

| Output | Variable |
|--------|----------|
| Trained model | `gbsa_model_mci`, `gbsa_model_cn` |
| Feature importances | `imp_mci`: mean decrease in impurity, sorted descending |
| Test survival matrix | `surv_gbsa_mci`: shape `(n_times × n_test_subjects)` |
| Test C-td | `c_gbsa_mci` |


In [ ]:
from sklearn.ensemble._gb import BaseGradientBoosting

# scikit-survival's GradientBoostingSurvivalAnalysis does not pass `criterion`, which newer
# scikit-learn versions require. Fill in sklearn's historical default. Guarded so that
# re-running this cell cannot wrap the init twice.
if not getattr(BaseGradientBoosting.__init__, '_criterion_patched', False):
    _orig_base_init = BaseGradientBoosting.__init__

    def _patched_base_init(self, *args, **kwargs):
        kwargs.setdefault('criterion', 'friedman_mse')
        _orig_base_init(self, *args, **kwargs)

    _patched_base_init._criterion_patched = True
    BaseGradientBoosting.__init__ = _patched_base_init
    print('Patched BaseGradientBoosting to default criterion="friedman_mse".')
else:
    print('BaseGradientBoosting already patched.')


In [ ]:
from modeling import gbsa_survival_cv, calc_gbsa_c, bootstrap_cindex_td

if RETRAIN:
    # gbsa_survival_cv returns: oof_c, train_c, imp, model, study, oof_preds
    oof_c_gbsa_mci, train_c_gbsa_mci, imp_mci, gbsa_model_mci, gbsa_study_mci, _ = gbsa_survival_cv(
        X_mci_train, y_ev_mci_train, y_dur_mci_train,
        feature_names=list(X_mci_train.columns), label='MCI->AD', n_trials=40)
    print(f'MCI OOF C-td: {oof_c_gbsa_mci:.4f} | train: {train_c_gbsa_mci:.4f} | gap: {train_c_gbsa_mci-oof_c_gbsa_mci:+.4f}')
    oof_c_gbsa_cn, train_c_gbsa_cn, imp_cn, gbsa_model_cn, gbsa_study_cn, _ = gbsa_survival_cv(
        X_cn_train, y_ev_cn_train, y_dur_cn_train,
        feature_names=list(X_cn_train.columns), label='CN->MCI or AD', n_trials=40)
    print(f'CN  OOF C-td: {oof_c_gbsa_cn:.4f} | train: {train_c_gbsa_cn:.4f} | gap: {train_c_gbsa_cn-oof_c_gbsa_cn:+.4f}')
    save_checkpoint('gbsa_mci', gbsa_model_mci)
    save_checkpoint('gbsa_cn',  gbsa_model_cn)
    save_checkpoint('gbsa_studies', (gbsa_study_mci, gbsa_study_cn))
else:
    gbsa_model_mci = load_checkpoint('gbsa_mci')
    gbsa_model_cn  = load_checkpoint('gbsa_cn')
    _st = load_checkpoint('gbsa_studies')
    gbsa_study_mci, gbsa_study_cn = _st if _st else (None, None)
    imp_mci = pd.Series(gbsa_model_mci.feature_importances_,
                        index=list(X_mci_train.columns)).sort_values(ascending=False)
    imp_cn  = pd.Series(gbsa_model_cn.feature_importances_,
                        index=list(X_cn_train.columns)).sort_values(ascending=False)

c_td_gbsa_mci, surv_gbsa_mci = calc_gbsa_c(gbsa_model_mci, X_mci_test, y_ev_mci_test, y_dur_mci_test)
c_td_gbsa_cn,  surv_gbsa_cn  = calc_gbsa_c(gbsa_model_cn,  X_cn_test,  y_ev_cn_test,  y_dur_cn_test)

# ── Eval-set survival matrix: needed for CP/CSD (mirrors Cox's fix) ─────────
# X_mci_test is aliased to X_mci_cal (see Section 4.2), so the block above only
# ever gave us a cal-set matrix. Conformal work needs the true held-out eval set.
c_td_gbsa_mci_eval, surv_gbsa_mci_eval = calc_gbsa_c(
    gbsa_model_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)
print(f'GBSA MCI->AD (eval set, informational only): {c_td_gbsa_mci_eval:.4f}')

ci_gbsa_mci = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, surv_gbsa_mci, n_bootstrap=500)
ci_gbsa_cn  = bootstrap_cindex_td(y_ev_cn_test,  y_dur_cn_test,  surv_gbsa_cn,  n_bootstrap=500)
results[('GBSA', 'MCI->AD')]       = (c_td_gbsa_mci, ci_gbsa_mci['lower'], ci_gbsa_mci['upper'])
results[('GBSA', 'CN->MCI or AD')] = (c_td_gbsa_cn,  ci_gbsa_cn['lower'],  ci_gbsa_cn['upper'])
print(f'GBSA MCI->AD:     {c_td_gbsa_mci:.4f}  95% CI [{ci_gbsa_mci["lower"]:.4f}, {ci_gbsa_mci["upper"]:.4f}]')
print(f'GBSA CN->MCI/AD:  {c_td_gbsa_cn:.4f}  95% CI [{ci_gbsa_cn["lower"]:.4f}, {ci_gbsa_cn["upper"]:.4f}]')

# Feature importance (MCI): labeled + color-coded
plot_importance(imp_mci, title='GBSA: Feature Importance (MCI → AD)',
                xlabel='Mean Decrease in Impurity (MDI)',
                save_path=FIG_DIR / 'gbsa_feature_importance_mci.png')

# KM plots: both cohorts
from postprocessing import km_risk_quartile
gbsa_risk_mci = np.log(surv_gbsa_mci.values + 1e-9).mean(axis=0) * -1
km_risk_quartile(gbsa_risk_mci, y_ev_mci_test, y_dur_mci_test, 'GBSA', 'MCI->AD')
gbsa_risk_cn = np.log(surv_gbsa_cn.values + 1e-9).mean(axis=0) * -1
km_risk_quartile(gbsa_risk_cn, y_ev_cn_test, y_dur_cn_test, 'GBSA', 'CN->MCI or AD')


In [ ]:
from postprocessing import plot_individual_survival_curves

i = 56
curve_gbsa = surv_gbsa_mci.iloc[:, i]
plot_individual_survival_curves(
    curves=[curve_gbsa],
    duration=float(y_dur_mci_test.iloc[i] if hasattr(y_dur_mci_test, 'iloc') else y_dur_mci_test[i]), event=int(y_ev_mci_test[i]),
    labels=['GBSA'])

In [ ]:
# ── SHAP Feature Importance: GBSA ────────────────────────────────────────────
# NOTE: explains relative risk score (model.predict), not predicted time.
# TreeExplainer is exact and fast for GBM-based models; falls back to
# KernelExplainer only if the model structure isn't TreeExplainer-compatible.
try:
    gbsa_explainer = shap.TreeExplainer(gbsa_model_mci)
    shap_values_gbsa = gbsa_explainer.shap_values(X_mci_cal)
    X_gbsa_shap_input = X_mci_cal
except Exception as e:
    print(f'TreeExplainer failed ({e}): falling back to KernelExplainer')
    def gbsa_model_predict(X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=X_mci_train.columns)
        return gbsa_model_mci.predict(X)
    gbsa_explainer = shap.KernelExplainer(
        gbsa_model_predict, X_mci_train.sample(100, random_state=RANDOM_SEED))
    X_gbsa_shap_input = X_mci_cal.sample(min(100, len(X_mci_cal)), random_state=RANDOM_SEED)
    shap_values_gbsa = gbsa_explainer.shap_values(X_gbsa_shap_input)

X_gbsa_labeled = X_gbsa_shap_input.rename(columns={c: feat_label(c) for c in X_gbsa_shap_input.columns})

plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values_gbsa, X_gbsa_labeled, plot_type='bar', show=False)
plt.title('SHAP Feature Importance: GBSA Model (MCI → AD)\n(explains relative risk score)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_gbsa_bar.png', dpi=150, bbox_inches='tight')
plt.show()


---
#### GBSA: What This Means

**Performance summary**

| Cohort | Test C-td | 95% CI | OOF gap |
|--------|-----------|--------|---------|
| MCI → AD | 0.8286 | [0.777, 0.869] | +0.046 |
| CN → MCI or AD | 0.7287 | [0.643, 0.830] | +0.099 |

The MCI test C-td of 0.829 is strong with a train/OOF gap of +0.046: a significant
improvement over the prior run (+0.137) where Optuna found an aggressive
`max_depth=6, min_samples_leaf=1` configuration. This run found `max_depth=2,
min_samples_leaf=15`, which is much healthier. The gap reduction confirms the
tightened search space is working as intended. Compared to Cox (0.842 vs 0.829 on
MCI), GBSA is slightly lower and within overlapping CIs.

Compared to Cox (0.842 vs 0.829 on MCI), GBSA is slightly lower and within overlapping confidence intervals. The non-linear interactions GBSA can capture are not adding meaningful signal beyond what the linear Cox model finds on this dataset. This is a common finding in clinical survival analysis where the dominant predictors have approximately monotonic relationships with risk.

**What the feature importance chart tells us**

- **FDG-PET dominates by a wide margin.** Glucose metabolism appears at nearly twice the importance of the next feature. The tree ensemble splits on FDG-PET at every depth level, suggesting a threshold-like relationship that Cox's linear log-hazard cannot fully represent.

- **Executive function measures appear prominently.** mPACC Trails B and mPACC Digit Span rank 7th and 9th, and are absent from Cox's top 20 entirely. Trails B measures processing speed and executive function, a non-linear predictor where subjects below a certain threshold convert much faster. GBSA is detecting this inflection point; Cox is not. This is the most meaningful difference between the two models and suggests executive function carries independent prognostic signal worth capturing in the ensemble.

- **APOE ε4 is nearly absent** despite being 5th in Cox. This is an MDI artifact: the metric is biased toward high-cardinality continuous features because they offer more split points. APOE ε4 is a 0/1/2 count variable, so trees rarely split on it even when it is genuinely predictive. The permutation importance plot in the DeepSurv section provides a less biased view.

- **MRI volumes are well-represented.** Hippocampal, Middle Temporal, Whole Brain, Entorhinal, and Fusiform volumes all appear, consistent with Cox and confirming that structural atrophy carries signal both linear and non-linear models exploit.

### 6.3 Weibull Accelerated Failure Time (AFT)

The Weibull AFT model is a **parametric** survival model that directly models log survival time as a linear function of features:

$$\log T_i = \mu + \boldsymbol{\gamma}^\top x_i + \sigma \epsilon_i$$

where $T_i$ is the survival time, $\sigma$ is the scale parameter, and $\epsilon_i$ follows an extreme value distribution. Unlike Cox, AFT makes a distributional assumption about event times and predicts absolute survival times rather than relative hazard ratios. A positive coefficient on a feature accelerates failure (shortens survival); a negative coefficient delays it.

A 5-fold CV grid search over `penalizer ∈ {0, 0.01, 0.1, 1.0, 5.0}` selects the best L2 regularization strength before fitting the final model.

| Output | Variable |
|--------|----------|
| Trained model | `aft_model_mci`, `aft_model_cn` |
| Test survival matrix | `aft_surv_mci` |
| Test C-td | stored in `results` |

In [ ]:
from lifelines import WeibullAFTFitter
from concordance import concordance_td
from modeling import bootstrap_cindex_td
from sklearn.model_selection import StratifiedKFold as _SKF

def _fit_aft_with_cv(X_tr, y_ev_tr, y_dur_tr, lbl):
    _skf = _SKF(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    _yev  = y_ev_tr.values  if hasattr(y_ev_tr,  'values') else np.array(y_ev_tr)
    _ydur = y_dur_tr.values if hasattr(y_dur_tr, 'values') else np.array(y_dur_tr)
    best_pen, best_c = 0.0, -1.0
    for pen in [0.0, 0.01, 0.1, 1.0, 5.0]:
        _fcs = []
        for tr, va in _skf.split(X_tr, _yev):
            _dtr = X_tr.iloc[tr].copy()
            _dtr['duration'] = _ydur[tr]; _dtr['event'] = _yev[tr]
            _dva = X_tr.iloc[va].copy()
            _dva['duration'] = _ydur[va]; _dva['event'] = _yev[va]
            try:
                _m = WeibullAFTFitter(penalizer=pen)
                _m.fit(_dtr, duration_col='duration', event_col='event')
                _fcs.append(_m.score(_dva, scoring_method='concordance_index'))
            except Exception: pass
        if _fcs:
            _mean = np.mean(_fcs)
            print(f'  [{lbl}] pen={pen:.2f}: CV C={_mean:.4f}')
            if _mean > best_c: best_c, best_pen = _mean, pen
    print(f'  [{lbl}] Best penalizer: {best_pen}  (CV C={best_c:.4f})')
    _data = X_tr.copy()
    _data['duration'] = _ydur; _data['event'] = _yev
    _model = WeibullAFTFitter(penalizer=best_pen)
    _model.fit(_data, duration_col='duration', event_col='event')
    return _model

if RETRAIN:
    aft_model_mci = _fit_aft_with_cv(X_mci_train, y_ev_mci_train, y_dur_mci_train, 'MCI->AD')
    save_checkpoint('aft_mci', aft_model_mci)
    try:
        aft_model_cn = _fit_aft_with_cv(X_cn_train, y_ev_cn_train, y_dur_cn_train, 'CN->MCI or AD')
        save_checkpoint('aft_cn', aft_model_cn)
        aft_cn_available = True
    except Exception as e:
        print(f'AFT CN training failed (expected for low-signal cohort): {e}')
        aft_model_cn = None; aft_cn_available = False
else:
    aft_model_mci = load_checkpoint('aft_mci')
    aft_model_cn  = load_checkpoint('aft_cn')
    aft_cn_available = aft_model_cn is not None

print('--- AFT MCI Model Summary ---')
aft_model_mci.print_summary()

# Feature importance: labeled + color-coded
_aft_p = aft_model_mci.params_['lambda_'].drop('Intercept', errors='ignore')
plot_importance(_aft_p.abs(), title='AFT: Feature Importance (MCI → AD)',
                xlabel='|AFT Coefficient| (lambda_)',
                save_path=FIG_DIR / 'aft_feature_importance_mci.png',
                sign_series=dict(_aft_p))

PROFILE_SUBJECT_IDX = 56

In [ ]:
from postprocessing import plot_individual_survival_curves, km_risk_quartile
_dur_mci = y_dur_mci_cal.values if hasattr(y_dur_mci_cal,'values') else np.array(y_dur_mci_cal)
_ev_mci  = np.array(y_ev_mci_cal, dtype=np.int32)
_test_mci = X_mci_cal.copy()
_test_mci['duration'] = _dur_mci; _test_mci['event'] = _ev_mci
aft_surv_mci  = aft_model_mci.predict_survival_function(_test_mci)
_tg_mci = aft_surv_mci.index.values.astype(np.float64)
_idx_mci = np.clip(np.searchsorted(_tg_mci, _dur_mci), 0, len(_tg_mci)-1).astype(np.int64)
aft_c_mci = concordance_td(_dur_mci, _ev_mci, aft_surv_mci.values, _idx_mci,
                            method='adj_antolini', ipcw=True)
ci_aft_mci = bootstrap_cindex_td(_ev_mci, _dur_mci, aft_surv_mci, n_bootstrap=500)
results[('AFT', 'MCI->AD')] = (aft_c_mci, ci_aft_mci['lower'], ci_aft_mci['upper'])
print(f'AFT MCI->AD:    {aft_c_mci:.4f}  95% CI [{ci_aft_mci["lower"]:.4f}, {ci_aft_mci["upper"]:.4f}]')
# CN cohort
aft_surv_cn = None
if aft_cn_available:
    _dur_cn = y_dur_cn_test.values if hasattr(y_dur_cn_test,'values') else np.array(y_dur_cn_test)
    _ev_cn  = np.array(y_ev_cn_test, dtype=np.int32)
    _test_cn = X_cn_test.copy()
    _test_cn['duration'] = _dur_cn; _test_cn['event'] = _ev_cn
    aft_surv_cn = aft_model_cn.predict_survival_function(_test_cn)
    _tg_cn  = aft_surv_cn.index.values.astype(np.float64)
    _idx_cn = np.clip(np.searchsorted(_tg_cn, _dur_cn), 0, len(_tg_cn)-1).astype(np.int64)
    aft_c_cn = concordance_td(_dur_cn, _ev_cn, aft_surv_cn.values, _idx_cn,
                               method='adj_antolini', ipcw=True)
    ci_aft_cn = bootstrap_cindex_td(_ev_cn, _dur_cn, aft_surv_cn, n_bootstrap=500)
    results[('AFT', 'CN->MCI or AD')] = (aft_c_cn, ci_aft_cn['lower'], ci_aft_cn['upper'])
    print(f'AFT CN->MCI/AD: {aft_c_cn:.4f}  95% CI [{ci_aft_cn["lower"]:.4f}, {ci_aft_cn["upper"]:.4f}]')
else:
    results[('AFT', 'CN->MCI or AD')] = (float('nan'), float('nan'), float('nan'))
    print('AFT CN->MCI/AD: training failed, recorded as NaN.')
# Individual survival curve
curve_aft = aft_surv_mci.iloc[:, PROFILE_SUBJECT_IDX]
plot_individual_survival_curves(
    curves=[curve_aft],
    duration=float(_dur_mci[PROFILE_SUBJECT_IDX]),
    event=int(_ev_mci[PROFILE_SUBJECT_IDX]),
    labels=['AFT'])
# KM: both cohorts
aft_risk_mci = np.log(aft_surv_mci.values + 1e-9).mean(axis=0) * -1
km_risk_quartile(aft_risk_mci, _ev_mci, _dur_mci, 'AFT', 'MCI->AD')
if aft_surv_cn is not None:
    aft_risk_cn = np.log(aft_surv_cn.values + 1e-9).mean(axis=0) * -1
    km_risk_quartile(aft_risk_cn, _ev_cn, _dur_cn, 'AFT', 'CN->MCI or AD')
else:
    print('AFT CN KM: skipped (model unavailable).')

In [ ]:
import shap

def _rename_cols(df):
    return df.rename(columns={c: feat_label(c) for c in df.columns})

def model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train.columns)
    return aft_model_mci.predict_expectation(X).values

explainer = shap.KernelExplainer(model_predict,
                                  X_mci_train.sample(100, random_state=RANDOM_SEED))
sample_size = min(100, len(X_mci_test))
X_test_sample_raw = X_mci_cal.sample(sample_size, random_state=RANDOM_SEED)
shap_values = explainer.shap_values(X_test_sample_raw)

X_test_labeled = _rename_cols(X_test_sample_raw)

plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values, X_test_labeled, plot_type='bar', show=False)
plt.title('SHAP Feature Importance: AFT Model (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_aft_bar.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(11, 9))
shap.summary_plot(shap_values, X_test_labeled, show=False)
plt.title('SHAP Summary: AFT Model (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_aft_summary.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(11, 6))
shap.waterfall_plot(
    shap.Explanation(values=shap_values[0],
                     base_values=explainer.expected_value,
                     data=X_test_labeled.iloc[0].values,
                     feature_names=X_test_labeled.columns.tolist()),
    show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_aft_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'SHAP expected value: {explainer.expected_value:.4f}')

---
#### AFT: What This Means

**Performance summary**

| Cohort | Test C-td | 95% CI | Width |
|--------|-----------|--------|-------|
| MCI → AD | 0.8359 | [0.787, 0.888] | 0.101 |
| CN → MCI or AD | 0.7719 | [0.692, 0.849] | 0.157 |

The MCI result (0.836) sits between Cox (0.843) and GBSA (0.823), and all three overlap
within their confidence intervals. No model is statistically distinguishable from the
others on MCI. The CN result (0.772) is the strongest of any individual model on that
cohort, edging out Cox (0.763) and notably outperforming GBSA (0.729). This suggests
the Weibull parametric assumption fits the CN → MCI/AD transition reasonably well. The time-to-event distribution in earlier-stage disease may be more cleanly Weibull-shaped
than in the MCI cohort where competing risks and heterogeneous trajectories add noise.

**What the SHAP chart tells us**

SHAP provides a model-agnostic view of feature contributions to individual predictions,
making it the most interpretable importance measure of the three models so far.
All bars are the same color here because SHAP magnitude does not encode direction.
The dot plot (summary plot) below shows directionality.

The top features are clinically coherent and largely consistent with Cox:

- **RAVLT Immediate Recall and MoCA** dominate, both capturing intact episodic memory
  and global cognition. High scores on these extend predicted survival time substantially. The AFT coefficient interpretation is direct: a unit increase in RAVLT Immediate Recall
  adds measurable years to predicted time-to-conversion.

- **Both amyloid measures appear in the top 5**: AV45-PET (Amyloid Burden) and
  CSF Amyloid-β (pg/mL). This is the clearest signal yet that amyloid pathology is
  driving conversion timing, and the AFT model is capturing it from two independent
  measurement modalities. The fact that both appear with similar magnitude suggests
  they are not simply redundant, and each contributes independent information.

- **FDG-PET ranks 4th**, consistent with its dominant role in GBSA. Glucose metabolism
  is a strong predictor regardless of whether the model is linear, tree-based, or
  parametric.

- **APOE ε4 reappears prominently (7th)**, recovering from its near-absence in GBSA's
  MDI chart. This confirms the GBSA result was an MDI artifact. APOE ε4 is genuinely
  predictive and SHAP, which permutes feature values rather than counting splits,
  captures its contribution correctly.

- **CDR-SB ranks 8th** rather than near the top as in Cox. AFT models time directly
  rather than ranking subjects, so a feature that sharply stratifies fast vs. slow
  converters (CDR-SB) has somewhat less leverage here than in a pure ranking model.

- **Sex (Male=1) appears for the first time** in any model's top 20. This is a real
  finding. Sex differences in AD conversion timing are well-documented, with women
  showing faster cognitive decline after MCI onset on average. The AFT model's
  parametric structure is sensitive enough to detect this signal that Cox and GBSA
  both missed.

### 6.4 DeepSurv: Neural Cox Proportional Hazards

DeepSurv replaces the linear log-risk term $\boldsymbol{\beta}^\top x$ in Cox PH
with a multilayer perceptron (MLP), learning a non-linear risk representation:

$$h(t \mid x) = h_0(t) \cdot \exp(f_\theta(x))$$

where $f_\theta$ is trained by backpropagating the Cox partial log-likelihood.
This captures complex feature interactions and non-monotonic relationships that
the linear Cox model cannot represent, at the cost of interpretability.

**Architecture:** Fully connected layers with ReLU activations, batch normalization,
dropout, and a single linear output node. Input features are standardized to zero
mean and unit variance via `StandardScaler` (fitted on the training split only to
prevent leakage of validation statistics).

**HPO search space** (Optuna, 25 trials, 5-fold CV, C-td objective):

| Parameter | Options / Range |
|-----------|----------------|
| Architecture | 6 MLP options: `[32,32]` to `[256,256,128]` |
| `dropout` | [0.05, 0.50] uniform |
| `lr` (learning rate) | [5e-5, 5e-2] log-uniform |
| `wd` (weight decay / L2) | [1e-5, 1e-2] log-uniform |
| `batch_size` | {32, 64, 128, 256} |

**Overfitting safeguards:** Early stopping (`patience=10`) per CV fold; dropout and
weight decay jointly tuned; scaler fitted only on training fold in each split.
Final model uses an 80/20 stratified split with `patience=15`.

| Output | Variable |
|--------|----------|
| Trained model | `ds_model_mci` (PycoxCoxPH with baseline hazards computed) |
| Feature scaler | `ds_scaler_mci` |
| Training loss history | `ds_loss_mci`: `{'train': [...], 'val': [...]}` |
| Test survival matrix | `surv_mci`: shape `(n_times × n_test_subjects)` |
| Test C-td | `ds_c_mci` |


In [ ]:
import importlib, modeling
from modeling import run_deepsurv, calc_deepsurv_c, bootstrap_cindex_td

if RETRAIN:
    print('Training DeepSurv: MCI cohort...')
    # run_deepsurv returns: oof_c, train_c, model, scaler, loss_history, study, oof_preds
    oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci, ds_study_mci, _ = run_deepsurv(
        X_mci_train, y_ev_mci_train, y_dur_mci_train, 'MCI->AD', n_trials=25)
    print(f'MCI OOF C-td: {oof_c_ds_mci:.4f} | train: {train_c_ds_mci:.4f} | gap: {train_c_ds_mci-oof_c_ds_mci:+.4f}')
    ds_model_mci.callbacks = []
    save_checkpoint('deepsurv_mci_metadata',
                    (oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci))
    save_checkpoint('deepsurv_mci_study', ds_study_mci)
    ds_cn_available = False
    try:
        print('Training DeepSurv: CN cohort...')
        oof_c_ds_cn, train_c_ds_cn, ds_model_cn, ds_scaler_cn, ds_loss_cn, ds_study_cn, _ = run_deepsurv(
            X_cn_train, y_ev_cn_train, y_dur_cn_train, 'CN->MCI or AD', n_trials=25)
        print(f'CN  OOF C-td: {oof_c_ds_cn:.4f} | train: {train_c_ds_cn:.4f} | gap: {train_c_ds_cn-oof_c_ds_cn:+.4f}')
        ds_model_cn.callbacks = []
        save_checkpoint('deepsurv_cn_metadata',
                        (oof_c_ds_cn, train_c_ds_cn, ds_model_cn, ds_scaler_cn, ds_loss_cn))
        ds_cn_available = True
    except Exception as e:
        print(f'DeepSurv CN training failed (expected for low-signal cohort): {e}')
        ds_model_cn = None; ds_scaler_cn = None; ds_loss_cn = {}; ds_cn_available = False
else:
    _ck = load_checkpoint('deepsurv_mci_metadata')
    oof_c_ds_mci, train_c_ds_mci, ds_model_mci, ds_scaler_mci, ds_loss_mci = _ck
    _X_tr_ck = ds_scaler_mci.transform(X_mci_train.values).astype('float32')
    _dur_ck = y_dur_mci_train.values if hasattr(y_dur_mci_train,'values') else np.array(y_dur_mci_train)
    _ev_ck  = y_ev_mci_train.values  if hasattr(y_ev_mci_train, 'values') else np.array(y_ev_mci_train)
    ds_model_mci.compute_baseline_hazards(
        input=_X_tr_ck,
        target=(_dur_ck.astype('float32'), _ev_ck.astype('float32')),
        batch_size=256)
    ds_study_mci = load_checkpoint('deepsurv_mci_study')
    _ck_cn = load_checkpoint('deepsurv_cn_metadata')
    if _ck_cn:
        oof_c_ds_cn, train_c_ds_cn, ds_model_cn, ds_scaler_cn, ds_loss_cn = _ck_cn
        _X_tr_cn_ck = ds_scaler_cn.transform(X_cn_train.values).astype('float32')
        _dur_cn_ck = y_dur_cn_train.values if hasattr(y_dur_cn_train,'values') else np.array(y_dur_cn_train)
        _ev_cn_ck  = y_ev_cn_train.values  if hasattr(y_ev_cn_train, 'values') else np.array(y_ev_cn_train)
        ds_model_cn.compute_baseline_hazards(
            input=_X_tr_cn_ck,
            target=(_dur_cn_ck.astype('float32'), _ev_cn_ck.astype('float32')),
            batch_size=256)
        ds_cn_available = True
    else:
        ds_model_cn = None; ds_scaler_cn = None; ds_loss_cn = {}; ds_cn_available = False

# ── Evaluate MCI (cal set, via X_mci_test alias) ─────────────────────────────
ds_c_mci, surv_mci = calc_deepsurv_c(
    ds_model_mci, ds_scaler_mci, X_mci_test, y_ev_mci_test, y_dur_mci_test)
ci_ds_mci = bootstrap_cindex_td(y_ev_mci_test, y_dur_mci_test, surv_mci, n_bootstrap=500)
results[('DeepSurv', 'MCI->AD')] = (ds_c_mci, ci_ds_mci['lower'], ci_ds_mci['upper'])
print(f'DeepSurv MCI->AD:    {ds_c_mci:.4f}  95% CI [{ci_ds_mci["lower"]:.4f}, {ci_ds_mci["upper"]:.4f}]')

# ── Eval-set survival matrix: needed for CP/CSD─────────
ds_c_mci_eval, surv_mci_eval = calc_deepsurv_c(
    ds_model_mci, ds_scaler_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)
print(f'DeepSurv MCI->AD (eval set, informational only): {ds_c_mci_eval:.4f}')

# ── Evaluate CN ──────────────────────────────────────────────────────────
surv_cn_ds = None
if ds_cn_available:
    ds_c_cn, surv_cn_ds = calc_deepsurv_c(
        ds_model_cn, ds_scaler_cn, X_cn_test, y_ev_cn_test, y_dur_cn_test)
    ci_ds_cn = bootstrap_cindex_td(y_ev_cn_test, y_dur_cn_test, surv_cn_ds, n_bootstrap=500)
    results[('DeepSurv', 'CN->MCI or AD')] = (ds_c_cn, ci_ds_cn['lower'], ci_ds_cn['upper'])
    print(f'DeepSurv CN->MCI/AD: {ds_c_cn:.4f}  95% CI [{ci_ds_cn["lower"]:.4f}, {ci_ds_cn["upper"]:.4f}]')
else:
    results[('DeepSurv', 'CN->MCI or AD')] = (float('nan'), float('nan'), float('nan'))
    print('DeepSurv CN->MCI/AD: training failed. Recorded as NaN.')

# ── Permutation importance (MCI, labeled) ────────────────────────────────
from pycox.evaluation import EvalSurv as _ES
_X_ts = ds_scaler_mci.transform(X_mci_test.values).astype('float32')
_bs   = ds_model_mci.predict_surv_df(_X_ts)
_bc   = _ES(_bs, np.array(y_dur_mci_test, dtype=np.float64),
             np.array(y_ev_mci_test, dtype=bool)).concordance_td()
_rng  = np.random.default_rng(RANDOM_SEED)
_drops = {}
for _j, _feat in enumerate(X_mci_test.columns):
    _Xp = _X_ts.copy(); _Xp[:, _j] = _rng.permutation(_Xp[:, _j])
    _sp = ds_model_mci.predict_surv_df(_Xp)
    _drops[_feat] = _bc - _ES(_sp, np.array(y_dur_mci_test, dtype=np.float64),
                               np.array(y_ev_mci_test, dtype=bool)).concordance_td()
ds_imp = pd.Series(_drops).sort_values(ascending=False)
plot_importance(ds_imp, title='DeepSurv: Permutation Importance (MCI → AD)',
                xlabel='Drop in C-td when feature permuted',
                save_path=FIG_DIR / 'deepsurv_feature_importance_mci.png')


In [ ]:
from postprocessing import plot_individual_survival_curves, km_risk_quartile

i = 56
curve_ds = surv_mci.iloc[:, i]
plot_individual_survival_curves(
    curves=[curve_ds],
    duration=float(y_dur_mci_test.iloc[i] if hasattr(y_dur_mci_test,'iloc') else y_dur_mci_test[i]),
    event=int(y_ev_mci_test[i]),
    labels=['DeepSurv'])

ds_risk_mci = np.log(surv_mci.values + 1e-9).mean(axis=0) * -1
km_risk_quartile(ds_risk_mci, y_ev_mci_test, y_dur_mci_test, 'DeepSurv', 'MCI->AD')

if surv_cn_ds is not None:
    ds_risk_cn = np.log(surv_cn_ds.values + 1e-9).mean(axis=0) * -1
    km_risk_quartile(ds_risk_cn, y_ev_cn_test, y_dur_cn_test, 'DeepSurv', 'CN->MCI or AD')
else:
    print('DeepSurv CN KM: skipped (model unavailable).')


In [ ]:
# ── SHAP Feature Importance: DeepSurv ────────────────────────────────────────
# NOTE: explains relative risk score (raw net output), not predicted time
# KernelExplainer used since DeepExplainer requires direct torch access that
# pycox's wrapper doesn't expose cleanly.
def ds_model_predict(X):
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=X_mci_train.columns)
    X_scaled = ds_scaler_mci.transform(X.values).astype('float32')
    return ds_model_mci.predict(X_scaled).flatten()

ds_explainer = shap.KernelExplainer(
    ds_model_predict, X_mci_train.sample(100, random_state=RANDOM_SEED))
sample_size_ds = min(100, len(X_mci_cal))
X_cal_sample_ds = X_mci_cal.sample(sample_size_ds, random_state=RANDOM_SEED)
shap_values_ds = ds_explainer.shap_values(X_cal_sample_ds)

X_ds_labeled = X_cal_sample_ds.rename(columns={c: feat_label(c) for c in X_cal_sample_ds.columns})

plt.figure(figsize=(11, 7))
shap.summary_plot(shap_values_ds, X_ds_labeled, plot_type='bar', show=False)
plt.title('SHAP Feature Importance: DeepSurv Model (MCI → AD)\n(explains relative risk score)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_deepsurv_bar.png', dpi=150, bbox_inches='tight')
plt.show()


---
#### DeepSurv: What This Means

**Performance summary**

| Cohort | Test C-td | 95% CI | OOF gap |
|--------|-----------|--------|---------|
| MCI → AD | 0.8231 | [0.775, 0.863] | +0.028 |
| CN → MCI or AD | 0.7491 | [0.659, 0.842] | +0.100 |

The MCI result (0.823) is the lowest of the four models this run, with a
moderate train/OOF gap (+0.028). The best trial found `arch_idx=3`
([256,128,64]), `dropout=0.22`, `lr=0.0014`, `batch=256`: a deeper
architecture than the prior run's [32,32], which contributed to slightly
more overfitting. Cox PH leads on MCI (0.842 vs 0.823), though CIs overlap.
The CN result (0.749) falls between AFT (0.772) and GBSA (0.729), consistent
with prior runs.

**What the permutation importance chart tells us**

Permutation importance is the most honest of the three importance measures used across
models. It directly measures how much the model's ranking ability degrades when a
feature's information is destroyed, rather than counting splits or computing SHAP
attributions.

- **FDG-PET and FAQ dominate jointly.** These are the two features whose permutation
  causes the largest drop in C-td. They are roughly equal in magnitude and both well above
  everything else. The neural network is learning a non-linear interaction between
  glucose metabolism and functional activities that neither Cox nor GBSA emphasized
  as strongly. FAQ captures what the patient can no longer do independently; FDG-PET
  captures why, and hypometabolism in the regions responsible for those functions.
  The joint importance suggests the network is weighting their combination rather
  than either alone.

- **APOE ε4 interaction terms are prominent in a way they weren't in Cox or GBSA.**
  APOE ε4 × Tau Burden ranks 4th and APOE ε4 × Amyloid Positivity ranks 6th-
  both higher than the raw APOE ε4 count alone. This is the clearest signal yet
  that the gene's risk is not additive but multiplicative with downstream pathology.
  Carrying APOE ε4 in the presence of elevated tau or amyloid positivity accelerates
  conversion substantially more than either risk factor alone. The neural network is
  capturing this interaction; the linear Cox model collapsed it into the raw APOE ε4
  coefficient.

- **Hippocampal Volume ranks 5th**: higher than in GBSA and consistent with Cox.
  Structural atrophy is a stable signal across all model families.

- **Age and Education both appear**, unlike in Cox or GBSA where they were low or
  absent. The network is learning that these demographic factors interact with
  biomarkers in ways that affect conversion timing. Older subjects with less
  education convert faster even after controlling for biomarker levels.

- **CSF Amyloid-β ranks last** despite being 5th in the SHAP chart for AFT. This
  is not a contradiction. Permutation importance measures the marginal contribution
  of each feature given all others are present. With AV45-PET (which measures the
  same biology) already in the model, CSF Amyloid-β adds little additional
  discriminative signal. The AFT SHAP result reflected absolute attribution including
  the shared variance; permutation importance reflects the unique variance.

In [ ]:
"""
Two panels per model × cohort:

  Left: Optimization history, a scatter of every trial's CV C-td plus the
           running best (red line). A still-rising best curve at trial N
           suggests increasing n_trials. A flat curve after ~10 trials
           indicates the search space is saturated.

  Right: Hyperparameter importance (fANOVA) a fraction of C-td variance
           explained by each parameter. Dominance by a single param (e.g. 'lr')
           signals the others are at appropriate defaults or the space is too
           narrow to matter.
"""

def plot_optuna_diagnostics(study, model_name, cohort_label, fig_dir):
    if study is None:
        print(f'  [{model_name} | {cohort_label}] Study not available '
              f'(loaded from checkpoint without study object).')
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Optimization history
    ax = axes[0]
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    ax.scatter(range(len(values)), values,
               alpha=0.45, s=22, color='#95a5a6', label='Trial C-td', zorder=2)
    ax.plot(range(len(best_so_far)), best_so_far,
            color='#e74c3c', lw=2, label='Best so far', zorder=3)
    ax.axhline(values[0], color='#bdc3c7', ls=':', lw=1)   # baseline (trial 0)
    ax.set(xlabel='Trial', ylabel='CV C-td (Antolini)',
           title=f'{model_name} [{cohort_label}]: Optimisation History')
    ax.legend(fontsize=9)

    # Hyperparameter importance (fANOVA) 
    ax2 = axes[1]
    try:
        importances = optuna.importance.get_param_importances(study)
        # Top 8 params by importance
        top_params = list(importances.keys())[:8]
        top_vals   = [importances[p] for p in top_params]
        colors = ['#3498db' if v == max(top_vals) else '#85c1e9' for v in top_vals]
        ax2.barh(top_params[::-1], top_vals[::-1], color=colors[::-1], alpha=0.9)
        ax2.set(xlabel='Importance (fANOVA fraction)',
                title=f'{model_name} [{cohort_label}]: Param Importance')
        ax2.axvline(0, color='black', lw=0.5)
    except Exception as e:
        ax2.text(0.5, 0.5,
                 f'Importance unavailable\n({e})',
                 ha='center', va='center', transform=ax2.transAxes,
                 fontsize=10, color='gray')

    plt.tight_layout()
    tag = f'{model_name.lower().replace(" ", "_")}_{cohort_label[:3].lower()}'
    plt.savefig(fig_dir / f'optuna_diag_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# GBSA
plot_optuna_diagnostics(gbsa_study_mci, 'GBSA', 'MCI->AD', FIG_DIR)
plot_optuna_diagnostics(gbsa_study_cn,  'GBSA', 'CN-> MCI or AD',   FIG_DIR)

# DeepSurv
plot_optuna_diagnostics(ds_study_mci, 'DeepSurv', 'MCI->AD', FIG_DIR)
if ds_cn_available:
    plot_optuna_diagnostics(ds_study_cn, 'DeepSurv', 'CN-> MCI or AD', FIG_DIR)
else:
    print('DeepSurv CN study unavailable, skipped.')


In [ ]:
from postprocessing import plot_individual_survival_curves

i = 56
plot_individual_survival_curves(
    curves=[
        surv_gbsa_mci.iloc[:, i],
        aft_surv_mci.iloc[:, i],
        surv_mci.iloc[:, i],
        surv_cox_mci.iloc[:, i],
    ],
    duration=float(y_dur_mci_test.iloc[i] if hasattr(y_dur_mci_test, 'iloc') else y_dur_mci_test[i]),
    event=int(y_ev_mci_test[i]),
    labels=['GBSA', 'AFT', 'DeepSurv', 'Cox PH'],
    title='Model Comparison: Survival Curves (MCI->AD)')

### 6.5 Individual Survival Curve Comparison

Plots the full predicted survival function $S(t \mid x)$ from all four models
overlaid for a single held-out subject. The vertical dashed line marks the subject's
actual observed duration; the marker shape indicates whether the event was observed
(◆ diamond) or censored (○ open circle).

**What to look for:**
- Models that agree on the general shape of the survival curve add corroborating evidence
- Large disagreements in predicted survival probability at the subject's event time
  indicate models have learned different risk signals
- A model whose $S(t)$ is high at the event time (close to 1.0 when the event occurs)
  indicates poor calibration for that subject


### 6.6 Cross-Model Feature Importance Comparison (SHAP)

Compares SHAP-based feature rankings across all four models on their common
feature set (Cox PH uses a reduced feature set, see 6.1, hat excludes a few
collinear columns present in the other three models' full feature set).

**Important caveat:** AFT's SHAP values explain *predicted survival time*
(`predict_expectation`), while Cox PH, GBSA, and DeepSurv's SHAP values explain
a *relative risk score* (partial hazard / model risk output). These are
different target quantities on different scales. This comparison is valid for
**feature ranking** (which features matter most, and in what relative order)
but the raw SHAP magnitudes are not directly comparable across models.

In [ ]:
# ── Cross-Model SHAP Comparison ───────────────────────────────────────────────
from scipy.stats import spearmanr

common_features = [c for c in X_mci_train_cox.columns if c in X_mci_train.columns]

def mean_abs_shap(shap_vals, feature_names, common):
    s = pd.Series(np.abs(shap_vals).mean(axis=0), index=feature_names)
    return s.reindex(common)

# AFT's shap_values/X_test_sample_raw were computed in Section 6.3 and remain
# in the kernel namespace at this point in a top-to-bottom run.
shap_values_aft = shap_values

imp_shap_aft  = mean_abs_shap(shap_values_aft,  X_test_sample_raw.columns, common_features)
imp_shap_cox  = mean_abs_shap(shap_values_cox,  X_cal_sample_cox.columns,  common_features)
imp_shap_gbsa = mean_abs_shap(shap_values_gbsa, X_gbsa_shap_input.columns, common_features)
imp_shap_ds   = mean_abs_shap(shap_values_ds,   X_cal_sample_ds.columns,   common_features)

shap_comparison = pd.DataFrame({
    'AFT': imp_shap_aft,
    'Cox PH': imp_shap_cox,
    'GBSA': imp_shap_gbsa,
    'DeepSurv': imp_shap_ds,
})

# Rank within each model (1 = most important)
shap_rank = shap_comparison.rank(ascending=False, method='min')

print('Mean |SHAP value| by feature and model (common feature set):')
print(shap_comparison.round(4).to_string())
print('\nRank (1 = most important) by feature and model:')
print(shap_rank.astype(int).to_string())

# Heatmap of ranks
fig, ax = plt.subplots(figsize=(8, max(6, len(common_features) * 0.35)))
im = ax.imshow(shap_rank.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(shap_rank.columns)))
ax.set_xticklabels(shap_rank.columns)
ax.set_yticks(range(len(shap_rank.index)))
ax.set_yticklabels([feat_label(f) for f in shap_rank.index])
for i in range(shap_rank.shape[0]):
    for j in range(shap_rank.shape[1]):
        ax.text(j, i, int(shap_rank.values[i, j]), ha='center', va='center', fontsize=8)
plt.colorbar(im, label='Rank (1 = most important)')
plt.title('SHAP Feature Rank Comparison: All 4 Models')
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_rank_comparison_all4.png', dpi=150, bbox_inches='tight')
plt.show()

# Spearman rank correlation between models' SHAP-based feature rankings
model_names = shap_rank.columns.tolist()
corr_matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for m1 in model_names:
    for m2 in model_names:
        corr_matrix.loc[m1, m2] = spearmanr(shap_rank[m1], shap_rank[m2]).correlation

print('\nSpearman rank correlation between models (SHAP-based feature rankings):')
print(corr_matrix.round(3).to_string())


In [ ]:
# ── 6.7  Feature Importance by Family ────────────────────────────────────────
FEATURE_FAMILIES = {
    'Demographics': ['AGE', 'PTGENDER_num', 'PTEDUCAT'],
    'Genetic / APOE': ['APOE4'],
    'APOE4 Interactions': ['APOE4_x_amyloid', 'APOE4_x_tau',
                            'APOE4_x_hippo_ICV', 'APOE4_amyloid_pos'],
    'Cognitive Assessments': ['MMSE', 'CDRSB', 'ADAS11', 'ADAS13', 'LDELTOTAL',
                               'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting',
                               'FAQ', 'MOCA', 'EcogPtTotal', 'EcogSPTotal',
                               'mPACCdigit', 'mPACCtrailsB'],
    'MRI Volumetric (Imaging)': ['Hippocampus_ICV', 'Entorhinal_ICV', 'Ventricles_ICV',
                                  'Fusiform_ICV', 'MidTemp_ICV', 'WholeBrain_ICV',
                                  'hippo_entorhinal_ratio', 'midtemp_fusiform_ratio'],
    'CSF/PET Biomarkers': ['ABETA', 'TAU', 'PTAU', 'FDG', 'AV45'],
}

def feature_to_family(feat):
    for family, feats in FEATURE_FAMILIES.items():
        if feat in feats:
            return family
    return 'Other'  # safety net, should be empty if families are complete

def family_shap_totals(shap_vals, feature_names):
    """Sum of mean(|SHAP|) per family, using this model's own full feature set."""
    per_feature = pd.Series(np.abs(shap_vals).mean(axis=0), index=feature_names)
    per_feature_df = per_feature.to_frame('mean_abs_shap')
    per_feature_df['family'] = [feature_to_family(f) for f in per_feature_df.index]
    return per_feature_df.groupby('family')['mean_abs_shap'].sum()

family_totals = pd.DataFrame({
    'AFT':      family_shap_totals(shap_values_aft,  X_test_sample_raw.columns),
    'Cox PH':   family_shap_totals(shap_values_cox,  X_cal_sample_cox.columns),
    'GBSA':     family_shap_totals(shap_values_gbsa, X_gbsa_shap_input.columns),
    'DeepSurv': family_shap_totals(shap_values_ds,   X_cal_sample_ds.columns),
})
family_totals = family_totals.reindex(FEATURE_FAMILIES.keys()).fillna(0)  

# Check for any features that fell through to 'Other', should not happen
for model_name, feats in [('AFT', X_test_sample_raw.columns),
                           ('Cox PH', X_cal_sample_cox.columns),
                           ('GBSA', X_gbsa_shap_input.columns),
                           ('DeepSurv', X_cal_sample_ds.columns)]:
    unassigned = [f for f in feats if feature_to_family(f) == 'Other']
    if unassigned:
        print(f'WARNING: {model_name} has unassigned features: {unassigned}')

print('Sum of mean(|SHAP value|) by feature family and model:')
print(family_totals.round(4).to_string())

# Normalize to % of each model's total SHAP mass, for a fairer cross-model
# comparison (raw magnitudes aren't comparable across models, see 6.6 caveat)
family_pct = family_totals.div(family_totals.sum(axis=0), axis=1) * 100
print('\nAs % of each model\'s total SHAP mass:')
print(family_pct.round(1).to_string())

# ── Grouped bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(family_pct.index))
width = 0.2
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
for i, model in enumerate(family_pct.columns):
    ax.bar(x + (i - 1.5) * width, family_pct[model], width, label=model, color=colors[i])
ax.set(xticks=x, xticklabels=family_pct.index,
       ylabel='% of model\'s total SHAP mass',
       title='Feature Importance by Family: All 4 Models (MCI → AD)')
plt.xticks(rotation=20, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'shap_family_comparison_all4.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Rank of families within each model ────────────────────────────────────────
family_rank = family_pct.rank(ascending=False, method='min')
print('\nFamily rank within each model (1 = most important):')
print(family_rank.astype(int).to_string())

## 7. Evaluation and Results

All metrics in this section are computed on the **held-out 20% test set**, which
was withheld from all training, cross-validation, and hyperparameter selection.
No model hyperparameter was selected based on test-set performance.

**Reported metric:** IPCW Antolini time-dependent C-td (`method='adj_antolini', ipcw=True`)
for all four models, evaluated consistently via `concordance_td`.


In [ ]:
import math

rows = []
for (model, cohort), metrics in results.items():
    c_pt  = metrics[0]
    ci_lo = metrics[1] if len(metrics) > 1 else float('nan')
    ci_hi = metrics[2] if len(metrics) > 2 else float('nan')
    if math.isnan(c_pt):
        c_str, ci_str, w_str = 'FAILED', '—', '—'
    else:
        c_str  = f'{c_pt:.4f}'
        ci_str = f'[{ci_lo:.4f}, {ci_hi:.4f}]'
        w_str  = f'{ci_hi - ci_lo:.4f}'
    rows.append({'Model': model, 'Cohort': cohort,
                 'C-td': c_str, '95% CI': ci_str, 'CI Width': w_str})

results_df = pd.DataFrame(rows)
# Sort: MCI first, CN second; within cohort by C-td descending (FAILED last)
results_df['_sort_cohort'] = results_df['Cohort'].map({'MCI->AD': 0, 'CN->MCI or AD': 1})
results_df['_sort_c'] = pd.to_numeric(results_df['C-td'], errors='coerce').fillna(-1)
results_df = results_df.sort_values(['_sort_cohort', '_sort_c'],
                                     ascending=[True, False]).drop(columns=['_sort_cohort','_sort_c'])

print('\n' + '='*75)
print('FINAL MODEL COMPARISON (held-out 20% test set)')
print('='*75)
print(results_df.to_string(index=False))
print('\nNote: FAILED = model did not converge on this cohort (expected for CN->MCI/AD).')
results_df.to_csv(OUT_DIR / 'model_comparison.csv', index=False)

# Bar chart: MCI cohort only (skip NaN/FAILED rows)
mci_res = results_df[(results_df['Cohort'].str.contains('MCI')) &
                      (results_df['C-td'] != 'FAILED')].copy()
mci_res['c_float'] = mci_res['C-td'].astype(float)
mci_res['lo'] = mci_res['95% CI'].str.extract(r'\[([0-9.]+)').astype(float)
mci_res['hi'] = mci_res['95% CI'].str.extract(r', ([0-9.]+)\]').astype(float)
mci_res = mci_res.sort_values('c_float')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(mci_res['Model'], mci_res['c_float'],
               color='#3498db', alpha=0.75, edgecolor='white', height=0.55)
ax.errorbar(mci_res['c_float'], mci_res['Model'],
            xerr=[(mci_res['c_float'] - mci_res['lo']).values,
                  (mci_res['hi'] - mci_res['c_float']).values],
            fmt='none', color='#2c3e50', capsize=4, lw=2, label='95% bootstrap CI')
ax.axvline(0.5, color='red', ls='--', alpha=0.5, label='Random (0.5)')
ax.set(xlabel='C-td (IPCW Antolini)', title='Model Comparison: MCI → AD', xlim=(0.4, 1.0))
for bar, val in zip(bars, mci_res['c_float']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison_mci.png', dpi=150, bbox_inches='tight')
plt.show()

# Separate panel: CN cohort, shows attempted models + outcomes
cn_res = results_df[results_df['Cohort'].str.contains('CN')].copy()
cn_res['c_float'] = pd.to_numeric(cn_res['C-td'], errors='coerce')
cn_success = cn_res.dropna(subset=['c_float'])
cn_failed  = cn_res[cn_res['C-td'] == 'FAILED']

fig, ax = plt.subplots(figsize=(10, 4))
if not cn_success.empty:
    cn_success = cn_success.sort_values('c_float')
    bars2 = ax.barh(cn_success['Model'], cn_success['c_float'],
                    color='#e67e22', alpha=0.75, edgecolor='white', height=0.55)
    cn_lo = cn_success['95% CI'].str.extract(r'\[([0-9.]+)').astype(float).values.flatten()
    cn_hi = cn_success['95% CI'].str.extract(r', ([0-9.]+)\]').astype(float).values.flatten()
    ax.errorbar(cn_success['c_float'], cn_success['Model'],
                xerr=[(cn_success['c_float'].values - cn_lo),
                       (cn_hi - cn_success['c_float'].values)],
                fmt='none', color='#2c3e50', capsize=4, lw=2, label='95% bootstrap CI')
for i_f, mdl in enumerate(cn_failed['Model']):
    ax.barh([mdl], [0.5], color='#bdc3c7', alpha=0.4, height=0.55, label='Did not converge' if i_f==0 else '')
    ax.text(0.51, i_f, 'Did not converge', va='center', fontsize=9, color='#7f8c8d')
ax.axvline(0.5, color='red', ls='--', alpha=0.5, label='Random (0.5)')
ax.set(xlabel='C-td (IPCW Antolini)', title='Model Outcomes: CN → MCI or AD', xlim=(0.3, 1.0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison_cn.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Bootstrap Resampling: Formal Cross-Model Performance Comparison ─────────
# HARDENED VERSION: safe for an unattended overnight run.
#
# Key change from v1: results are written to CSV after EVERY iteration, not
# only at the end. If the run is interrupted at any point (Kaggle session
# timeout, browser disconnect, any error), everything completed so far is
# already on disk. Re-running this cell picks up where it left off instead
# of starting over.
#
# ── Before you launch the real overnight run ────────────────────────────────
# 1. SMOKE TEST FIRST. Set N_BOOTSTRAP=2 and run once, interactively, to
#    confirm the whole pipeline executes end-to-end with your actual data
#    objects (X_mci_imp, y_ev_mci, y_dur_mci, OUT_DIR, etc. below MUST match
#    your notebook's real variable names. Check this before trusting the
#    overnight run). Only move to the real N once this succeeds cleanly.
# 2. RUN IT AS A BACKGROUND / COMMITTED EXECUTION, not an interactive tab
#    left open. On Kaggle: "Save Version" -> "Save & Run All (Commit)" runs
#    detached from your browser session, so closing the tab or your laptop
#    sleeping won't kill it. An interactive session left open in a browser
#    tab is the single most common way an overnight run silently dies.
# 3. Double check the variable names referenced below (X_mci_imp, y_ev_mci,
#    y_dur_mci, RANDOM_SEED, OUT_DIR, and the four fitting/scoring helper
#    functions) actually exist with those exact names earlier in your
#    notebook. A NameError here fails fast (good) rather than 4 hours in.

import time
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODEL_COMPARISON_BOOTSTRAP = False  # flip to True when ready to launch
N_BOOTSTRAP = 100                     # full run (set RUN_MODEL_COMPARISON_BOOTSTRAP = True to launch, about 7 hours)
BOOT_N_TRIALS = 8                        # lightweight refit, not full HPO
CHECKPOINT_PATH = OUT_DIR / 'model_comparison_bootstrap.csv'

def _fit_and_score_cox(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, model, scaler, _ = run_cox_ph(X_tr, ev_tr, dur_tr, label='BootCox', n_trials=n_trials)
    c_td, _ = calc_cox_ph_c(model, scaler, X_ev, ev_ev, dur_ev)
    return c_td

def _fit_and_score_gbsa(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, _, model, _, _ = gbsa_survival_cv(X_tr, ev_tr, dur_tr,
        feature_names=list(X_tr.columns), label='BootGBSA', n_trials=n_trials)
    c_td, _ = calc_gbsa_c(model, X_ev, ev_ev, dur_ev)
    return c_td

def _fit_and_score_aft(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials=None):
    model = _fit_aft_with_cv(X_tr, ev_tr, dur_tr, 'BootAFT')
    _data = X_ev.copy()
    _data['duration'] = np.array(dur_ev, dtype=np.float64)
    _data['event'] = np.array(ev_ev, dtype=np.int32)
    surv = model.predict_survival_function(_data)
    tg = surv.index.values.astype(np.float64)
    idx = np.clip(np.searchsorted(tg, np.array(dur_ev)), 0, len(tg) - 1).astype(np.int64)
    return concordance_td(np.array(dur_ev, dtype=np.float64), np.array(ev_ev, dtype=np.int32),
                           surv.values, idx, method='adj_antolini', ipcw=True)

def _fit_and_score_deepsurv(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, n_trials):
    _, _, model, scaler, _, _, _ = run_deepsurv(X_tr, ev_tr, dur_tr, 'BootDeepSurv', n_trials=n_trials)
    c_td, _ = calc_deepsurv_c(model, scaler, X_ev, ev_ev, dur_ev)
    return c_td

FITTERS = {
    'Cox PH': _fit_and_score_cox,
    'GBSA': _fit_and_score_gbsa,
    'AFT': _fit_and_score_aft,
    'DeepSurv': _fit_and_score_deepsurv,
}

def _run_one_iteration(b):
    """Runs one bootstrap iteration for all four models. Returns a dict row.
    Broad try/except at this level (not just per-model) so a failure in the
    resampling/split step itself can't kill the whole overnight run.

    IMPORTANT: only the TRAINING side is bootstrap-resampled (with
    replacement). Evaluation is always against the fixed, untouched
    X_mci_eval / y_ev_mci_eval / y_dur_mci_eval used elsewhere in this
    notebook for the main C-td results. Resampling the eval set too (as an
    earlier version of this cell did) lets a duplicated subject land in both
    the train and "held-out" splits, leaking train data into evaluation and
    disproportionately inflating scores for the more flexible models
    (GBSA, DeepSurv) that can more easily fit memorized rows. Keeping eval
    fixed removes that leakage entirely and ties every bootstrap iteration
    to the same population your main results table already reports against.
    """
    row = {'iteration': b}
    try:
        rng = np.random.default_rng(RANDOM_SEED + b)
        boot_idx = rng.choice(len(X_mci_train), size=len(X_mci_train), replace=True)
        X_tr = X_mci_train.iloc[boot_idx].reset_index(drop=True)
        ev_tr = (y_ev_mci_train.iloc[boot_idx].reset_index(drop=True)
                 if hasattr(y_ev_mci_train, 'iloc') else y_ev_mci_train[boot_idx])
        dur_tr = (y_dur_mci_train.iloc[boot_idx].reset_index(drop=True)
                  if hasattr(y_dur_mci_train, 'iloc') else y_dur_mci_train[boot_idx])

        # Guard: some fitters need >=2 members per class in the resampled
        # training data. With n~670 and a ~40% event rate this should never
        # trip, but a silent hang/crash hours in over an edge case is worse
        # than a clear message now.
        n_events = int(np.sum(np.asarray(ev_tr) == 1))
        n_censored = int(np.sum(np.asarray(ev_tr) == 0))
        if n_events < 2 or n_censored < 2:
            row['error'] = f'degenerate resample (events={n_events}, censored={n_censored})'
            for name in FITTERS:
                row[name] = np.nan
            return row

        # Fixed, untouched eval set: same every iteration, never resampled.
        X_ev, ev_ev, dur_ev = X_mci_eval, y_ev_mci_eval, y_dur_mci_eval

        for name, fitter in FITTERS.items():
            try:
                row[name] = fitter(X_tr, ev_tr, dur_tr, X_ev, ev_ev, dur_ev, BOOT_N_TRIALS)
            except Exception as e:
                print(f'  [bootstrap {b}] {name} failed: {e}')
                row[name] = np.nan
        row['error'] = ''
    except Exception as e:
        print(f'  [bootstrap {b}] ITERATION-LEVEL failure (resampling/split step): {e}')
        row['error'] = str(e)
        for name in FITTERS:
            row.setdefault(name, np.nan)
    return row

if RUN_MODEL_COMPARISON_BOOTSTRAP:
    # ── Resume support: skip iterations already completed on disk ───────────
    if CHECKPOINT_PATH.exists():
        done_df = pd.read_csv(CHECKPOINT_PATH)
        done_iters = set(done_df['iteration'].tolist())
        print(f'Found existing checkpoint with {len(done_iters)} completed iterations. Resuming.')
    else:
        done_df = pd.DataFrame(columns=['iteration'] + list(FITTERS.keys()) + ['error'])
        done_iters = set()
        print('No existing checkpoint found. Starting fresh.')

    start_time = time.time()
    for b in range(N_BOOTSTRAP):
        if b in done_iters:
            continue

        iter_start = time.time()
        row = _run_one_iteration(b)
        iter_elapsed = time.time() - iter_start

    
        row_df = pd.DataFrame([row])
        row_df.to_csv(CHECKPOINT_PATH, mode='a', header=not CHECKPOINT_PATH.exists(), index=False)
        done_df = pd.concat([done_df, row_df], ignore_index=True)

        n_done = len(done_df)
        elapsed_total = time.time() - start_time
        avg_per_iter = elapsed_total / max(1, (n_done - len(done_iters)))
        remaining = N_BOOTSTRAP - n_done
        eta_hrs = (remaining * avg_per_iter) / 3600

        scores_str = {k: (round(row[k], 4) if not pd.isna(row[k]) else 'NaN') for k in FITTERS}
        print(f'[{n_done}/{N_BOOTSTRAP}] iter {b} done in {iter_elapsed/60:.1f} min '
              f'| scores={scores_str} | ETA remaining: {eta_hrs:.1f} hr')

    print(f'\nAll {N_BOOTSTRAP} iterations complete (or already were). '
          f'Checkpoint at: {CHECKPOINT_PATH}')

    # ── Pivotal intervals on pairwise differences ────────────────────────────
    boot_df = pd.read_csv(CHECKPOINT_PATH)
    n_failed = boot_df['error'].fillna('').astype(bool).sum() if 'error' in boot_df else 0
    if n_failed:
        print(f'Note: {n_failed}/{len(boot_df)} iterations had at least one issue logged '
              f'check the "error" column before trusting the intervals below.')

    from itertools import combinations
    print('\nPairwise pivotal 95% CIs on C-td difference (model A - model B):')
    print('(interval containing 0 => not statistically distinguishable)\n')
    pairwise_results = []
    for m1, m2 in combinations(FITTERS.keys(), 2):
        diffs = (boot_df[m1] - boot_df[m2]).dropna().values
        if len(diffs) < 2:
            print(f'{m1} - {m2}: insufficient completed iterations to compute an interval.')
            continue
        point_diff = diffs.mean()
        lo = 2 * point_diff - np.percentile(diffs, 97.5)
        hi = 2 * point_diff - np.percentile(diffs, 2.5)
        contains_zero = lo <= 0 <= hi
        pairwise_results.append(dict(pair=f'{m1} - {m2}', mean_diff=point_diff,
                                      ci_lo=lo, ci_hi=hi, contains_zero=contains_zero,
                                      n=len(diffs)))
        print(f'{m1:>10} - {m2:<10}: mean diff={point_diff:+.4f}  95% CI=[{lo:+.4f}, {hi:+.4f}]  '
              f'n={len(diffs)}  {"NOT distinguishable" if contains_zero else "DISTINGUISHABLE"}')

    pd.DataFrame(pairwise_results).to_csv(OUT_DIR / 'model_comparison_pairwise_ci.csv', index=False)
else:
    print('RUN_MODEL_COMPARISON_BOOTSTRAP is False. Set True once ready  ')

## 8. Key Findings and Recommendations

### Results Summary

Four survival models were trained and evaluated on two ADNI cohorts using
held-out 20% test sets. All metrics are IPCW Antolini time-dependent C-td
with 500-resample bootstrap 95% confidence intervals.

| Model | MCI → AD C-td | 95% CI | CN → MCI/AD C-td | 95% CI |
|-------|--------------|--------|-----------------|--------|
| Cox PH | **0.8422** | [0.806, 0.884] | 0.7633 | [0.682, 0.849] |
| AFT | 0.8418 | [0.806, 0.884] | **0.7719** | [0.692, 0.849] |
| GBSA | 0.8286 | [0.777, 0.869] | 0.7287 | [0.643, 0.830] |
| DeepSurv | 0.8231 | [0.775, 0.863] | 0.7491 | [0.659, 0.842] |

A C-td above 0.75 is generally considered strong for Alzheimer's progression
prediction at ADNI's level of censoring. All four models clear this threshold
on both cohorts. Confidence intervals overlap substantially across all models
on MCI. No single approach is statistically distinguishable from the others
at this test set size which tells us the dominant biological signal is being
captured by every method and additional model complexity yields diminishing returns.

---

### What the Models Agree On

Across all four model families, the same core features dominate: **FDG-PET
glucose metabolism**, **RAVLT Immediate Recall**, **MoCA**, **FAQ (Functional
Activities)**, and **Hippocampal Volume** consistently appear in every model's
top importance rankings. Convergence across a linear model (Cox), a tree
ensemble (GBSA), a parametric model (AFT), and a neural network (DeepSurv)
is the strongest possible evidence that these features carry genuine biological
signal rather than method-specific artifacts.

The AT(N) biomarker framework, Amyloid (AV45-PET, CSF Amyloid-β), Tau
(CSF Total Tau, Phospho-Tau), and Neurodegeneration (FDG-PET, Hippocampal
Volume), is well-represented across all models, validating that the pipeline
is capturing established AD pathophysiology.

---

### Where the Models Differ

**Non-linear models detect executive function decline that linear models miss.**
GBSA and DeepSurv both prominently feature mPACC Trails B and mPACC Digit
Span, measures of processing speed and executive function, that are absent
from Cox PH's top features. These have threshold-like relationships with
conversion risk: subjects below a critical performance level convert
substantially faster. Cox's linear log-hazard cannot represent this inflection
point. This is the most clinically meaningful structural difference between
the approaches.

**APOE ε4 interactions carry more signal than the raw allele count.**
DeepSurv's permutation importance elevates APOE ε4 × Tau Burden and APOE ε4
× Amyloid Positivity above the raw gene count, suggesting the genetic risk is
multiplicative with downstream pathology rather than additive. Carrying APOE ε4
alongside elevated tau or confirmed amyloid positivity is substantially more
dangerous than either risk factor alone: a relationship the neural network
captures but Cox collapses into a single coefficient.

**GBSA overfitting is model-run dependent.** The MCI train/OOF gap was +0.046
in this run (vs +0.137 in the prior run), reflecting that Optuna found a
well-regularized configuration (`max_depth=2, min_samples_leaf=15`) this time.
GBSA performance should be treated as having higher variance across retraining
runs than the other three models.

---

### MCI → AD vs CN → MCI/AD: Two Different Problems

The CN cohort results (0.729–0.772) are consistently weaker than MCI
(0.823–0.842) for structural reasons, not methodological ones:

- **Lower event rate:** 17.7% of CN subjects converted vs 40.2% of MCI
  subjects. Fewer events mean less signal and wider CIs
- **Earlier disease stage:** the pathology driving eventual conversion may not
  yet be measurable at the CN baseline visit
- **Longer trajectories:** CN subjects have longer pre-event windows, but
  ADNI's observation period captures less of the full arc

The models also rank differently across cohorts. **AFT is the best CN model
(0.772)**, outperforming Cox (0.763), DeepSurv (0.749), and GBSA (0.729).
The Weibull parametric structure fits the CN → MCI/AD transition timing better
than any other family, suggesting that time-to-first-transition in cognitively
normal subjects follows a more regular distributional shape than in the
already-symptomatic MCI cohort, where heterogeneous trajectories and competing
risks add noise that non-parametric methods handle no better than a well-fit
parametric model.

---

### Recommendations

**For MCI → AD prediction: use Cox PH.**

Cox PH achieves the highest test C-td (0.842), a minimal train/OOF gap
(+0.006), and the tightest confidence interval of any model. Its elastic-net
regularized coefficients are directly interpretable as log hazard ratios, so a
clinician can read the feature importance chart and understand exactly which
baseline measurements drive the prediction and in which direction. For any
clinical decision support context, interpretability is not optional; it is a
prerequisite for trust and regulatory acceptance. AFT is statistically
indistinguishable from Cox on MCI (0.8418 vs 0.8422, fully overlapping CIs)
and is a strong alternative if absolute time-to-event predictions are preferred
over relative risk rankings.

**For CN → MCI/AD prediction: use AFT.**

AFT is the best-performing model on CN (0.772) and uniquely provides absolute
time-to-event estimates, not just risk rankings, which is more actionable for
counseling patients and families about expected conversion timelines. The
Weibull distributional assumption also makes the model's structure explicit and
testable, which matters for clinical validation and regulatory contexts.

**GBSA is competitive but should not be deployed without stability testing.**
Its performance varies meaningfully across retraining runs due to Optuna
finding different tree configurations. Before any deployment use, run at least
3–5 retraining seeds and confirm the OOF gap stays below 0.08. If it does not,
constrain `max_depth ≤ 3` and `min_samples_leaf ≥ 10` in the HPO search space.

**DeepSurv adds interpretive value even where it doesn't win on C-td.**
The APOE ε4 interaction signal and executive function features detected by the
neural network are clinically meaningful findings that the other models miss or
underweight. For research purposes: understanding which features matter and
how they interact. DeepSurv's permutation importance chart is the most
informative output in this pipeline.

## 9. Conformal Prediction

Standard survival models output either a risk score (Cox, GBSA, DeepSurv) or a
predicted event time (AFT), but neither comes with a **coverage guarantee** for
individual patients. A model may rank subjects well on average (high C-td) while
still producing predictions whose uncertainty is poorly characterised.

**Conformal prediction (CP)** addresses this directly. Given a miscoverage level
$\alpha$, CP constructs a prediction interval $[L_i, U_i]$ such that:

$$P(T_i \in [L_i, U_i]) \geq 1 - \alpha$$

This guarantee is **distribution-free**, so it holds without any assumptions about
the shape of the survival distribution, relying only on exchangeability of the
calibration and evaluation samples.

**Framework:** We implement the conformalized survival analysis approach of
Candès, Lei & Ren (2023, *JRSS-B*), with censoring handling adapted from
Qin et al. (2025, *Biometrics*).

**Models conformalized (all four):**

| Model | CP approach | Nonconformity score |
|-------|------------|---------------------|
| Weibull AFT | Direct (predicts time) | Standardised log-time residual |
| Cox PH | Survival function inversion at $p=0.5$ | Raw time residual |
| GBSA | Survival function inversion at $p=0.5$ | Raw time residual |
| DeepSurv | Survival function inversion at $p=0.5$ | Raw time residual |

Cox PH, GBSA, and DeepSurv all output a survival curve rather than a direct time
prediction, so each requires the same inversion step as AFT's baseline comparison. See Section 9.4 (and 9.4a/9.4b) for a sensitivity analysis on the choice of $p=0.5$
across all three inversion-based models.

**Data split:** The existing held-out test set is divided 50/50 into a
**calibration set** (computes conformal quantile $\hat{q}$) and an
**evaluation set** (measures empirical coverage). Neither split was seen
during model training or hyperparameter selection.


### 9.1 Conformalized AFT

The Weibull AFT model directly predicts $\log T_i$, making it the most natural
entry point for conformal regression. The nonconformity score for each calibration
subject is the standardised residual on the log-time scale:

$$s_i = \frac{\log T_i - \hat{\mu}_i}{\hat{\sigma}}$$

where $\hat{\mu}_i = \log(\widehat{\text{median}}_i)$ is the model's predicted
log median conversion time and $\hat{\sigma}$ is the fitted Weibull scale.

**Censoring handling:** Censored subjects whose observed follow-up exceeds their
predicted median are assigned $s_i = -\infty$ (trivially covered). Those censored
before their predicted median receive $s_i = +\infty$ (conservative).

The conformal quantile is:

$$\hat{q}_\alpha = \text{Quantile}\!\left(\{s_i\},\;\frac{\lceil (n_{\text{cal}}+1)(1-\alpha) \rceil}{n_{\text{cal}}}\right)$$

Prediction interval for evaluation subject $j$:

$$\left[\exp(\hat{\mu}_j - \hat{q}_\alpha \cdot \hat{\sigma}),\;\exp(\hat{\mu}_j + \hat{q}_\alpha \cdot \hat{\sigma})\right]$$

Evaluated at $\alpha \in \{0.10, 0.20\}$ (90% and 80% target coverage).


In [ ]:
# ── 9.1  Conformalized AFT  ─────────────────────────────────────────
# Cal and eval come directly from the 70/20/10 split
ev_cal_aft   = np.array(y_ev_mci_cal,   dtype=np.int32)
ev_eval_aft  = np.array(y_ev_mci_eval,  dtype=np.int32)
dur_cal_aft  = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_aft = np.array(y_dur_mci_eval, dtype=np.float64)
X_cal_aft    = X_mci_cal.reset_index(drop=True)
X_eval_aft   = X_mci_eval.reset_index(drop=True)
print(f'AFT CP cal  : n={len(X_cal_aft)}, events={ev_cal_aft.sum()} ({ev_cal_aft.mean()*100:.1f}%)')
print(f'AFT CP eval : n={len(X_eval_aft)}, events={ev_eval_aft.sum()} ({ev_eval_aft.mean()*100:.1f}%)')

# ── Extract predicted median (in years) ───────────────────────────────────────
def _aft_predict_log_median(model, X):
    X_clean = X.drop(columns=['duration', 'event'], errors='ignore')
    median_t = model.predict_median(X_clean)
    return np.log(np.clip(median_t.values, 1e-6, None))

def _aft_sigma(model):
    rho_intercept = model.params_['rho_']['Intercept']
    return float(np.exp(-rho_intercept))

mu_cal  = _aft_predict_log_median(aft_model_mci, X_cal_aft)
mu_eval = _aft_predict_log_median(aft_model_mci, X_eval_aft)
sigma   = _aft_sigma(aft_model_mci)
print(f'Weibull scale sigma: {sigma:.4f}')
pred_median_cal  = np.exp(mu_cal)
pred_median_eval = np.exp(mu_eval)
print(f'Predicted median cal- mean: {pred_median_cal.mean():.3f} yr, '
      f'std: {pred_median_cal.std():.3f} yr')
print(f'Predicted median eval- mean: {pred_median_eval.mean():.3f} yr, '
      f'std: {pred_median_eval.std():.3f} yr')

# ── Nonconformity scores (observed events only, raw time scale) ───────────────
obs_cal_mask   = ev_cal_aft == 1
scores_cal_aft = pred_median_cal[obs_cal_mask] - dur_cal_aft[obs_cal_mask]
print(f'\nCalibration scores (observed only): n={len(scores_cal_aft)}, '
      f'mean={scores_cal_aft.mean():.3f}, std={scores_cal_aft.std():.3f}')

# ── Conformal quantile helper ─────────────────────────────────────────────────
def _conformal_quantile(scores, alpha, n_cal):
    level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
    return float(np.quantile(scores, level))

# ── Prediction intervals and coverage ────────────────────────────────────────
results_cp_aft = {}
n_cal_aft = len(scores_cal_aft)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_aft, alpha, n_cal_aft)
    lo      = np.clip(pred_median_eval - q_hat, 0, None)
    hi      = pred_median_eval + q_hat
    widths  = hi - lo
    obs     = ev_eval_aft == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_aft[obs] >= lo[obs]) & (dur_eval_aft[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'median_width={np.median(widths):.3f} yr')
    results_cp_aft[alpha] = dict(
        q_hat=q_hat, lo=lo, hi=hi, widths=widths,
        coverage=coverage, n_obs=n_obs, covered=covered,
        mu_eval=mu_eval, pred_median_eval=pred_median_eval,
    )

### 9.2 AFT Conformal Prediction Intervals Visualisation

**Plot 1:** Prediction intervals sorted by predicted median conversion time.
Each subject is a vertical bar $[L_i, U_i]$; the dot marks the predicted median.
Blue = observed event, covered. Red = observed event, not covered. Grey = censored.

**Plot 2:** Interval width vs. predicted conversion time shows whether the model
is more certain about high-risk vs. low-risk subjects. Under the Weibull AFT
parametric structure, width scales with $\\exp(\\hat{\\mu}_i)$, so high-risk subjects
(short predicted time) receive narrower absolute intervals.


In [ ]:
# ── 9.2  AFT CP Visualisation ────────────────────────────────────────────────
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    r      = results_cp_aft[alpha]
    mu_yrs = np.exp(r['mu_eval'])
    order  = np.argsort(mu_yrs)
    ax     = axes[ax_idx]
    obs    = ev_eval_aft == 1
    for rank, idx in enumerate(order):
        is_obs    = obs[idx]
        true_t    = dur_eval_aft[idx]
        covered_i = (true_t >= r['lo'][idx]) and (true_t <= r['hi'][idx])
        col       = '#bdc3c7' if not is_obs else ('#2980b9' if covered_i else '#e74c3c')
        ab        = 0.5 if not is_obs else 0.75
        ax.plot([rank, rank], [r['lo'][idx], r['hi'][idx]], color=col, lw=1.2, alpha=ab)
        ax.scatter(rank, mu_yrs[idx], color=col, s=8, zorder=3, alpha=ab)
        if is_obs:
            ax.scatter(rank, true_t, color=col, s=18,
                       marker='D' if covered_i else 'x', zorder=4)
    ax.set(xlabel='Subjects (sorted by predicted conversion time)',
           ylabel='Years from baseline',
           title=f'AFT CP  |  alpha={alpha}  target={(1-alpha)*100:.0f}%\n'
                 f'Empirical coverage = {r["coverage"]:.1%}  '
                 f'({r["covered"]}/{r["n_obs"]} observed events)')
    ax.legend(handles=[
        Line2D([0],[0], color='#2980b9', lw=2, label='Covered (observed)'),
        Line2D([0],[0], color='#e74c3c', lw=2, label='Not covered (observed)'),
        Line2D([0],[0], color='#bdc3c7', lw=2, label='Censored'),
    ], fontsize=8, loc='upper left')
plt.suptitle('Conformalized AFT: Prediction Intervals (MCI -> AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_aft_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

# Coverage summary
print('='*60)
print('AFT CP COVERAGE SUMMARY')
print(f'{"Alpha":<8}{"Target":<10}{"Empirical":<12}{"Median Width":<14}{"q_hat"}')
print('-'*60)
for alpha, r in results_cp_aft.items():
    print(f'{alpha:<8.2f}{1-alpha:<10.0%}{r["coverage"]:<12.1%}'
          f'{np.median(r["widths"]):<14.3f}{r["q_hat"]:.4f}')
print('='*60)

# Width vs predicted time
fig, ax = plt.subplots(figsize=(8, 5))
for alpha, col in [(0.10, '#2980b9'), (0.20, '#e67e22')]:
    r = results_cp_aft[alpha]
    ax.scatter(np.exp(r['mu_eval']), r['widths'], alpha=0.5, s=18, color=col,
               label=f'alpha={alpha}  target={(1-alpha)*100:.0f}%')
ax.set(xlabel='Predicted median conversion time (years)',
       ylabel='Prediction interval width (years)',
       title='AFT CP: Interval Width vs. Predicted Conversion Time')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_aft_width_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.3 Conformalized Cox PH Using Survival Function Inversion

Cox PH outputs a survival function $\hat{S}(t \mid x_i)$, not a predicted time.
We extract a pseudo-time by inverting at $p = 0.5$:

$$\hat{t}_i = \inf\{t : \hat{S}(t \mid x_i) \leq 0.5\}$$

The nonconformity score is the raw time residual:

$$s_i = T_i - \hat{t}_i$$

with the same $\pm\infty$ censoring handling as AFT.

The prediction interval is:

$$[\hat{t}_j - \hat{q}_\alpha,\; \hat{t}_j + \hat{q}_\alpha]$$

clipped to $[0, \infty)$. Note that unlike AFT, this produces **constant-width**
intervals for all subject, which is a limitation of the raw residual score. More
sophisticated implementations use IPCW-weighted quantiles to allow adaptive width.

In [ ]:
# ── 9.3  Conformalized Cox PH ────────────────────────────────────────────────
# Cal and eval come directly from the 70/20/10 split, same as AFT (9.1)
ev_cal_cox    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_cox   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_cox   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_cox  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_cox_cal  = surv_cox_mci
surv_cox_eval = surv_cox_mci_eval

print(f'Cox CP cal  : n={surv_cox_cal.shape[1]}, events={ev_cal_cox.sum()}')
print(f'Cox CP eval : n={surv_cox_eval.shape[1]}, events={ev_eval_cox.sum()}')

# ── Invert survival function at S(t) = 0.5 ───────────────────────────────────
def _invert_surv(surv_matrix, p=0.5):
    times   = surv_matrix.index.values.astype(np.float64)
    medians = np.full(surv_matrix.shape[1], times[-1])
    for j in range(surv_matrix.shape[1]):
        below = np.where(surv_matrix.iloc[:, j].values <= p)[0]
        if len(below) > 0:
            medians[j] = times[below[0]]
    return medians

t_hat_cal  = _invert_surv(surv_cox_cal)
t_hat_eval = _invert_surv(surv_cox_eval)
print(f'Cox predicted median: cal mean : {t_hat_cal.mean():.3f} yr')
print(f'Cox predicted median: eval mean: {t_hat_eval.mean():.3f} yr')

# ── Nonconformity scores (observed events only, raw time scale) ───────────────
obs_cal_mask_cox = ev_cal_cox == 1
scores_cal_cox = t_hat_cal[obs_cal_mask_cox] - dur_cal_cox[obs_cal_mask_cox]
print(f'Calibration scores (observed only): n={len(scores_cal_cox)}, '
      f'mean={scores_cal_cox.mean():.3f}, std={scores_cal_cox.std():.3f}')

# ── Conformal quantile helper (shared across all four models) ────────────────
def _conformal_quantile(scores, alpha, n_cal):
    level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
    return float(np.quantile(scores, level))

# ── Prediction intervals and coverage ────────────────────────────────────────
results_cp_cox = {}
n_cal_cox = len(scores_cal_cox)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_cox, alpha, n_cal_cox)
    lo      = np.clip(t_hat_eval - q_hat, 0, None)
    hi      = t_hat_eval + q_hat
    widths  = hi - lo
    obs     = ev_eval_cox == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_cox[obs] >= lo[obs]) & (dur_eval_cox[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_cox[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                  coverage=coverage, n_obs=n_obs, covered=covered,
                                  t_hat_eval=t_hat_eval)


### 9.3a Conformalized GBSA

GBSA, like Cox PH, outputs a survival function rather than a predicted time, so
it uses the same inversion-at-$p=0.5$ approach and the same `_invert_surv` /
`_conformal_quantile` helpers defined in Section 9.3. Cal and eval sets are the
same true 70/20/10 split used by every other model in this notebook.

In [ ]:
# ── 9.3a  Conformalized GBSA ──────────────────────────────────────────────────
ev_cal_gbsa    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_gbsa   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_gbsa   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_gbsa  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_gbsa_cal_ = surv_gbsa_mci
surv_gbsa_eval_ = surv_gbsa_mci_eval

print(f'GBSA CP cal  : n={surv_gbsa_cal_.shape[1]}, events={ev_cal_gbsa.sum()}')
print(f'GBSA CP eval : n={surv_gbsa_eval_.shape[1]}, events={ev_eval_gbsa.sum()}')

t_hat_cal_gbsa  = _invert_surv(surv_gbsa_cal_)
t_hat_eval_gbsa = _invert_surv(surv_gbsa_eval_)
print(f'GBSA predicted median: cal mean : {t_hat_cal_gbsa.mean():.3f} yr')
print(f'GBSA predicted median: eval mean: {t_hat_eval_gbsa.mean():.3f} yr')

obs_cal_mask_gbsa = ev_cal_gbsa == 1
scores_cal_gbsa = t_hat_cal_gbsa[obs_cal_mask_gbsa] - dur_cal_gbsa[obs_cal_mask_gbsa]
print(f'Calibration scores (observed only): n={len(scores_cal_gbsa)}, '
      f'mean={scores_cal_gbsa.mean():.3f}, std={scores_cal_gbsa.std():.3f}')

results_cp_gbsa = {}
n_cal_gbsa = len(scores_cal_gbsa)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_gbsa, alpha, n_cal_gbsa)
    lo      = np.clip(t_hat_eval_gbsa - q_hat, 0, None)
    hi      = t_hat_eval_gbsa + q_hat
    widths  = hi - lo
    obs     = ev_eval_gbsa == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_gbsa[obs] >= lo[obs]) & (dur_eval_gbsa[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_gbsa[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                   coverage=coverage, n_obs=n_obs, covered=covered,
                                   t_hat_eval=t_hat_eval_gbsa)


### 9.3b Conformalized DeepSurv

DeepSurv also outputs a survival function, so it follows the same pattern as
Cox PH and GBSA above.

In [ ]:
# ── 9.3b  Conformalized DeepSurv ──────────────────────────────────────────────
ev_cal_ds    = np.array(y_ev_mci_cal,  dtype=np.int32)
ev_eval_ds   = np.array(y_ev_mci_eval, dtype=np.int32)
dur_cal_ds   = np.array(y_dur_mci_cal,  dtype=np.float64)
dur_eval_ds  = np.array(y_dur_mci_eval, dtype=np.float64)
surv_ds_cal_ = surv_mci
surv_ds_eval_ = surv_mci_eval

print(f'DeepSurv CP cal  : n={surv_ds_cal_.shape[1]}, events={ev_cal_ds.sum()}')
print(f'DeepSurv CP eval : n={surv_ds_eval_.shape[1]}, events={ev_eval_ds.sum()}')

t_hat_cal_ds  = _invert_surv(surv_ds_cal_)
t_hat_eval_ds = _invert_surv(surv_ds_eval_)
print(f'DeepSurv predicted median- cal mean : {t_hat_cal_ds.mean():.3f} yr')
print(f'DeepSurv predicted median- eval mean: {t_hat_eval_ds.mean():.3f} yr')

obs_cal_mask_ds = ev_cal_ds == 1
scores_cal_ds = t_hat_cal_ds[obs_cal_mask_ds] - dur_cal_ds[obs_cal_mask_ds]
print(f'Calibration scores (observed only): n={len(scores_cal_ds)}, '
      f'mean={scores_cal_ds.mean():.3f}, std={scores_cal_ds.std():.3f}')

results_cp_deepsurv = {}
n_cal_ds = len(scores_cal_ds)

for alpha in [0.10, 0.20]:
    q_hat   = _conformal_quantile(scores_cal_ds, alpha, n_cal_ds)
    lo      = np.clip(t_hat_eval_ds - q_hat, 0, None)
    hi      = t_hat_eval_ds + q_hat
    widths  = hi - lo
    obs     = ev_eval_ds == 1
    n_obs   = obs.sum()
    covered = ((dur_eval_ds[obs] >= lo[obs]) & (dur_eval_ds[obs] <= hi[obs])).sum()
    coverage = covered / n_obs if n_obs > 0 else np.nan
    print(f'alpha={alpha:.2f}  q_hat={q_hat:.4f} yr  '
          f'coverage={coverage:.1%} ({covered}/{n_obs})  '
          f'width={widths[0]:.3f} yr (constant)')
    results_cp_deepsurv[alpha] = dict(q_hat=q_hat, lo=lo, hi=hi, widths=widths,
                                       coverage=coverage, n_obs=n_obs, covered=covered,
                                       t_hat_eval=t_hat_eval_ds)


In [ ]:
# ── 9.4  Cox CP Visualisation ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    r     = results_cp_cox[alpha]
    order = np.argsort(r['t_hat_eval'])
    ax    = axes[ax_idx]
    obs   = ev_eval_cox == 1
    for rank, idx in enumerate(order):
        is_obs    = obs[idx]
        true_t    = dur_eval_cox[idx]
        covered_i = (true_t >= r['lo'][idx]) and (true_t <= r['hi'][idx])
        col       = '#bdc3c7' if not is_obs else ('#2980b9' if covered_i else '#e74c3c')
        ab        = 0.5 if not is_obs else 0.75
        ax.plot([rank, rank], [r['lo'][idx], r['hi'][idx]], color=col, lw=1.2, alpha=ab)
        ax.scatter(rank, r['t_hat_eval'][idx], color=col, s=8, zorder=3, alpha=ab)
        if is_obs:
            ax.scatter(rank, true_t, color=col, s=18,
                       marker='D' if covered_i else 'x', zorder=4)
    ax.set(xlabel='Subjects (sorted by predicted conversion time)',
           ylabel='Years from baseline',
           title=f'Cox PH CP  |  alpha={alpha}  target={(1-alpha)*100:.0f}%\n'
                 f'Empirical coverage = {r["coverage"]:.1%}  '
                 f'({r["covered"]}/{r["n_obs"]} observed events)')
    ax.legend(handles=[
        Line2D([0],[0], color='#2980b9', lw=2, label='Covered (observed)'),
        Line2D([0],[0], color='#e74c3c', lw=2, label='Not covered (observed)'),
        Line2D([0],[0], color='#bdc3c7', lw=2, label='Censored'),
    ], fontsize=8, loc='upper left')
plt.suptitle('Conformalized Cox PH: Prediction Intervals (MCI -> AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_cox_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*60)
print('Cox PH CP COVERAGE SUMMARY')
print(f'{"Alpha":<8}{"Target":<10}{"Empirical":<12}{"Width (yrs)":<14}{"q_hat"}')
print('-'*60)
for alpha, r in results_cp_cox.items():
    print(f'{alpha:<8.2f}{1-alpha:<10.0%}{r["coverage"]:<12.1%}'
          f'{r["widths"][0]:<14.3f}{r["q_hat"]:.4f}')
print('='*60)


### 9.4 Cox PH Inversion Threshold Sensitivity Analysis

The Cox PH survival function $\hat{S}(t \mid x_i)$ does not directly output a
predicted time, so it must be inverted at a chosen probability threshold $p$ to
obtain a pseudo time $\hat{t}_i(p) = \inf\{t : \hat{S}(t \mid x_i) \leq p\}$.

The choice of $p$ is a methodological decision with meaningful consequences:

- **Small $p$** (like 0.3): corresponds to a more conservative predicted time
  (the model must be very confident before declaring likely conversion), producing
  wider intervals and higher coverage
- **Large $p$** (like 0.7): corresponds to an aggressive predicted time,
  producing narrower intervals but risking undercoverage

We run a sensitivity analysis over $p \in \{0.3, 0.4, 0.5, 0.6, 0.7\}$ to
quantify how this choice affects empirical coverage and interval width, and to
select the threshold that best balances the two.

In [ ]:
# ── Cox CP: Sensitivity Analysis on Inversion Threshold p ───────────────────
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
sensitivity_rows = []
results_cp_cox_sensitivity = {}

for p in thresholds:
    t_hat_cal_p  = _invert_surv(surv_cox_cal,  p=p)
    t_hat_eval_p = _invert_surv(surv_cox_eval, p=p)

    obs_mask_p     = ev_cal_cox == 1
    scores_cox_p   = t_hat_cal_p[obs_mask_p] - dur_cal_cox[obs_mask_p]
    n_cal_cox_p    = len(scores_cox_p)

    for alpha in [0.10, 0.20]:
        q_hat   = _conformal_quantile(scores_cox_p, alpha, n_cal_cox_p)
        lo      = np.clip(t_hat_eval_p - q_hat, 0, None)
        hi      = t_hat_eval_p + q_hat
        widths  = hi - lo
        obs     = ev_eval_cox == 1
        n_obs   = obs.sum()
        covered = ((dur_eval_cox[obs] >= lo[obs]) & (dur_eval_cox[obs] <= hi[obs])).sum()
        coverage = covered / n_obs if n_obs > 0 else np.nan

        results_cp_cox_sensitivity[(p, alpha)] = dict(
            q_hat=q_hat, lo=lo, hi=hi, widths=widths,
            coverage=coverage, n_obs=n_obs, covered=covered,
            t_hat_eval=t_hat_eval_p,
        )
        sensitivity_rows.append(dict(
            p=p, alpha=alpha,
            target=f'{(1-alpha)*100:.0f}%',
            empirical=f'{coverage:.1%}',
            q_hat=round(q_hat, 4),
            median_width=round(float(np.median(widths)), 3),
        ))

# Print table
sens_df = pd.DataFrame(sensitivity_rows)
print('='*72)
print('COX PH CP: SENSITIVITY ANALYSIS ON INVERSION THRESHOLD p')
print('='*72)
for alpha in [0.10, 0.20]:
    sub = sens_df[sens_df['alpha'] == alpha]
    print(f'\nalpha={alpha:.2f}  (target {(1-alpha)*100:.0f}% coverage)')
    print(f'  {"p":<6} {"Empirical":<12} {"q_hat (yr)":<14} {"Median Width (yr)"}')
    print('  ' + '-'*48)
    for _, row in sub.iterrows():
        print(f'  {row["p"]:<6.1f} {row["empirical"]:<12} {row["q_hat"]:<14.4f} {row["median_width"]}')
print('='*72)

# Plot coverage and width vs threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    ax  = axes[ax_idx]
    ax2 = ax.twinx()
    coverages  = [results_cp_cox_sensitivity[(p, alpha)]['coverage'] for p in thresholds]
    widths_med = [np.median(results_cp_cox_sensitivity[(p, alpha)]['widths']) for p in thresholds]
    l1, = ax.plot(thresholds, coverages, 'o-', color='#2980b9', lw=2, label='Empirical coverage')
    l2, = ax2.plot(thresholds, widths_med, 's--', color='#e67e22', lw=2, label='Median width (yr)')
    ax.axhline(1 - alpha, color='#2980b9', ls=':', lw=1.5, alpha=0.6,
               label=f'Target {(1-alpha)*100:.0f}%')
    ax.set(xlabel='Inversion threshold p', ylabel='Empirical coverage',
           title=f'Cox CP: Sensitivity to p\nalpha={alpha} (target {(1-alpha)*100:.0f}%)')
    ax2.set_ylabel('Median interval width (years)', color='#e67e22')
    ax.set_xticks(thresholds)
    ax.set_ylim(0, 1.05)
    ax.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=9, loc='lower left')

plt.suptitle('Cox PH CP: Inversion Threshold Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_cox_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cross-Model Predicted-Time Consistency ──────────────────────────────────
# Checklist item 3: "Could one model predict individual A converts in 2 years,
# while another predicts 10 years?" Strong aggregate concordance (Table 1)
# doesn't guarantee agreement on each subject's absolute predicted timeline.
# this checks that directly by comparing each of the four models' predicted
# median conversion time, subject by subject, across the same 96-subject
# eval set used throughout Section 9's conformal prediction work.
#
# Reuses the already-computed predicted median times stored in each model's
# results_cp_* dict (not recomputed), so this reflects exactly the same
# point predictions used for the CP intervals in Sections 5.4-5.8.

from scipy.stats import spearmanr
from itertools import combinations

pred_times = {
    'AFT':      results_cp_aft[0.20]['pred_median_eval'],
    'Cox PH':   results_cp_cox[0.20]['t_hat_eval'],
    'GBSA':     results_cp_gbsa[0.20]['t_hat_eval'],
    'DeepSurv': results_cp_deepsurv[0.20]['t_hat_eval'],
}

n_eval = len(pred_times['Cox PH'])
print(f'Comparing predicted median conversion time across all {n_eval} eval-set subjects '
      f'(observed-event and censored alike).\n')

# ── Pairwise Spearman correlation + mean absolute difference ────────────────
pair_rows = []
for m1, m2 in combinations(pred_times.keys(), 2):
    t1, t2 = pred_times[m1], pred_times[m2]
    rho, pval = spearmanr(t1, t2)
    mad = np.mean(np.abs(t1 - t2))
    max_diff = np.max(np.abs(t1 - t2))
    pair_rows.append(dict(pair=f'{m1} vs {m2}', spearman_rho=rho, spearman_p=pval,
                           mean_abs_diff_yr=mad, max_abs_diff_yr=max_diff))

pair_df = pd.DataFrame(pair_rows).sort_values('spearman_rho', ascending=False)
print('Pairwise agreement on predicted median conversion time:')
print(pair_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

print(f'\nSummary across all 6 model pairs:')
print(f'  Spearman rho range : {pair_df["spearman_rho"].min():.3f} - {pair_df["spearman_rho"].max():.3f}')
print(f'  Mean abs diff range: {pair_df["mean_abs_diff_yr"].min():.2f} - {pair_df["mean_abs_diff_yr"].max():.2f} years')
print(f'  Largest single-subject disagreement: {pair_df["max_abs_diff_yr"].max():.2f} years '
      f'({pair_df.loc[pair_df["max_abs_diff_yr"].idxmax(), "pair"]})')

# ── Which pair agrees most/least tightly? ────────────────────────────────────
tightest = pair_df.iloc[0]
loosest  = pair_df.iloc[-1]
print(f'\nTightest-agreeing pair : {tightest["pair"]}  (rho={tightest["spearman_rho"]:.3f}, '
      f'mean abs diff={tightest["mean_abs_diff_yr"]:.2f} yr)')
print(f'Loosest-agreeing pair  : {loosest["pair"]}  (rho={loosest["spearman_rho"]:.3f}, '
      f'mean abs diff={loosest["mean_abs_diff_yr"]:.2f} yr)')

# ── Visualization: pairwise scatter grid of predicted times ─────────────────
model_names = list(pred_times.keys())
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, m_row in enumerate(model_names):
    for j, m_col in enumerate(model_names):
        ax = axes[i, j]
        if i == j:
            ax.hist(pred_times[m_row], bins=15, color='#2980b9', alpha=0.7)
            ax.set_title(m_row, fontsize=10)
        elif i > j:
            x, y = pred_times[m_col], pred_times[m_row]
            ax.scatter(x, y, s=12, alpha=0.5, color='#2980b9')
            lims = [0, max(x.max(), y.max()) * 1.05]
            ax.plot(lims, lims, color='#e74c3c', ls='--', lw=1, alpha=0.7)
            rho = spearmanr(x, y)[0]
            ax.text(0.05, 0.92, f'ρ={rho:.2f}', transform=ax.transAxes, fontsize=9,
                    va='top', color='#2c3e50')
        else:
            ax.axis('off')
        if i == 3:
            ax.set_xlabel(m_col, fontsize=9)
        if j == 0:
            ax.set_ylabel(m_row, fontsize=9)
plt.suptitle('Cross-Model Agreement on Predicted Median Conversion Time\n'
             '(diagonal: distribution per model; lower triangle: pairwise scatter, dashed = perfect agreement)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cross_model_time_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sanity check: identify the subject driving the 61.2yr AFT vs Cox PH gap ──
t_aft = results_cp_aft[0.20]['pred_median_eval']
t_cox = results_cp_cox[0.20]['t_hat_eval']

diffs = np.abs(t_aft - t_cox)
worst_idx = np.argmax(diffs)

print(f'Subject index: {worst_idx}')
print(f'AFT predicted median:    {t_aft[worst_idx]:.2f} years')
print(f'Cox PH predicted median: {t_cox[worst_idx]:.2f} years')
print(f'Difference:              {diffs[worst_idx]:.2f} years')
print()
print(f'Actual observed duration: {y_dur_mci_eval.iloc[worst_idx] if hasattr(y_dur_mci_eval, "iloc") else y_dur_mci_eval[worst_idx]:.2f} years')
print(f'Event observed (1) or censored (0): {y_ev_mci_eval.iloc[worst_idx] if hasattr(y_ev_mci_eval, "iloc") else y_ev_mci_eval[worst_idx]}')

# Optional: see this subject's raw feature values, if you want to check for
# anything unusual (extreme covariate values, missingness that got imputed
# oddly, etc.) that might explain why AFT predicts so differently from Cox
print()
print('Feature values for this subject:')
print(X_mci_eval.iloc[worst_idx])

### 9.4a GBSA Inversion Threshold Sensitivity Analysis

Same rationale as 9.4 (Cox): GBSA also requires inverting its survival function
at a chosen probability threshold $p$ to obtain a pseudo-time, so the same
sensitivity check applies. Reuses `thresholds`, `_invert_surv`, and
`_conformal_quantile` from Section 9.4.

In [ ]:
# ── GBSA CP: Sensitivity Analysis on Inversion Threshold p ─────────────────
sensitivity_rows_gbsa = []
results_cp_gbsa_sensitivity = {}

for p in thresholds:
    t_hat_cal_p  = _invert_surv(surv_gbsa_cal_,  p=p)
    t_hat_eval_p = _invert_surv(surv_gbsa_eval_, p=p)

    obs_mask_p    = ev_cal_gbsa == 1
    scores_p      = t_hat_cal_p[obs_mask_p] - dur_cal_gbsa[obs_mask_p]
    n_cal_p       = len(scores_p)

    for alpha in [0.10, 0.20]:
        q_hat   = _conformal_quantile(scores_p, alpha, n_cal_p)
        lo      = np.clip(t_hat_eval_p - q_hat, 0, None)
        hi      = t_hat_eval_p + q_hat
        widths  = hi - lo
        obs     = ev_eval_gbsa == 1
        n_obs   = obs.sum()
        covered = ((dur_eval_gbsa[obs] >= lo[obs]) & (dur_eval_gbsa[obs] <= hi[obs])).sum()
        coverage = covered / n_obs if n_obs > 0 else np.nan

        results_cp_gbsa_sensitivity[(p, alpha)] = dict(
            q_hat=q_hat, lo=lo, hi=hi, widths=widths,
            coverage=coverage, n_obs=n_obs, covered=covered, t_hat_eval=t_hat_eval_p)
        sensitivity_rows_gbsa.append(dict(
            p=p, alpha=alpha, target=f'{(1-alpha)*100:.0f}%',
            empirical=f'{coverage:.1%}', q_hat=round(q_hat, 4),
            median_width=round(float(np.median(widths)), 3)))

sens_df_gbsa = pd.DataFrame(sensitivity_rows_gbsa)
print('='*72)
print('GBSA CP: SENSITIVITY ANALYSIS ON INVERSION THRESHOLD p')
print('='*72)
for alpha in [0.10, 0.20]:
    sub = sens_df_gbsa[sens_df_gbsa['alpha'] == alpha]
    print(f'\nalpha={alpha:.2f}  (target {(1-alpha)*100:.0f}% coverage)')
    print(f'  {"p":<6} {"Empirical":<12} {"q_hat (yr)":<14} {"Median Width (yr)"}')
    print('  ' + '-'*48)
    for _, row in sub.iterrows():
        print(f'  {row["p"]:<6.1f} {row["empirical"]:<12} {row["q_hat"]:<14.4f} {row["median_width"]}')
print('='*72)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    ax  = axes[ax_idx]
    ax2 = ax.twinx()
    coverages  = [results_cp_gbsa_sensitivity[(p, alpha)]['coverage'] for p in thresholds]
    widths_med = [np.median(results_cp_gbsa_sensitivity[(p, alpha)]['widths']) for p in thresholds]
    l1, = ax.plot(thresholds, coverages, 'o-', color='#27ae60', lw=2, label='Empirical coverage')
    l2, = ax2.plot(thresholds, widths_med, 's--', color='#e67e22', lw=2, label='Median width (yr)')
    ax.axhline(1 - alpha, color='#27ae60', ls=':', lw=1.5, alpha=0.6, label=f'Target {(1-alpha)*100:.0f}%')
    ax.set(xlabel='Inversion threshold p', ylabel='Empirical coverage',
           title=f'GBSA CP: Sensitivity to p\nalpha={alpha} (target {(1-alpha)*100:.0f}%)')
    ax2.set_ylabel('Median interval width (years)', color='#e67e22')
    ax.set_xticks(thresholds); ax.set_ylim(0, 1.05)
    ax.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=9, loc='lower left')
plt.suptitle('GBSA: Inversion Threshold Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'gbsa_sensitivity_p.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.4b DeepSurv: Inversion Threshold Sensitivity Analysis

Same rationale, applied to DeepSurv.

In [ ]:
# ── DeepSurv CP: Sensitivity Analysis on Inversion Threshold p ─────────────
sensitivity_rows_ds = []
results_cp_ds_sensitivity = {}

for p in thresholds:
    t_hat_cal_p  = _invert_surv(surv_ds_cal_,  p=p)
    t_hat_eval_p = _invert_surv(surv_ds_eval_, p=p)

    obs_mask_p    = ev_cal_ds == 1
    scores_p      = t_hat_cal_p[obs_mask_p] - dur_cal_ds[obs_mask_p]
    n_cal_p       = len(scores_p)

    for alpha in [0.10, 0.20]:
        q_hat   = _conformal_quantile(scores_p, alpha, n_cal_p)
        lo      = np.clip(t_hat_eval_p - q_hat, 0, None)
        hi      = t_hat_eval_p + q_hat
        widths  = hi - lo
        obs     = ev_eval_ds == 1
        n_obs   = obs.sum()
        covered = ((dur_eval_ds[obs] >= lo[obs]) & (dur_eval_ds[obs] <= hi[obs])).sum()
        coverage = covered / n_obs if n_obs > 0 else np.nan

        results_cp_ds_sensitivity[(p, alpha)] = dict(
            q_hat=q_hat, lo=lo, hi=hi, widths=widths,
            coverage=coverage, n_obs=n_obs, covered=covered, t_hat_eval=t_hat_eval_p)
        sensitivity_rows_ds.append(dict(
            p=p, alpha=alpha, target=f'{(1-alpha)*100:.0f}%',
            empirical=f'{coverage:.1%}', q_hat=round(q_hat, 4),
            median_width=round(float(np.median(widths)), 3)))

sens_df_ds = pd.DataFrame(sensitivity_rows_ds)
print('='*72)
print('DEEPSURV CP: SENSITIVITY ANALYSIS ON INVERSION THRESHOLD p')
print('='*72)
for alpha in [0.10, 0.20]:
    sub = sens_df_ds[sens_df_ds['alpha'] == alpha]
    print(f'\nalpha={alpha:.2f}  (target {(1-alpha)*100:.0f}% coverage)')
    print(f'  {"p":<6} {"Empirical":<12} {"q_hat (yr)":<14} {"Median Width (yr)"}')
    print('  ' + '-'*48)
    for _, row in sub.iterrows():
        print(f'  {row["p"]:<6.1f} {row["empirical"]:<12} {row["q_hat"]:<14.4f} {row["median_width"]}')
print('='*72)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    ax  = axes[ax_idx]
    ax2 = ax.twinx()
    coverages  = [results_cp_ds_sensitivity[(p, alpha)]['coverage'] for p in thresholds]
    widths_med = [np.median(results_cp_ds_sensitivity[(p, alpha)]['widths']) for p in thresholds]
    l1, = ax.plot(thresholds, coverages, 'o-', color='#8e44ad', lw=2, label='Empirical coverage')
    l2, = ax2.plot(thresholds, widths_med, 's--', color='#e67e22', lw=2, label='Median width (yr)')
    ax.axhline(1 - alpha, color='#8e44ad', ls=':', lw=1.5, alpha=0.6, label=f'Target {(1-alpha)*100:.0f}%')
    ax.set(xlabel='Inversion threshold p', ylabel='Empirical coverage',
           title=f'DeepSurv CP: Sensitivity to p\nalpha={alpha} (target {(1-alpha)*100:.0f}%)')
    ax2.set_ylabel('Median interval width (years)', color='#e67e22')
    ax.set_xticks(thresholds); ax.set_ylim(0, 1.05)
    ax.legend([l1, l2], [l1.get_label(), l2.get_label()], fontsize=9, loc='lower left')
plt.suptitle('DeepSurv: Inversion Threshold Sensitivity Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'deepsurv_sensitivity_p.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.5 Within-Interval Percentile Analysis

Coverage rate tells us *how often* the true event time falls inside the interval,
but not *where* inside the interval it tends to fall. A well calibrated interval
should have true events distributed **uniformly** across $[L_i, U_i]$,  not
systematically clustered near one edge.

We compute the within-interval percentile for each observed event evaluation subject:

$$\pi_i = \frac{T_i - L_i}{U_i - L_i} \in [0, 1]$$

where $\pi_i = 0$ means the true event happened exactly at the lower bound and
$\pi_i = 1$ means it happened at the upper bound. Under ideal calibration,
$\{\pi_i\}$ should be approximately uniform on $[0, 1]$, giving a mean of 0.50.

Systematic deviation from uniformity indicates directional bias in the model's
point predictions, so a skew toward 0 means the model consistently predicts
conversion *too late* (true events happen earlier than the interval center),
and skew toward 1 means the model predicts *too early*.

In [ ]:
# ── Within-Interval Percentile Analysis  ───────────────────────

fig, axes = plt.subplots(4, 2, figsize=(14, 20), sharey=True)

for row_idx, (model_name, results, ev_eval, dur_eval) in enumerate([
    ('AFT (Weibull)',   results_cp_aft,      ev_eval_aft,  dur_eval_aft),
    ('Cox PH (p=0.5)',  results_cp_cox,      ev_eval_cox,  dur_eval_cox),
    ('GBSA (p=0.5)',    results_cp_gbsa,     ev_eval_gbsa, dur_eval_gbsa),
    ('DeepSurv (p=0.5)',results_cp_deepsurv, ev_eval_ds,   dur_eval_ds),
]):
    for col_idx, alpha in enumerate([0.10, 0.20]):
        r        = results[alpha]
        obs      = ev_eval == 1
        lo_obs   = r['lo'][obs]
        hi_obs   = r['hi'][obs]
        true_obs = dur_eval[obs]
        width_obs = hi_obs - lo_obs

        valid = width_obs > 0
        pct   = (true_obs[valid] - lo_obs[valid]) / width_obs[valid]
        pct   = np.clip(pct, 0, 1)

        ax = axes[row_idx][col_idx]
        ax.hist(pct, bins=10, range=(0, 1), color='#2980b9', alpha=0.75,
                edgecolor='white', density=True)
        ax.axhline(1.0, color='#e74c3c', ls='--', lw=2, label='Uniform (ideal)')
        ax.set(xlabel='Within-interval percentile  (0=lower bound, 1=upper bound)',
               ylabel='Density',
               title=f'{model_name}  |  alpha={alpha}  target={(1-alpha)*100:.0f}%\n'
                     f'n={valid.sum()} observed events  '
                     f'(mean={pct.mean():.2f}, ideal=0.50)')
        ax.set_xlim(0, 1)
        ax.legend(fontsize=9)
        print(f'{model_name}  alpha={alpha:.2f}: '
              f'mean={pct.mean():.3f}  median={np.median(pct):.3f}  std={pct.std():.3f}')

plt.suptitle('Within-Interval Percentile Distribution: All 4 Models\n'
             '(uniform = well-calibrated; skewed = systematic bias)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_within_interval_pct_all4.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.6 Cross-Model Comparison on All 4 Models

All four models are conformalized using the same methodology: observed-events-only
calibration with raw time-scale nonconformity scores (predicted median − actual
time), producing **constant-width** intervals shifted around each subject's point
prediction. AFT predicts time directly; Cox PH, GBSA, and DeepSurv all require
survival-function inversion at $p=0.5$ to obtain a pseudo-time.

Under this formulation, interval width is determined entirely by $\hat{q}_\alpha$,
a single scalar computed from the calibration set, and does not vary across
patients which motivates the adaptive-width CSD approach in Section 10.

In [ ]:
# ── Cross-Model CP Comparison  ─────────────────────────────────
alpha = 0.10
r_aft = results_cp_aft[alpha]
r_cox = results_cp_cox[alpha]
r_gbsa = results_cp_gbsa[alpha]
r_ds   = results_cp_deepsurv[alpha]

rows = []
for model, r in [
    ('AFT (Weibull)', r_aft),
    ('Cox PH (p=0.5)', r_cox),
    ('GBSA (p=0.5)', r_gbsa),
    ('DeepSurv (p=0.5)', r_ds),
]:
    rows.append({
        'Model'             : model,
        'Target coverage'   : f'{(1-alpha)*100:.0f}%',
        'Empirical coverage': f'{r["coverage"]:.1%}',
        'Median width (yr)' : f'{np.median(r["widths"]):.3f}',
        'Min width (yr)'    : f'{np.min(r["widths"]):.3f}',
        'Max width (yr)'    : f'{np.max(r["widths"]):.3f}',
    })
comp_df = pd.DataFrame(rows)
print(comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
model_names = [r['Model'] for r in rows]
widths_med  = [float(r['Median width (yr)']) for r in rows]
colors = ['#2980b9', '#c0392b', '#27ae60', '#8e44ad']
ax.bar(model_names, widths_med, color=colors, alpha=0.8)
ax.set(ylabel='Median interval width (years)',
       title=f'CP Interval Width Comparison: All 4 Models (alpha={alpha})')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cp_width_comparison_all4.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.7 Key Findings on Conformal Prediction

**Coverage results** (70/20/10 split, MCI → AD cohort, true held-out eval set,
Cox PH, GBSA, and DeepSurv all evaluated on the same 96-subject eval population
as AFT):

| Model | α | Target | Empirical | Median Width |
|-------|---|--------|-----------|--------------|
| AFT (Weibull) | 0.10 | 90% | 87.2% | 12.5 yr |
| AFT (Weibull) | 0.20 | 80% | 82.1% | 7.2 yr |
| Cox PH (p=0.5) | 0.10 | 90% | 94.9% | 9.2 yr |
| Cox PH (p=0.5) | 0.20 | 80% | 87.2% | 7.0 yr |
| GBSA (p=0.5) | 0.10 | 90% | 87.2% | 8.9 yr |
| GBSA (p=0.5) | 0.20 | 80% | 74.4% | 5.6 yr |
| DeepSurv (p=0.5) | 0.10 | 90% | 89.7% | 10.8 yr |
| DeepSurv (p=0.5) | 0.20 | 80% | 74.4% | 5.2 yr |


**Sensitivity analysis (inversion threshold $p$):** Coverage varies
substantially with $p$ across all three inversion based models (Cox, GBSA,
DeepSurv, see 9.4/9.4a/9.4b). $p=0.5$ (the default) achieves near nominal
coverage without cherry picking for all three, though it is the single
optimal threshold only for Cox specifically, so it's a good default, but not a
universally optimal one.

**Within interval percentile analysis:** True conversion times cluster toward
the lower bound of prediction intervals across all four models with the mean $\pi$
consistently well below the ideal 0.50 (Section 9.5): AFT 0.187/0.229, Cox PH
0.235/0.259, GBSA 0.193/0.253, DeepSurv 0.174/0.260 ($\alpha=0.10/0.20$
respectively). This indicates all four models systematically overestimate
conversion time, meaning their point predictions are biased late. This is consistent
across four structurally different model families (linear Cox, parametric
AFT, tree-based GBSA, neural DeepSurv), indicating the bias is a property of
the data/interval-construction mechanism rather than any one model's
specification.

**Resolved (see Section 13):** Two candidate explanations were tested directly.
Longitudinal covariates via landmarking (Section 12) did not reduce the bias.
IPCW-weighted calibration (Section 13.1–13.3), targeting the censoring-driven
hypothesis originally proposed here, also had no measurable effect. The bias
was ultimately traced to the **symmetric construction of fixed-width intervals**
(Section 13.4/13.5) a structural mismatch with right-skewed time-to-event
distributions, which CSD's percentile-based intervals substantially correct
for all four models. This is the strongest, most generalizable finding in the
notebook; see Section 13.5 for full results.

**Constant-width limitation:** All four models produce intervals of fixed width
for all patients regardless of individual risk profile. A high-risk patient
(predicted conversion in 1 year) receives the same interval width as a
low-risk patient (predicted conversion in 15 years). Adaptive-width intervals,
where width scales with individual prediction uncertainty, are pursued in
Section 10 via the Conformalized Survival Distribution (CSD) framework of Qi
et al. (ICML 2024) implemented for all four models, and shown in Section 13
to also correct the late-prediction bias described above.

## 10. Conformalized Survival Distributions (CSD)

Section 9 demonstrated that both all models can produce **constant width**
conformal prediction intervals where every patient receives the same interval width
regardless of their individual risk profile. This is a known limitation of
mean-based conformal prediction (Chernozhukov et al., 2021).

The **Conformalized Survival Distribution (CSD)** framework of Qi et al.
(ICML 2024) addresses this by operating on the full survival curve rather than
a single point prediction. Instead of computing one global correction $\hat{q}$,
CSD computes a **separate correction at each percentile level** $\rho$ of the
survival curve:

$$\hat{q}(\rho) = \text{Quantile}\!\left[\rho;\, \{\hat{q}_M(\rho \mid x_j) - t_j\}_{j \in \mathcal{D}_\text{cal}}\right]$$

The adjusted percentile time for a new subject is then:

$$\hat{q}'_M(\rho \mid x_i) = \hat{q}_M(\rho \mid x_i) - \hat{q}(\rho)$$

Because the correction varies across $\rho$, the resulting intervals are
**patient specific in shape**: the survival curve is shifted differently at
different probability levels, producing intervals whose width reflects the
model's local uncertainty rather than a global average.

**Key properties (Qi et al., Theorems 3.1 and 3.2):**
- CSD does not change the ranking of predicted survival times → C-index is preserved
- For every percentile $\rho$, exactly $\rho$% of subjects fall within
  $[\hat{q}'_M(\rho \mid x_i), \infty]$ → D-calibration is guaranteed

**Censoring handling:** Given the ~60% censoring rate in the ADNI MCI cohort,
we use observed-events-only calibration (consistent with Section 9). KM sampling
(the full censoring-aware approach from Qi et al) is noted as a natural
extension for future work.

**Data used:** The same 70/20/10 split from Section 9. Calibration set (n=192)
is used to compute per-percentile corrections; evaluation set (n=96) is used
to measure empirical coverage of the calibrated intervals.

In [ ]:
# ── 10.1  CSD Core: Per-Percentile Calibration ──────────────────────────────
# Percentile levels to calibrate (survival probabilities)
# rho=0.1 means S(t)=0.1 → 90th percentile of event time distribution
percentile_levels = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

def compute_pcts(surv_matrix, rho_levels):
    """
    Compute Percentile Times (PCTs) for each subject and each rho level.
    PCT(rho, i) = first time t where S(t | x_i) <= rho
    Returns array of shape (n_subjects, n_rho_levels)
    """
    times = surv_matrix.index.values.astype(np.float64)
    n_subjects = surv_matrix.shape[1]
    pcts = np.full((n_subjects, len(rho_levels)), times[-1])
    for j in range(n_subjects):
        curve = surv_matrix.iloc[:, j].values
        for k, rho in enumerate(rho_levels):
            below = np.where(curve <= rho)[0]
            if len(below) > 0:
                pcts[j, k] = times[below[0]]
    return pcts

def csd_calibrate(pcts_cal, dur_cal, ev_cal, rho_levels):
    """
    Compute per percentile conformal corrections using observed events only.
    Returns q_hat array of shape (n_rho_levels,)
    """
    obs_mask = ev_cal == 1
    pcts_obs  = pcts_cal[obs_mask]       # shape: (n_obs, n_rho)
    dur_obs   = dur_cal[obs_mask]        # shape: (n_obs,)
    n_obs     = obs_mask.sum()

    q_hats = np.zeros(len(rho_levels))
    for k, rho in enumerate(rho_levels):
        scores = pcts_obs[:, k] - dur_obs   # predicted - actual
        level  = min(np.ceil((n_obs + 1) * rho) / n_obs, 1.0)
        q_hats[k] = np.quantile(scores, level)

    return q_hats

def csd_adjust_pcts(pcts_eval, q_hats):
    """
    Apply per percentile corrections to evaluation set PCTs.
    Returns adjusted PCTs of shape (n_eval_subjects, n_rho_levels)
    """
    return pcts_eval - q_hats[np.newaxis, :]   # broadcast correction

def csd_coverage(pcts_adj, dur_eval, ev_eval, rho_levels):
    """
    Compute empirical coverage at each rho level.
    Coverage(rho) = fraction of observed-event subjects with t_i >= PCT'(rho, i)
    (i.e., true event time falls in [PCT'(rho), inf])
    """
    obs_mask  = ev_eval == 1
    pcts_obs  = pcts_adj[obs_mask]
    dur_obs   = dur_eval[obs_mask]
    coverages = np.zeros(len(rho_levels))
    for k in range(len(rho_levels)):
        coverages[k] = (dur_obs >= pcts_obs[:, k]).mean()
    return coverages

print('CSD helper functions defined.')
print(f'Percentile levels: {percentile_levels}')

on AFT

In [ ]:
# ── 10.2  CSD Applied to AFT ─────────────────────────────────────────────────

# Step 1: Compute PCTs on calibration set
pcts_cal_aft  = compute_pcts(aft_surv_mci, percentile_levels)

# Step 2: Get cal/eval labels
dur_cal_csd_aft  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_csd_aft   = np.array(y_ev_mci_cal,   dtype=np.int32)

# Step 3: Compute PCTs on evaluation set
# We need AFT survival curves on the eval set
_test_eval_aft = X_mci_eval.copy().reset_index(drop=True)
aft_surv_eval  = aft_model_mci.predict_survival_function(_test_eval_aft)

dur_eval_csd_aft = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_csd_aft  = np.array(y_ev_mci_eval,  dtype=np.int32)

pcts_eval_aft = compute_pcts(aft_surv_eval, percentile_levels)

# Step 4: Calibrate: compute per percentile corrections
q_hats_aft = csd_calibrate(
    pcts_cal_aft, dur_cal_csd_aft, ev_cal_csd_aft, percentile_levels
)

print('AFT CSD corrections (q_hat per percentile level):')
print(f'{"rho":<8} {"q_hat (yr)":<12} {"interpretation"}')
print('-'*50)
for rho, q in zip(percentile_levels, q_hats_aft):
    direction = 'shift earlier' if q > 0 else 'shift later'
    print(f'{rho:<8.1f} {q:<12.4f} {direction}')

# Step 5: Apply corrections to eval set
pcts_adj_aft = csd_adjust_pcts(pcts_eval_aft, q_hats_aft)
pcts_adj_aft = np.clip(pcts_adj_aft, 0, None)  # no negative times

# Step 6: Coverage at each rho level
coverages_aft = csd_coverage(pcts_adj_aft, dur_eval_csd_aft, ev_eval_csd_aft, percentile_levels)

print('\nAFT CSD: Empirical coverage vs target:')
print(f'{"rho":<8} {"Target":<10} {"Empirical":<12}')
print('-'*32)
for rho, cov in zip(percentile_levels, coverages_aft):
    print(f'{rho:<8.1f} {rho:<10.0%} {cov:<12.1%}')

on Cox

In [ ]:
# ── 10.3  CSD Applied to Cox PH ──────────────────────────────────────────────
# Cal/eval mirror AFT (10.2): full 192-subject cal set, full 96-subject eval set
pcts_cal_cox  = compute_pcts(surv_cox_mci,      percentile_levels)
pcts_eval_cox = compute_pcts(surv_cox_mci_eval, percentile_levels)

dur_cal_csd_cox  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_csd_cox   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_csd_cox = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_csd_cox  = np.array(y_ev_mci_eval,  dtype=np.int32)

# Calibrate
q_hats_cox = csd_calibrate(
    pcts_cal_cox, dur_cal_csd_cox, ev_cal_csd_cox, percentile_levels
)

print('Cox PH CSD corrections (q_hat per percentile level):')
print(f'{"rho":<8} {"q_hat (yr)":<12} {"interpretation"}')
print('-'*50)
for rho, q in zip(percentile_levels, q_hats_cox):
    direction = 'shift earlier' if q > 0 else 'shift later'
    print(f'{rho:<8.1f} {q:<12.4f} {direction}')

# Apply corrections
pcts_adj_cox = csd_adjust_pcts(pcts_eval_cox, q_hats_cox)
pcts_adj_cox = np.clip(pcts_adj_cox, 0, None)

def enforce_monotonic_pcts(pcts_adj):
    """Rearrangement fix for quantile crossing (Chernozhukov et al.).
    pcts_adj columns go rho=0.1 (late time) -> rho=0.9 (early time),
    so each row must be non-increasing left to right."""
    return np.sort(pcts_adj, axis=1)[:, ::-1]

pcts_adj_aft = enforce_monotonic_pcts(pcts_adj_aft)
pcts_adj_cox = enforce_monotonic_pcts(pcts_adj_cox)

# Coverage
coverages_cox = csd_coverage(pcts_adj_cox, dur_eval_csd_cox, ev_eval_csd_cox, percentile_levels)

print('\nCox PH CSD: Empirical coverage vs target:')
print(f'{"rho":<8} {"Target":<10} {"Empirical":<12}')
print('-'*32)
for rho, cov in zip(percentile_levels, coverages_cox):
    print(f'{rho:<8.1f} {rho:<10.0%} {cov:<12.1%}')


### 10.3a CSD Applied to GBSA

Same CSD procedure as AFT (10.2) and Cox PH (10.3), applied to GBSA's survival
matrices. `compute_pcts`, `csd_calibrate`, `csd_adjust_pcts`, `csd_coverage`, and
`enforce_monotonic_pcts` were all defined generically in 10.1/10.3 and are reused
here unchanged.

In [ ]:
# ── 10.3a  CSD Applied to GBSA ────────────────────────────────────────────────
pcts_cal_gbsa  = compute_pcts(surv_gbsa_mci,      percentile_levels)
pcts_eval_gbsa = compute_pcts(surv_gbsa_mci_eval, percentile_levels)

dur_cal_csd_gbsa  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_csd_gbsa   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_csd_gbsa = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_csd_gbsa  = np.array(y_ev_mci_eval,  dtype=np.int32)

q_hats_gbsa = csd_calibrate(
    pcts_cal_gbsa, dur_cal_csd_gbsa, ev_cal_csd_gbsa, percentile_levels
)

print('GBSA CSD corrections (q_hat per percentile level):')
print(f'{"rho":<8} {"q_hat (yr)":<12} {"interpretation"}')
print('-'*50)
for rho, q in zip(percentile_levels, q_hats_gbsa):
    direction = 'shift earlier' if q > 0 else 'shift later'
    print(f'{rho:<8.1f} {q:<12.4f} {direction}')

pcts_adj_gbsa = csd_adjust_pcts(pcts_eval_gbsa, q_hats_gbsa)
pcts_adj_gbsa = np.clip(pcts_adj_gbsa, 0, None)
pcts_adj_gbsa = enforce_monotonic_pcts(pcts_adj_gbsa)

coverages_gbsa = csd_coverage(pcts_adj_gbsa, dur_eval_csd_gbsa, ev_eval_csd_gbsa, percentile_levels)

print('\nGBSA CSD: Empirical coverage vs target:')
print(f'{"rho":<8} {"Target":<10} {"Empirical":<12}')
print('-'*32)
for rho, cov in zip(percentile_levels, coverages_gbsa):
    print(f'{rho:<8.1f} {rho:<10.0%} {cov:<12.1%}')


### 10.3b CSD Applied to DeepSurv

Same procedure, applied to DeepSurv's survival matrices.

In [ ]:
# ── 10.3b  CSD Applied to DeepSurv ────────────────────────────────────────────
pcts_cal_ds  = compute_pcts(surv_mci,      percentile_levels)
pcts_eval_ds = compute_pcts(surv_mci_eval, percentile_levels)

dur_cal_csd_ds  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_csd_ds   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_csd_ds = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_csd_ds  = np.array(y_ev_mci_eval,  dtype=np.int32)

q_hats_ds = csd_calibrate(
    pcts_cal_ds, dur_cal_csd_ds, ev_cal_csd_ds, percentile_levels
)

print('DeepSurv CSD corrections (q_hat per percentile level):')
print(f'{"rho":<8} {"q_hat (yr)":<12} {"interpretation"}')
print('-'*50)
for rho, q in zip(percentile_levels, q_hats_ds):
    direction = 'shift earlier' if q > 0 else 'shift later'
    print(f'{rho:<8.1f} {q:<12.4f} {direction}')

pcts_adj_ds = csd_adjust_pcts(pcts_eval_ds, q_hats_ds)
pcts_adj_ds = np.clip(pcts_adj_ds, 0, None)
pcts_adj_ds = enforce_monotonic_pcts(pcts_adj_ds)

coverages_ds = csd_coverage(pcts_adj_ds, dur_eval_csd_ds, ev_eval_csd_ds, percentile_levels)

print('\nDeepSurv CSD: Empirical coverage vs target:')
print(f'{"rho":<8} {"Target":<10} {"Empirical":<12}')
print('-'*32)
for rho, cov in zip(percentile_levels, coverages_ds):
    print(f'{rho:<8.1f} {rho:<10.0%} {cov:<12.1%}')


In [ ]:
# ── 10.4  CSD Visualisation ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (model_name, coverages, rho_levels) in enumerate([
    ('AFT (Weibull)', coverages_aft, percentile_levels),
    ('Cox PH', coverages_cox, percentile_levels),
]):
    ax = axes[ax_idx]
    ax.plot(rho_levels, rho_levels, 'k--', lw=1.5, label='Perfect calibration')
    ax.plot(rho_levels, coverages, 'o-', color='#2980b9', lw=2,
            markersize=7, label='CSD empirical coverage')
    ax.fill_between(rho_levels, rho_levels, coverages,
                    alpha=0.15, color='#2980b9')
    ax.set(xlabel='Target coverage level (rho)',
           ylabel='Empirical coverage',
           title=f'{model_name}: CSD Calibration\n'
                 f'P-P plot: empirical vs target coverage')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.set_aspect('equal')

plt.suptitle('CSD Post-Processing: Coverage Calibration (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'csd_pp_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Adaptive width plot ───────────────────────────────────────────────────────
# Width = PCT(rho=0.9) - PCT(rho=0.1) for each subject (80% central interval)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (model_name, pcts_adj, pcts_orig, dur_eval, ev_eval) in enumerate([
    ('AFT (Weibull)', pcts_adj_aft, pcts_eval_aft,
     dur_eval_csd_aft, ev_eval_csd_aft),
    ('Cox PH', pcts_adj_cox, pcts_eval_cox,
     dur_eval_csd_cox, ev_eval_csd_cox),
]):
    ax = axes[ax_idx]

    # 80% central interval of event time: 10th pct = rho=0.9 col, 90th pct = rho=0.1 col
    lo_orig = pcts_orig[:, -1]  # rho=0.9 → 10th percentile of event time (early)
    hi_orig = pcts_orig[:, 0]   # rho=0.1 → 90th percentile of event time (late)
    lo_adj  = pcts_adj[:, -1]
    hi_adj  = pcts_adj[:, 0]
    width_orig = hi_orig - lo_orig
    width_adj  = hi_adj  - lo_adj
    mid_adj    = pcts_adj[:, 4]  # rho=0.5 symmetric, no change needed

    order = np.argsort(mid_adj)
    xs    = np.arange(len(order))

    ax.scatter(xs, width_orig[order], s=12, alpha=0.5,
               color='#bdc3c7', label='Pre-CSD width')
    ax.scatter(xs, width_adj[order],  s=12, alpha=0.7,
               color='#2980b9', label='Post-CSD width')
    ax.set(xlabel='Subjects (sorted by predicted median conversion time)',
           ylabel='80% interval width (years)',
           title=f'{model_name}: CSD Interval Width\n'
                 f'(PCT 10% to 90%)')
    ax.legend(fontsize=9)

plt.suptitle('CSD: Adaptive Interval Width per Subject (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'csd_adaptive_width.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Adaptive width plot: AFT + Cox PH ───────────────────────────────────────
# Width = PCT(rho=0.9) - PCT(rho=0.1) for each subject (80% central interval)
# NOTE: pcts columns run rho=0.1 (late time) -> rho=0.9 (early time), so the
# "lo" (early/10th pct) end is column -1 and "hi" (late/90th pct) is column 0.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, (model_name, pcts_adj, pcts_orig, dur_eval, ev_eval) in enumerate([
    ('AFT (Weibull)', pcts_adj_aft, pcts_eval_aft,
     dur_eval_csd_aft, ev_eval_csd_aft),
    ('Cox PH', pcts_adj_cox, pcts_eval_cox,
     dur_eval_csd_cox, ev_eval_csd_cox),
]):
    ax = axes[ax_idx]
    lo_orig = pcts_orig[:, -1]   # rho=0.9 -> 10th pct of event time (early)
    hi_orig = pcts_orig[:, 0]    # rho=0.1 -> 90th pct of event time (late)
    lo_adj  = pcts_adj[:, -1]
    hi_adj  = pcts_adj[:, 0]
    width_orig = hi_orig - lo_orig
    width_adj  = hi_adj  - lo_adj
    mid_adj    = pcts_adj[:, 4]  # rho=0.5 -> predicted median (symmetric, no swap needed)
    order = np.argsort(mid_adj)
    xs    = np.arange(len(order))
    ax.scatter(xs, width_orig[order], s=12, alpha=0.5,
               color='#bdc3c7', label='Pre-CSD width')
    ax.scatter(xs, width_adj[order],  s=12, alpha=0.7,
               color='#2980b9', label='Post-CSD width')
    ax.set(xlabel='Subjects (sorted by predicted median conversion time)',
           ylabel='80% interval width (years)',
           title=f'{model_name}: CSD Interval Width\n(PCT 10% to 90%)')
    ax.legend(fontsize=9)
plt.suptitle('CSD: Adaptive Interval Width per Subject (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'csd_adaptive_width.png', dpi=150, bbox_inches='tight')
plt.show()


### 10.5 Adaptive Width: GBSA + DeepSurv

Same plot as above, for the remaining two models.

In [ ]:
# ── Adaptive width plot: GBSA + DeepSurv ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax_idx, (model_name, pcts_adj, pcts_orig, dur_eval, ev_eval) in enumerate([
    ('GBSA', pcts_adj_gbsa, pcts_eval_gbsa,
     dur_eval_csd_gbsa, ev_eval_csd_gbsa),
    ('DeepSurv', pcts_adj_ds, pcts_eval_ds,
     dur_eval_csd_ds, ev_eval_csd_ds),
]):
    ax = axes[ax_idx]
    lo_orig = pcts_orig[:, -1]
    hi_orig = pcts_orig[:, 0]
    lo_adj  = pcts_adj[:, -1]
    hi_adj  = pcts_adj[:, 0]
    width_orig = hi_orig - lo_orig
    width_adj  = hi_adj  - lo_adj
    mid_adj    = pcts_adj[:, 4]
    order = np.argsort(mid_adj)
    xs    = np.arange(len(order))
    ax.scatter(xs, width_orig[order], s=12, alpha=0.5,
               color='#bdc3c7', label='Pre-CSD width')
    ax.scatter(xs, width_adj[order],  s=12, alpha=0.7,
               color='#2980b9', label='Post-CSD width')
    ax.set(xlabel='Subjects (sorted by predicted median conversion time)',
           ylabel='80% interval width (years)',
           title=f'{model_name}: CSD Interval Width\n(PCT 10% to 90%)')
    ax.legend(fontsize=9)
plt.suptitle('CSD Adaptive Interval Width per Subject: GBSA & DeepSurv (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'csd_adaptive_width_gbsa_ds.png', dpi=150, bbox_inches='tight')
plt.show()


### 10.6 Fixed Width CP vs Adaptive Width CSD: Same Scale Comparison

The single most important visual for demonstrating what CSD accomplished: each
model's fixed-width (Section 9) and CSD adaptive-width (Section 10) 80% intervals,
plotted on shared axes per model so the width difference is directly comparable
rather than an artifact of different plot scales.

In [ ]:
# ── Fixed vs Adaptive: Same-Scale Comparison ──────────────────
fig, axes = plt.subplots(4, 2, figsize=(16, 24), sharex=True, sharey=True)

def plot_intervals(ax, lo, hi, order_key, dur_eval, ev_eval, title):
    order = np.argsort(order_key)
    xs = np.arange(len(order))
    obs_mask = ev_eval[order] == 1
    covered = (dur_eval[order] >= lo[order]) & (dur_eval[order] <= hi[order])

    ax.vlines(xs[~obs_mask], lo[order][~obs_mask], hi[order][~obs_mask],
               color='lightgray', lw=1, label='Censored', zorder=1)
    cov_obs = obs_mask & covered
    ax.vlines(xs[cov_obs], lo[order][cov_obs], hi[order][cov_obs],
               color='#2980b9', lw=1.5, zorder=2)
    ax.scatter(xs[cov_obs], dur_eval[order][cov_obs], color='#2980b9',
               marker='D', s=30, label='Covered (observed)', zorder=3)
    not_cov = obs_mask & ~covered
    ax.vlines(xs[not_cov], lo[order][not_cov], hi[order][not_cov],
               color='#c0392b', lw=1.5, zorder=2)
    ax.scatter(xs[not_cov], dur_eval[order][not_cov], color='#c0392b',
               marker='x', s=40, label='Not covered (observed)', zorder=3)

    n_obs, n_cov = obs_mask.sum(), cov_obs.sum()
    ax.set_title(f'{title}\nCoverage = {n_cov/n_obs:.1%} ({n_cov}/{n_obs})')
    ax.legend(fontsize=8, loc='upper left')

model_configs = [
    ('AFT (Weibull)',
     results_cp_aft[0.20]['lo'], results_cp_aft[0.20]['hi'], dur_eval_aft, ev_eval_aft,
     pcts_adj_aft[:,-1], pcts_adj_aft[:,0], pcts_adj_aft[:,4], dur_eval_csd_aft, ev_eval_csd_aft),
    ('Cox PH',
     results_cp_cox[0.20]['lo'], results_cp_cox[0.20]['hi'], dur_eval_cox, ev_eval_cox,
     pcts_adj_cox[:,-1], pcts_adj_cox[:,0], pcts_adj_cox[:,4], dur_eval_csd_cox, ev_eval_csd_cox),
    ('GBSA',
     results_cp_gbsa[0.20]['lo'], results_cp_gbsa[0.20]['hi'], dur_eval_gbsa, ev_eval_gbsa,
     pcts_adj_gbsa[:,-1], pcts_adj_gbsa[:,0], pcts_adj_gbsa[:,4], dur_eval_csd_gbsa, ev_eval_csd_gbsa),
    ('DeepSurv',
     results_cp_deepsurv[0.20]['lo'], results_cp_deepsurv[0.20]['hi'], dur_eval_ds, ev_eval_ds,
     pcts_adj_ds[:,-1], pcts_adj_ds[:,0], pcts_adj_ds[:,4], dur_eval_csd_ds, ev_eval_csd_ds),
]

for row_idx, (name, lo_f, hi_f, dur_f, ev_f, lo_a, hi_a, mid_a, dur_a, ev_a) in enumerate(model_configs):
    plot_intervals(axes[row_idx, 0], lo_f, hi_f, (lo_f+hi_f)/2, dur_f, ev_f,
                   f'{name}: Fixed Width (80% target)')
    plot_intervals(axes[row_idx, 1], lo_a, hi_a, mid_a, dur_a, ev_a,
                   f'{name}: Adaptive CSD (80% central)')

axes[0, 0].set_ylim(0, 60)   # ← add this line

for ax in axes[-1, :]:
    ax.set_xlabel('Subjects (sorted)')
for ax in axes[:, 0]:
    ax.set_ylabel('Years from baseline')


plt.tight_layout()
plt.savefig(FIG_DIR / 'fixed_vs_adaptive_all4_same_scale.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 14.4  Fixed-width vs CSD interval figures, no coverage in titles ─────────
# Main text: AFT and GBSA. Appendix: Cox PH and DeepSurv.
# Needs model_configs from Section 10.6. Prints coverage and where misses fall,
# so you can check the numbers still match the paper.
from pathlib import Path

PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_YMAX = 25
RC = {'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5,
      'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.7}
CFG = {c[0]: c for c in model_configs}   # names: 'AFT (Weibull)', 'Cox PH', 'GBSA', 'DeepSurv'

def interval_fig_clean(names, fname, ymax=PAPER_YMAX):
    missing = [n for n in names if n not in CFG]
    if missing:
        raise KeyError(f'Not in model_configs: {missing}. Available: {list(CFG)}')
    with plt.rc_context(RC):
        fig, axes = plt.subplots(len(names), 2, figsize=(7.0, 1.75 * len(names) + 0.45),
                                 sharex=True, sharey=True, squeeze=False)
        for r, name in enumerate(names):
            _, lo_f, hi_f, dur_f, ev_f, lo_a, hi_a, mid_a, dur_a, ev_a = CFG[name]
            panels = [(lo_f, hi_f, (np.asarray(lo_f) + np.asarray(hi_f)) / 2, dur_f, ev_f, 'fixed width'),
                      (lo_a, hi_a, mid_a, dur_a, ev_a, 'CSD adaptive')]
            for c, (lo, hi, key, dur, ev, label) in enumerate(panels):
                ax = axes[r, c]
                lo, hi, key, dur, ev = map(np.asarray, (lo, hi, key, dur, ev))
                order = np.argsort(key)
                xs = np.arange(len(order))
                lo, hi, dur, ev = lo[order], hi[order], dur[order], ev[order]
                obs = ev == 1
                cov = (dur >= lo) & (dur <= hi)
                ax.vlines(xs[~obs], lo[~obs], hi[~obs], color='lightgray', lw=0.8, label='Censored', zorder=1)
                ci, nc = obs & cov, obs & ~cov
                ax.vlines(xs[ci], lo[ci], hi[ci], color='#2980b9', lw=1.0, zorder=2)
                ax.scatter(xs[ci], dur[ci], color='#2980b9', marker='D', s=9, label='Covered', zorder=3)
                ax.vlines(xs[nc], lo[nc], hi[nc], color='#c0392b', lw=1.0, zorder=2)
                ax.scatter(xs[nc], dur[nc], color='#c0392b', marker='x', s=12, label='Not covered', zorder=3)
                ax.set_title(f'{name}: {label}')
                ax.set_ylim(0, ymax)
                print(f'{name:<14} {label:<13} coverage {ci.sum()}/{obs.sum()} = {ci.sum() / obs.sum():.1%} | '
                      f'missed below {int((obs & (dur < lo)).sum())}, missed above {int((obs & (dur > hi)).sum())}')
        axes[0, 0].legend(loc='upper left', frameon=False)
        for ax in axes[-1, :]:
            ax.set_xlabel('Subjects (sorted)')
        for ax in axes[:, 0]:
            ax.set_ylabel('Years from baseline')
        fig.tight_layout()
        path = PAPER_FIG_DIR / fname
        fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
        plt.show()
        plt.close(fig)
        print('saved', path, '\n')

interval_fig_clean(['AFT (Weibull)', 'GBSA'], 'AFT_GBSA_CP_CSD_curves.png')   # main text
interval_fig_clean(['Cox PH', 'DeepSurv'], 'COX_DS_CP_CSD_curves.png')       # appendix

### 10.8 Fixed Width vs CSD vs CiPOT: Table 2 Numbers and Figure 4


In [ ]:
# ── 10.8  Fixed width vs CSD vs CiPOT: Table 2 numbers, widths, Figure 4 ─────
# REPLACES the old "14.5 Reviewer follow-ups" cell (cell after 10.7). Paste it
# right after the 10.7 interval-figure cell. Needs Sections 9 and 10 to have run.
# Runs in seconds. Outputs:
#   OUT_DIR/review_followups.csv          -> Table 2 (pi, coverage), miss counts, widths
#   paper/WI_three_methods_CoxPH.png      -> new Figure 4 (main text)
#   paper/WI_three_methods_all_models.png -> appendix figure (all four models)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RHO_EXT = np.array([0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95])
ALPHAS = {0.10: (0.95, 0.05), 0.20: (0.90, 0.10)}   # alpha -> (rho for lower end, rho for upper end)
COL = {r: k for k, r in enumerate(RHO_EXT)}
METHODS = ['Fixed width', 'CSD', 'CiPOT']

MODELS = {   # name: (surv_cal, surv_eval, fixed-width CP results, eval event, eval duration)
    'Cox PH':        (surv_cox_mci,  surv_cox_mci_eval,  results_cp_cox,      ev_eval_cox,  dur_eval_cox),
    'AFT (Weibull)': (aft_surv_mci,  aft_surv_eval,      results_cp_aft,      ev_eval_aft,  dur_eval_aft),
    'GBSA':          (surv_gbsa_mci, surv_gbsa_mci_eval, results_cp_gbsa,     ev_eval_gbsa, dur_eval_gbsa),
    'DeepSurv':      (surv_mci,      surv_mci_eval,      results_cp_deepsurv, ev_eval_ds,   dur_eval_ds),
}
dur_cal_all = np.asarray(y_dur_mci_cal, dtype=float)
ev_cal_all = np.asarray(y_ev_mci_cal, dtype=int)


def _finite_level(rho, n):
    return min(np.ceil((n + 1) * rho) / n, 1.0)


def _surv_at(surv_matrix, times_obs):
    """Predicted survival probability of each subject at its own observed time (step function)."""
    grid = surv_matrix.index.values.astype(float)
    vals = surv_matrix.values
    idx = np.searchsorted(grid, np.asarray(times_obs, dtype=float), side='right') - 1
    out = np.ones(len(times_obs))
    has = idx >= 0
    out[has] = vals[idx[has], np.arange(len(times_obs))[has]]
    return out


def _pi_and_summary(lo, hi, dur, ev):
    """Per-subject pi for observed events, plus coverage, miss direction, and widths."""
    lo, hi, dur, ev = (np.asarray(a, dtype=float) for a in (lo, hi, dur, ev))
    w_all = hi - lo                                   # widths for every eval subject
    o = ev == 1
    lo_o, hi_o, t = lo[o], hi[o], dur[o]
    cov = (t >= lo_o) & (t <= hi_o)
    w = hi_o - lo_o
    ok = w > 0
    pi = np.clip((t[ok] - lo_o[ok]) / w[ok], 0, 1)
    summ = dict(n_obs=int(o.sum()),
                coverage_pct=round(100 * float(cov.mean()), 1),
                mean_pi=round(float(pi.mean()), 3),
                miss_below=int((t < lo_o).sum()),
                miss_above=int((t > hi_o).sum()),
                median_width_yr=round(float(np.median(w_all)), 2),
                mean_width_yr=round(float(np.mean(w_all)), 2))
    return pi, summ


rows, PI = [], {}   # PI[(model, alpha, method)] = per-subject pi array
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    # CSD on the extended grid
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    # Simplified CiPOT: recalibrated probability level = rho-quantile of S_i(T_i) over calibration events
    o_cal = ev_cal_all == 1
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_levels = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cipot = enforce_monotonic_pcts(compute_pcts(s_eval, p_levels))

    for a, (r_lo, r_hi) in ALPHAS.items():
        bounds = {'Fixed width': (R_fixed[a]['lo'], R_fixed[a]['hi']),
                  'CSD': (pcts_csd[:, COL[r_lo]], pcts_csd[:, COL[r_hi]]),
                  'CiPOT': (pcts_cipot[:, COL[r_lo]], pcts_cipot[:, COL[r_hi]])}
        for m in METHODS:
            pi, summ = _pi_and_summary(*bounds[m], dur_e, ev_e)
            PI[(name, a, m)] = pi
            rows.append(dict(model=name, alpha=a, method=m, **summ))

review_df = pd.DataFrame(rows)
review_df.to_csv(OUT_DIR / 'review_followups.csv', index=False)
pd.set_option('display.width', 200)
print(review_df.to_string(index=False))
print('\nsaved', OUT_DIR / 'review_followups.csv')

# ── Numbers the paper quotes ─────────────────────────────────────────────────
print('\n=== Table 2: mean pi (coverage %) ===')
for name in MODELS:
    for a in ALPHAS:
        cells = []
        for m in METHODS:
            r = review_df[(review_df.model == name) & (review_df.alpha == a) & (review_df.method == m)].iloc[0]
            cells.append(f'{r.mean_pi:.3f} ({r.coverage_pct:.1f})')
        print(f'{name:<14} {a:.2f} & ' + ' & '.join(cells) + r' \\')

print('\n=== Misses pooled across the four models (Section 5.6) ===')
pooled = review_df.groupby(['alpha', 'method'])[['miss_below', 'miss_above']].sum()
print(pooled.to_string())

print('\n=== Median interval width, years (optional sentence in 5.6) ===')
print(review_df.pivot_table(index=['model', 'alpha'], columns='method',
                            values='median_width_yr').round(2).to_string())

# ── Figures ──────────────────────────────────────────────────────────────────
PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
RC = {'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5,
      'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.7}
METHOD_COLORS = {'Fixed width': '#95a5a6', 'CSD': '#2980b9', 'CiPOT': '#16a085'}
FIG_ALPHA = 0.20


def _hist_panel(ax, pi, title):
    ax.hist(pi, bins=10, range=(0, 1), density=True, color=title[1], alpha=0.85, edgecolor='white')
    ax.axhline(1.0, color='#e74c3c', ls='--', lw=1.1, label='Uniform (ideal)')
    ax.axvline(pi.mean(), color='black', ls=':', lw=1.0, label='Mean $\\pi$')
    ax.set_title(f'{title[0]} (mean $\\pi$ = {pi.mean():.2f})')
    ax.set_xlim(0, 1)


# Figure 4 (main text): Cox PH, three methods, one row
with plt.rc_context(RC):
    fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.1), sharey=True)
    for ax, m in zip(axes, METHODS):
        _hist_panel(ax, PI[('Cox PH', FIG_ALPHA, m)], (m, METHOD_COLORS[m]))
        ax.set_xlabel('Within-interval percentile $\\pi$')
    axes[0].set_ylabel('Density')
    axes[0].legend(loc='upper right', frameon=False)
    fig.tight_layout()
    path = PAPER_FIG_DIR / 'WI_three_methods_CoxPH.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show(); plt.close(fig)
    print('saved', path)

# Appendix figure: all four models (rows) x three methods (columns)
with plt.rc_context(RC):
    fig, axes = plt.subplots(len(MODELS), 3, figsize=(7.0, 1.6 * len(MODELS) + 0.3),
                             sharex=True, sharey=True)
    for r, name in enumerate(MODELS):
        for c, m in enumerate(METHODS):
            _hist_panel(axes[r, c], PI[(name, FIG_ALPHA, m)], (f'{name}: {m}', METHOD_COLORS[m]))
        axes[r, 0].set_ylabel('Density')
    for ax in axes[-1, :]:
        ax.set_xlabel('Within-interval percentile $\\pi$')
    axes[0, 0].legend(loc='upper right', frameon=False)
    fig.tight_layout()
    path = PAPER_FIG_DIR / 'WI_three_methods_all_models.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show(); plt.close(fig)
    print('saved', path)


### 10.9 Figure 5: Per-Subject Intervals, Three Methods


In [ ]:
# ── 10.9  Figure 5: per-subject intervals, fixed width vs CSD vs CiPOT ───────
# Paste directly AFTER cell 10.8 (uses MODELS, RHO_EXT, COL, _surv_at,
# _finite_level, dur_cal_all, ev_cal_all, PAPER_FIG_DIR, RC from it).
# Same interval computation as 10.8, so coverage printed here matches Table 2.
# Outputs:
#   paper/CI_three_methods_CoxPH.png      -> new Figure 5 (main text, one model)
#   paper/CI_three_methods_all_models.png -> appendix (all four models)
import numpy as np
import matplotlib.pyplot as plt

FIG5_ALPHA = 0.20
R_LO, R_HI = 0.90, 0.10            # interval = [rho 0.90 time, rho 0.10 time] at alpha = 0.20
FIG5_YMAX = 25

BOUNDS = {}   # BOUNDS[(model, method)] = (lo, hi, sort_key, dur, ev)
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    o_cal = ev_cal_all == 1
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_levels = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cipot = enforce_monotonic_pcts(compute_pcts(s_eval, p_levels))
    mid = COL[0.50]
    lo_f, hi_f = np.asarray(R_fixed[FIG5_ALPHA]['lo']), np.asarray(R_fixed[FIG5_ALPHA]['hi'])
    BOUNDS[(name, 'Fixed width')] = (lo_f, hi_f, (lo_f + hi_f) / 2, dur_e, ev_e)
    BOUNDS[(name, 'CSD')] = (pcts_csd[:, COL[R_LO]], pcts_csd[:, COL[R_HI]], pcts_csd[:, mid], dur_e, ev_e)
    BOUNDS[(name, 'CiPOT')] = (pcts_cipot[:, COL[R_LO]], pcts_cipot[:, COL[R_HI]], pcts_cipot[:, mid], dur_e, ev_e)


def _interval_panel(ax, lo, hi, key, dur, ev, title):
    lo, hi, key, dur, ev = (np.asarray(a, dtype=float) for a in (lo, hi, key, dur, ev))
    order = np.argsort(key)
    xs = np.arange(len(order))
    lo, hi, dur, ev = lo[order], hi[order], dur[order], ev[order]
    obs = ev == 1
    cov = (dur >= lo) & (dur <= hi)
    ax.vlines(xs[~obs], lo[~obs], hi[~obs], color='lightgray', lw=0.8, label='Censored', zorder=1)
    ci, nc = obs & cov, obs & ~cov
    ax.vlines(xs[ci], lo[ci], hi[ci], color='#2980b9', lw=1.0, zorder=2)
    ax.scatter(xs[ci], dur[ci], color='#2980b9', marker='D', s=9, label='Covered', zorder=3)
    ax.vlines(xs[nc], lo[nc], hi[nc], color='#c0392b', lw=1.0, zorder=2)
    ax.scatter(xs[nc], dur[nc], color='#c0392b', marker='x', s=12, label='Not covered', zorder=3)
    ax.set_title(title)
    ax.set_ylim(0, FIG5_YMAX)
    print(f'{title:<28} coverage {ci.sum()}/{obs.sum()} = {ci.sum() / obs.sum():.1%} | '
          f'missed below {int((obs & (dur < lo)).sum())}, missed above {int((obs & (dur > hi)).sum())}')


def interval_fig_three(names, fname):
    with plt.rc_context(RC):
        fig, axes = plt.subplots(len(names), 3, figsize=(7.0, 1.75 * len(names) + 0.45),
                                 sharex=True, sharey=True, squeeze=False)
        for r, name in enumerate(names):
            for c, m in enumerate(METHODS):
                title = m if len(names) == 1 else f'{name}: {m}'
                _interval_panel(axes[r, c], *BOUNDS[(name, m)], title)
            axes[r, 0].set_ylabel('Years from baseline')
        axes[0, 0].legend(loc='upper left', frameon=False)
        for ax in axes[-1, :]:
            ax.set_xlabel('Subjects (sorted)')
        fig.tight_layout()
        path = PAPER_FIG_DIR / fname
        fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.03, facecolor='white')
        plt.show(); plt.close(fig)
        print('saved', path, '\n')


interval_fig_three(['Cox PH'], 'CI_three_methods_CoxPH.png')            # Figure 5, main text
interval_fig_three(list(MODELS), 'CI_three_methods_all_models.png')     # appendix


### 10.10 Reviewer Checks
Re-centered fixed width, IPCW-weighted mean pi, grid-maximum medians, interval widths, and the full CiPOT. Prints the LaTeX rows for Table 2 and the sensitivity table.


In [ ]:
# ── 10.10  Reviewer checks: re-centered fixed width, IPCW-weighted π,
#           grid-maximum medians, interval widths, full CiPOT ────────────────
# Paste AFTER cells 10.8 and 10.9 (reuses MODELS, RHO_EXT, COL, ALPHAS,
# METHODS, _surv_at, _finite_level, dur_cal_all, ev_cal_all, OUT_DIR from 10.8).
# Runs in seconds. Output: OUT_DIR/reviewer_checks.csv + LaTeX rows printed
# for Table 2 (π, coverage, width) and the sensitivity table.
#
# What each check answers
#   Re-centered fixed width : shift the symmetric interval by the median calibration
#                             residual. If mean π moves to ~0.5, the fixed-width problem
#                             is LOCATION; if it stays low, SHAPE matters too.
#   IPCW-weighted π         : weight each observed evaluation event by 1/G(T_i) so later
#                             converters (the ones censoring removes) count more. Tests
#                             whether the bias survives correcting for who gets observed.
#   Grid-maximum medians    : how many subjects' curves never drop below 0.5, so their
#                             predicted median defaults to the last grid time.
#   Full CiPOT              : the published version, which also uses censored calibration
#                             subjects (conformity scores drawn uniformly on [0, S(c_i)]).
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter

R_CIPOT = 100           # pseudo-uniform draws per subject for full CiPOT
G_FLOOR = 0.05          # same clipping as Section 13.2

# Censoring survival G(t), fit on the TRAINING split only (event and censoring flipped)
_kmf_c = KaplanMeierFitter().fit(np.asarray(y_dur_mci_train, float),
                                 event_observed=1 - np.asarray(y_ev_mci_train, int))
def _G(t):
    g = _kmf_c.survival_function_at_times(np.asarray(t, float)).values
    return np.clip(g, G_FLOOR, 1.0)


def _median_from_curve(surv):
    """Predicted median: first grid time with S <= 0.5, else the last grid time."""
    grid = surv.index.values.astype(float)
    vals = surv.values
    out = np.full(vals.shape[1], grid[-1])
    for j in range(vals.shape[1]):
        below = np.where(vals[:, j] <= 0.5)[0]
        if len(below):
            out[j] = grid[below[0]]
    return out, grid[-1]


def _q_level(n, alpha):
    return min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)


def _evaluate(lo, hi, dur, ev):
    lo, hi, dur, ev = (np.asarray(a, float) for a in (lo, hi, dur, ev))
    o = ev == 1
    t, l, h = dur[o], lo[o], hi[o]
    w = h - l
    ok = w > 0
    pi = np.clip((t[ok] - l[ok]) / w[ok], 0, 1)
    wts = 1.0 / _G(t[ok])
    return dict(mean_pi=float(pi.mean()),
                ipcw_mean_pi=float(np.sum(wts * pi) / np.sum(wts)),
                coverage_pct=100 * float(((t >= l) & (t <= h)).mean()),
                miss_below=int((t < l).sum()), miss_above=int((t > h).sum()),
                median_width_yr=float(np.median(hi - lo)))


rows, grid_rows = [], []
for name, (s_cal, s_eval, R_fixed, ev_e, dur_e) in MODELS.items():
    ev_e, dur_e = np.asarray(ev_e, int), np.asarray(dur_e, float)
    o_cal = ev_cal_all == 1

    # ── point predictions (curve-inverted median, same rule for every model)
    t_cal, gmax_c = _median_from_curve(s_cal)
    t_ev, gmax_e = _median_from_curve(s_eval)
    resid = dur_cal_all[o_cal] - t_cal[o_cal]          # actual - predicted
    shift = float(np.median(resid))                    # re-centering offset (years)

    # ── grid-maximum check
    at_max = np.isclose(t_ev, gmax_e)
    grid_rows.append(dict(model=name, n_eval=len(t_ev), n_at_grid_max=int(at_max.sum()),
                          n_obs_events_at_grid_max=int((at_max & (ev_e == 1)).sum()),
                          grid_max_yr=round(float(gmax_e), 1),
                          median_shift_yr=round(shift, 2)))

    # ── adaptive percentile times (same as 10.8)
    q_csd = csd_calibrate(compute_pcts(s_cal, RHO_EXT), dur_cal_all, ev_cal_all, RHO_EXT)
    pcts_csd = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(compute_pcts(s_eval, RHO_EXT), q_csd), 0, None))
    ipot = _surv_at(s_cal, dur_cal_all)[o_cal]
    p_simple = np.array([np.quantile(ipot, _finite_level(r, len(ipot))) for r in RHO_EXT])
    pcts_cp = enforce_monotonic_pcts(compute_pcts(s_eval, p_simple))

    # full CiPOT: events -> R copies of S_i(T_i); censored -> S_i(c_i) * pseudo-uniform grid
    s_at_obs = _surv_at(s_cal, dur_cal_all)
    u = (np.arange(R_CIPOT) + 0.5) / R_CIPOT
    scores_full = np.concatenate([np.repeat(s_at_obs[o_cal], R_CIPOT),
                                  np.outer(s_at_obs[~o_cal], u).ravel()])
    n_subj = len(dur_cal_all)
    p_full = np.array([np.quantile(scores_full, _finite_level(r, n_subj)) for r in RHO_EXT])
    pcts_full = enforce_monotonic_pcts(compute_pcts(s_eval, p_full))

    for a, (r_lo, r_hi) in ALPHAS.items():
        n_ev = int(o_cal.sum())
        # plain fixed width from curve medians (matches Table 2 for Cox, GBSA, DeepSurv)
        q_plain = float(np.quantile(np.abs(resid), _q_level(n_ev, a)))
        # re-centered fixed width
        c_cal = t_cal[o_cal] + shift
        q_rc = float(np.quantile(np.abs(dur_cal_all[o_cal] - c_cal), _q_level(n_ev, a)))
        c_ev = t_ev + shift

        bounds = {
            'Fixed width (Table 2)': (R_fixed[a]['lo'], R_fixed[a]['hi']),
            'Fixed width (curve median)': (np.clip(t_ev - q_plain, 0, None), t_ev + q_plain),
            'Fixed width, re-centered': (np.clip(c_ev - q_rc, 0, None), c_ev + q_rc),
            'CSD': (pcts_csd[:, COL[r_lo]], pcts_csd[:, COL[r_hi]]),
            'CiPOT (simplified)': (pcts_cp[:, COL[r_lo]], pcts_cp[:, COL[r_hi]]),
            'CiPOT (full, with censored)': (pcts_full[:, COL[r_lo]], pcts_full[:, COL[r_hi]]),
        }
        for m, (lo, hi) in bounds.items():
            rows.append(dict(model=name, alpha=a, method=m, **_evaluate(lo, hi, dur_e, ev_e)))

        # sensitivity: fixed width without the grid-maximum subjects
        keep = ~at_max
        rows.append(dict(model=name, alpha=a, method='Fixed width, excl. grid-max subjects',
                         **_evaluate(np.asarray(R_fixed[a]['lo'])[keep], np.asarray(R_fixed[a]['hi'])[keep],
                                     dur_e[keep], ev_e[keep])))

chk = pd.DataFrame(rows).round(3)
chk.to_csv(OUT_DIR / 'reviewer_checks.csv', index=False)
grid_df = pd.DataFrame(grid_rows)
grid_df.to_csv(OUT_DIR / 'grid_max_check.csv', index=False)
pd.set_option('display.width', 220)
print(chk.to_string(index=False))
print('\nGrid-maximum medians on the evaluation set:')
print(grid_df.to_string(index=False))


def _cell(df, model, a, method, key='mean_pi'):
    r = df[(df.model == model) & (df.alpha == a) & (df.method == method)].iloc[0]
    return r


# ── LaTeX rows: Table 2 (π (coverage) and median width, four constructions)
print('\n=== Table 2 rows: π (coverage) & width, for Fixed | Re-centered | CSD | CiPOT ===')
order = ['Fixed width (Table 2)', 'Fixed width, re-centered', 'CSD', 'CiPOT (simplified)']
short = {'Cox PH': 'Cox PH', 'AFT (Weibull)': 'AFT', 'GBSA': 'GBSA', 'DeepSurv': 'DeepSurv'}
for name in MODELS:
    for i, a in enumerate(ALPHAS):
        cells = []
        for m in order:
            r = _cell(chk, name, a, m)
            cells += [f'{r.mean_pi:.3f} ({r.coverage_pct:.1f})', f'{r.median_width_yr:.1f}']
        label = short[name] if i == 0 else ''
        print(f'{label:<9} & {a:.2f} & ' + ' & '.join(cells) + r' \\')

# ── LaTeX rows: sensitivity table (α = 0.20)
print('\n=== Sensitivity table rows (alpha = 0.20): IPCW-weighted mean π; full CiPOT ===')
for name in MODELS:
    f = _cell(chk, name, 0.20, 'Fixed width (Table 2)')
    rc = _cell(chk, name, 0.20, 'Fixed width, re-centered')
    c = _cell(chk, name, 0.20, 'CSD')
    p = _cell(chk, name, 0.20, 'CiPOT (simplified)')
    full = _cell(chk, name, 0.20, 'CiPOT (full, with censored)')
    print(f'{short[name]:<9} & {f.ipcw_mean_pi:.3f} & {rc.ipcw_mean_pi:.3f} & {c.ipcw_mean_pi:.3f} & '
          f'{p.ipcw_mean_pi:.3f} & {full.mean_pi:.3f} & {full.ipcw_mean_pi:.3f} & {full.coverage_pct:.1f}' + r' \\')

# ── Sanity check: AFT plain fixed width uses the model's own median in Table 2,
#    so 'Fixed width (curve median)' may differ slightly for AFT only.
print('\nIf the AFT "curve median" row differs from Table 2, that is why (Section 9.1 uses the '
      'parametric median). The re-centered row uses curve medians for every model.')


## 11. Train/Cal/Eval Split Ratio Comparison: 70/20/10 vs 80/10/10

**Motivation:** The original split was 80/10/10 (train/cal/eval). It was changed
to 70/20/10 to test whether a larger calibration set improves conformal coverage
stability. This section builds the infrastructure to answer that question directly
by rerunning the full pipeline under both ratios and comparing q_hat and interval
width for all four models.

**Important: this section requires retraining all four models under the
alternate split.** 

**How this works:** `make_split()` reproduces the same two-step stratified split
logic from Section 4.2 (Cell 30) with the cal/eval fraction parameterized, so
both ratios use the identical splitting procedure, only the fractions differ.

In [ ]:
# ── Parameterized split function (mirrors Section 4.2 / Cell 30 exactly) ─────
from sklearn.model_selection import train_test_split as _tts_split

def make_split(X_imp, y_ev, y_dur, cal_frac, eval_frac, seed):
    """
    Reproduces the same two-step stratified split as Cell 30, with cal/eval
    fractions parameterized. train_frac is implied as 1 - cal_frac - eval_frac.
    """
    tmp_frac = cal_frac + eval_frac
    X_train, X_tmp, ev_train, ev_tmp, dur_train, dur_tmp = _tts_split(
        X_imp, y_ev, y_dur, test_size=tmp_frac, random_state=seed, stratify=y_ev)
    X_cal, X_eval, ev_cal, ev_eval, dur_cal, dur_eval = _tts_split(
        X_tmp, ev_tmp, dur_tmp, test_size=eval_frac / tmp_frac,
        random_state=seed, stratify=ev_tmp)
    return dict(
        X_train=X_train, X_cal=X_cal, X_eval=X_eval,
        y_ev_train=ev_train, y_ev_cal=ev_cal, y_ev_eval=ev_eval,
        y_dur_train=dur_train, y_dur_cal=dur_cal, y_dur_eval=dur_eval,
    )

# Current pipeline split (for reference / sanity check against Cell 30's output)
split_70_20_10 = make_split(X_mci_imp, y_ev_mci, y_dur_mci, cal_frac=0.20, eval_frac=0.10, seed=RANDOM_SEED)
# Original split, to compare against
split_80_10_10 = make_split(X_mci_imp, y_ev_mci, y_dur_mci, cal_frac=0.10, eval_frac=0.10, seed=RANDOM_SEED)

for label, split in [('70/20/10', split_70_20_10), ('80/10/10', split_80_10_10)]:
    print(f'{label}: train n={len(split["X_train"])}, '
          f'cal n={len(split["X_cal"])} (events={split["y_ev_cal"].sum()}), '
          f'eval n={len(split["X_eval"])} (events={split["y_ev_eval"].sum()})')


### 11.1 Retrain all 4 models under each split

This retrains everything from scratch under both splits. Structure mirrors
Sections 6.1–6.4 (Cox, GBSA, AFT, DeepSurv) exactly, only the input data
changes. **Set `RETRAIN_SPLIT_COMPARISON = True` and expect this to take a
while** (DeepSurv + GBSA HPO dominate the runtime).

For each split, this fits all 4 models on that split's train set, then runs the
same fixed width CP (Section 9 style) and CSD (Section 10 style) calibration on
that split's cal/eval sets, and records q_hat + median width for α=0.10 and
α=0.20.

In [ ]:
# ── Retrain + conformalize all 4 models under both splits ────────────────────
RETRAIN_SPLIT_COMPARISON = False  # set True to actually run this (takes a while)

def fit_and_conformalize_all_models(split, label, n_trials_gbsa=40, n_trials_ds=25, n_trials_cox=30):
    """Fits Cox, GBSA, AFT, DeepSurv on split['X_train'] and runs fixed-width CP
    on split['X_cal'] / split['X_eval']. Returns a list of result-row dicts."""
    X_train, X_cal, X_eval = split['X_train'], split['X_cal'], split['X_eval']
    ev_train, ev_cal, ev_eval = split['y_ev_train'], split['y_ev_cal'], split['y_ev_eval']
    dur_train, dur_cal, dur_eval = split['y_dur_train'], split['y_dur_cal'], split['y_dur_eval']

    rows = []

    # --- Cox PH ---
    _, _, cox_m, cox_s, _ = run_cox_ph(X_train, ev_train, dur_train, label=f'{label}-Cox', n_trials=n_trials_cox)
    _, surv_cal_c  = calc_cox_ph_c(cox_m, cox_s, X_cal,  ev_cal,  dur_cal)
    _, surv_eval_c = calc_cox_ph_c(cox_m, cox_s, X_eval, ev_eval, dur_eval)
    t_cal_c, t_eval_c = _invert_surv(surv_cal_c), _invert_surv(surv_eval_c)
    scores_c = t_cal_c[ev_cal==1] - np.array(dur_cal)[ev_cal==1]
    for alpha in [0.10, 0.20]:
        q = _conformal_quantile(scores_c, alpha, len(scores_c))
        lo, hi = np.clip(t_eval_c - q, 0, None), t_eval_c + q
        obs = ev_eval == 1
        cov = ((np.array(dur_eval)[obs] >= lo[obs]) & (np.array(dur_eval)[obs] <= hi[obs])).mean()
        rows.append(dict(split=label, model='Cox PH', alpha=alpha, q_hat=q,
                          coverage=cov, median_width=np.median(hi-lo)))

    # --- GBSA ---
    _, _, _, gbsa_m, _, _ = gbsa_survival_cv(X_train, ev_train, dur_train,
        feature_names=list(X_train.columns), label=f'{label}-GBSA', n_trials=n_trials_gbsa)
    _, surv_cal_g  = calc_gbsa_c(gbsa_m, X_cal,  ev_cal,  dur_cal)
    _, surv_eval_g = calc_gbsa_c(gbsa_m, X_eval, ev_eval, dur_eval)
    t_cal_g, t_eval_g = _invert_surv(surv_cal_g), _invert_surv(surv_eval_g)
    scores_g = t_cal_g[ev_cal==1] - np.array(dur_cal)[ev_cal==1]
    for alpha in [0.10, 0.20]:
        q = _conformal_quantile(scores_g, alpha, len(scores_g))
        lo, hi = np.clip(t_eval_g - q, 0, None), t_eval_g + q
        obs = ev_eval == 1
        cov = ((np.array(dur_eval)[obs] >= lo[obs]) & (np.array(dur_eval)[obs] <= hi[obs])).mean()
        rows.append(dict(split=label, model='GBSA', alpha=alpha, q_hat=q,
                          coverage=cov, median_width=np.median(hi-lo)))

    # --- AFT (Weibull) ---
    _data = X_train.copy(); _data['duration'] = np.array(dur_train); _data['event'] = np.array(ev_train)
    aft_m = _fit_aft_with_cv(X_train, ev_train, dur_train, f'{label}-AFT')
    _cal_df  = X_cal.copy();  _cal_df['duration'] = np.array(dur_cal);  _cal_df['event'] = np.array(ev_cal)
    _eval_df = X_eval.copy(); _eval_df['duration'] = np.array(dur_eval); _eval_df['event'] = np.array(ev_eval)
    mu_cal_a  = np.log(np.clip(aft_m.predict_median(X_cal).values, 1e-6, None))
    mu_eval_a = np.log(np.clip(aft_m.predict_median(X_eval).values, 1e-6, None))
    pred_cal_a, pred_eval_a = np.exp(mu_cal_a), np.exp(mu_eval_a)
    scores_a = pred_cal_a[np.array(ev_cal)==1] - np.array(dur_cal)[np.array(ev_cal)==1]
    for alpha in [0.10, 0.20]:
        q = _conformal_quantile(scores_a, alpha, len(scores_a))
        lo, hi = np.clip(pred_eval_a - q, 0, None), pred_eval_a + q
        obs = np.array(ev_eval) == 1
        cov = ((np.array(dur_eval)[obs] >= lo[obs]) & (np.array(dur_eval)[obs] <= hi[obs])).mean()
        rows.append(dict(split=label, model='AFT (Weibull)', alpha=alpha, q_hat=q,
                          coverage=cov, median_width=np.median(hi-lo)))

    # --- DeepSurv ---
    _, _, ds_m, ds_s, _, _, _ = run_deepsurv(X_train, ev_train, dur_train, f'{label}-DeepSurv', n_trials=n_trials_ds)
    _, surv_cal_d  = calc_deepsurv_c(ds_m, ds_s, X_cal,  ev_cal,  dur_cal)
    _, surv_eval_d = calc_deepsurv_c(ds_m, ds_s, X_eval, ev_eval, dur_eval)
    t_cal_d, t_eval_d = _invert_surv(surv_cal_d), _invert_surv(surv_eval_d)
    scores_d = t_cal_d[np.array(ev_cal)==1] - np.array(dur_cal)[np.array(ev_cal)==1]
    for alpha in [0.10, 0.20]:
        q = _conformal_quantile(scores_d, alpha, len(scores_d))
        lo, hi = np.clip(t_eval_d - q, 0, None), t_eval_d + q
        obs = np.array(ev_eval) == 1
        cov = ((np.array(dur_eval)[obs] >= lo[obs]) & (np.array(dur_eval)[obs] <= hi[obs])).mean()
        rows.append(dict(split=label, model='DeepSurv', alpha=alpha, q_hat=q,
                          coverage=cov, median_width=np.median(hi-lo)))

    return rows

if RETRAIN_SPLIT_COMPARISON:
    all_rows = []
    all_rows += fit_and_conformalize_all_models(split_70_20_10, '70/20/10')
    all_rows += fit_and_conformalize_all_models(split_80_10_10, '80/10/10')
    split_comparison_df = pd.DataFrame(all_rows)
    split_comparison_df.to_csv(FIG_DIR.parent / 'split_ratio_comparison.csv', index=False)
    print(split_comparison_df.to_string(index=False))
else:
    print('RETRAIN_SPLIT_COMPARISON is False: set to True and re-run to generate the split comparison table.')
    print('Expected runtime: dominated by GBSA (40 trials x2 splits) and DeepSurv (25 trials x2 splits) HPO.')


### 11.2 Split Comparison Table + Visualization

Run this after `RETRAIN_SPLIT_COMPARISON = True` has produced `split_comparison_df`.

In [ ]:
# ── Pivot table: q_hat and width by model, split, and alpha ──────────────────
if 'split_comparison_df' in dir():
    pivot_q = split_comparison_df.pivot_table(
        index=['model', 'alpha'], columns='split', values='q_hat').round(4)
    pivot_w = split_comparison_df.pivot_table(
        index=['model', 'alpha'], columns='split', values='median_width').round(3)
    pivot_c = split_comparison_df.pivot_table(
        index=['model', 'alpha'], columns='split', values='coverage').round(3)

    print('=== q_hat by split ===')
    print(pivot_q)
    print('\n=== Median width (yr) by split ===')
    print(pivot_w)
    print('\n=== Empirical coverage by split ===')
    print(pivot_c)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax_idx, alpha in enumerate([0.10, 0.20]):
        ax = axes[ax_idx]
        sub = split_comparison_df[split_comparison_df['alpha'] == alpha]
        models = sub['model'].unique()
        x = np.arange(len(models))
        width = 0.35
        w_70 = [sub[(sub['model']==m) & (sub['split']=='70/20/10')]['median_width'].values[0] for m in models]
        w_80 = [sub[(sub['model']==m) & (sub['split']=='80/10/10')]['median_width'].values[0] for m in models]
        ax.bar(x - width/2, w_80, width, label='80/10/10 (original)', color='#95a5a6')
        ax.bar(x + width/2, w_70, width, label='70/20/10 (current)', color='#2980b9')
        ax.set(xticks=x, xticklabels=models, ylabel='Median interval width (years)',
               title=f'Interval Width by Split: alpha={alpha}')
        ax.legend()
        plt.setp(ax.get_xticklabels(), rotation=15)
    plt.suptitle('Does a larger calibration set (70/20/10) reduce interval width vs the original 80/10/10?', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'split_ratio_width_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('split_comparison_df not found: run Section 11.1 with RETRAIN_SPLIT_COMPARISON=True first.')


## 12. Longitudinal Conformal Prediction (Landmark Analysis)

Every model and every CP/CSD result up to this point uses **baseline-only**
covariates: `df_bl`, one row per subject, frozen at the first visit. This
section tests whether incorporating each patient's covariates **as of the
prediction time** (updated at each available visit) reduces the systematic
late prediction bias found in Section 9.5 (mean within interval percentile
well below the ideal 0.50 for all four models).

**Approach:** standard landmarking (van Houwelingen, 2007). For a landmark
time τ, only subjects still event free and under observation past τ are
included, using their most recent covariate snapshot at or before τ, and
predicting *residual* time from τ rather than raw time from baseline.

**Scope:** Cox PH only, MCI → AD only, as a proof of concept: CSD/GBSA/AFT/
DeepSurv extension is deferred until this validates.

In [ ]:
# ── 12.1  Landmark Dataset Construction ──────────────────────────────────────
LANDMARK_TIMES = [0.0, 0.5, 1.0, 1.5, 2.0]  # years since baseline

# NOTE: 'surv_mci' gets overwritten in Section 6.4 (DeepSurv) with a survival
# PROBABILITY MATRIX of the same name. rebuild the survival LABELS fresh here
# under a distinct name
from preprocessing import build_survival_labels
surv_mci_labels = build_survival_labels(df, df_bl, 'MCI', 'AD',
                                         exclusion_rids=REVERSION_EXCLUSIONS)

# Reuse the SAME baseline derived normalization constants as Section 4.1 (Cell 28)
# so landmark snapshot z-scores stay on the same scale as baseline features used
# everywhere else in the notebook, not re-centered per landmark subgroup, which
# would leak landmark-cohort-specific distribution info.
_mu_abeta, _sd_abeta = df_bl['ABETA'].mean(), df_bl['ABETA'].std()
_mu_av45,  _sd_av45  = df_bl['AV45'].mean(),  df_bl['AV45'].std()
_mu_tau,   _sd_tau   = df_bl['TAU'].mean(),   df_bl['TAU'].std()

def build_landmark_snapshot(df, surv, landmark_time):
    """
    For landmark time tau:
      1. Filter surv to subjects still at risk (duration > tau)
      2. Pull each at-risk subject's most recent visit with Years_bl <= tau
      3. Reconstruct the core_features feature set on that snapshot
      4. Return covariates + residual duration/event, tagged with landmark_time
    """
    at_risk_rids = surv.index[surv['duration'] > landmark_time]
    if len(at_risk_rids) == 0:
        return None

    snap = (
        df[(df['RID'].isin(at_risk_rids)) & (df['Years_bl'] <= landmark_time)]
        .sort_values(['RID', 'Years_bl'])
        .groupby('RID').tail(1)
        .copy()
    )
    snap = snap.set_index('RID').reindex(at_risk_rids).dropna(subset=['Years_bl'])

    if len(snap) == 0:
        return None

    # ── Reconstruct engineered features (mirrors Section 4.1 / Cell 28) ──────
    # NOTE: _ICV ratios are tiny (~0.001-0.01), which produces enormous, unstable
    # Cox coefficients (seen as exp(coef) ~1e+20 on APOE4_x_hippo_ICV in the first
    # landmark fit). Rescale by x100 so features sit near unit scale. This does
    # NOT change the ratio features below (ratio of two x100-scaled values is
    # unaffected) and Cox coefficients are otherwise scale-invariant in their
    # z-statistics/significance, only the raw coefficient magnitudes shift to a
    # sane, interpretable range.
    ICV_SCALE = 100
    for region in ['Hippocampus', 'Entorhinal', 'Ventricles', 'Fusiform', 'MidTemp', 'WholeBrain']:
        snap[f'{region}_ICV'] = (snap[region] / snap['ICV']) * ICV_SCALE
    snap['hippo_entorhinal_ratio'] = snap['Hippocampus_ICV'] / snap['Entorhinal_ICV']
    snap['midtemp_fusiform_ratio'] = snap['MidTemp_ICV'] / snap['Fusiform_ICV']

    snap['ABETA_z'] = (snap['ABETA'] - _mu_abeta) / (_sd_abeta + 1e-9)
    snap['AV45_z']  = (snap['AV45']  - _mu_av45)  / (_sd_av45  + 1e-9)
    snap['ABETA_z_flipped'] = -snap['ABETA_z']
    snap['amyloid_composite'] = snap[['ABETA_z_flipped', 'AV45_z']].mean(axis=1, skipna=True)
    snap['amyloid_positive'] = (snap['ABETA'] < 880).astype(float)
    _fallback = snap['ABETA'].isna() & snap['AV45'].notna()
    snap.loc[_fallback, 'amyloid_positive'] = (snap.loc[_fallback, 'AV45'] > 1.11).astype(float)

    snap['TAU_z'] = (snap['TAU'] - _mu_tau) / (_sd_tau + 1e-9)
    snap['PTGENDER_num'] = (snap['PTGENDER'] == 'Male').astype(int)

    snap['APOE4_x_amyloid']   = snap['APOE4'] * snap['amyloid_composite'].fillna(0)
    snap['APOE4_x_tau']       = snap['APOE4'] * snap['TAU_z'].fillna(0)
    snap['APOE4_x_hippo_ICV'] = snap['APOE4'] * snap['Hippocampus_ICV'].fillna(
        snap['Hippocampus_ICV'].median())
    snap['APOE4_amyloid_pos'] = snap['APOE4'] * snap['amyloid_positive'].fillna(0)

    # ── Residual outcome from this landmark ───────────────────────────────────
    snap['duration_resid'] = surv.loc[snap.index, 'duration'] - landmark_time
    snap['event']          = surv.loc[snap.index, 'event']
    snap['landmark_time']  = landmark_time
    snap['n_visits_used']  = df[df['RID'].isin(snap.index) & (df['Years_bl'] <= landmark_time)] \
                                .groupby('RID').size().reindex(snap.index).values
    snap['RID']            = snap.index

    return snap[core_features + ['duration_resid', 'event', 'landmark_time', 'n_visits_used', 'RID']]

landmark_frames = [build_landmark_snapshot(df, surv_mci_labels, tau) for tau in LANDMARK_TIMES]
landmark_stack = pd.concat([f for f in landmark_frames if f is not None], ignore_index=True)

print(f'Landmark stack: {len(landmark_stack)} (subject, landmark) rows '
      f'from {landmark_stack["RID"].nunique()} unique subjects across {len(LANDMARK_TIMES)} landmarks')
print(landmark_stack.groupby('landmark_time').agg(
    n_subjects=('RID', 'nunique'), n_events=('event', 'sum'),
    mean_visits=('n_visits_used', 'mean')))

In [ ]:
# ── 12.2  Subject Level Cal/Eval Split + Cox Landmark Supermodel ─────────────
from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split as _lm_tts

# Split at the SUBJECT level (not row level), the same subject appears at
# multiple landmarks, so splitting rows would leak a person's data across
# cal/eval, breaking the exchangeability conformal prediction depends on.
unique_rids = landmark_stack['RID'].unique()
rid_events = surv_mci_labels.loc[unique_rids, 'event']
lm_train_rids, lm_caleval_rids = _lm_tts(
    unique_rids, test_size=0.30, random_state=RANDOM_SEED, stratify=rid_events)
lm_cal_rids, lm_eval_rids = _lm_tts(
    lm_caleval_rids, test_size=0.333, random_state=RANDOM_SEED,
    stratify=surv_mci_labels.loc[lm_caleval_rids, 'event'])

lm_train = landmark_stack[landmark_stack['RID'].isin(lm_train_rids)].copy()
lm_cal   = landmark_stack[landmark_stack['RID'].isin(lm_cal_rids)].copy()
lm_eval  = landmark_stack[landmark_stack['RID'].isin(lm_eval_rids)].copy()

print(f'Landmark train: {len(lm_train)} rows / {lm_train["RID"].nunique()} subjects')
print(f'Landmark cal  : {len(lm_cal)} rows / {lm_cal["RID"].nunique()} subjects')
print(f'Landmark eval : {len(lm_eval)} rows / {lm_eval["RID"].nunique()} subjects')

# Median impute for this proof of concept (train derived medians only, applied
# to cal/eval, same no-leakage principle as the main pipeline's imputer).
for col in core_features:
    med = lm_train[col].median()
    lm_train[col] = lm_train[col].fillna(med)
    lm_cal[col]   = lm_cal[col].fillna(med)
    lm_eval[col]  = lm_eval[col].fillna(med)

# ── Fit landmark supermodel Cox PH ────────────────────────────────────────────
# landmark_time included as a covariate (van Houwelingen's landmark supermodel).
# cluster_col='RID' accounts for within subject correlation across landmarks;
# falls back to no clustering if the installed lifelines version doesn't
# support it, point estimates/predictions are unaffected either way, only
# standard error estimates would be slightly anti conservative without it.
cox_fit_df = lm_train[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']].copy()
cox_lm = CoxPHFitter(penalizer=0.1)
try:
    cox_lm.fit(cox_fit_df, duration_col='duration_resid', event_col='event',
               cluster_col='RID')
    print('Fit with cluster_col=RID (robust SEs for within-subject correlation).')
except TypeError:
    cox_lm.fit(cox_fit_df.drop(columns='RID'), duration_col='duration_resid',
               event_col='event')
    print('cluster_col not supported by this lifelines version: fit without it '
          '(point estimates/predictions valid; SEs may be mildly anti-conservative).')

print(cox_lm.print_summary())


### 12.3 Conformal Prediction on the Landmark Model, Comparison to Baseline

Applies the same fixed width CP + within interval percentile methodology from
Sections 9.3/9.5 to the landmark supermodel's predictions, evaluated separately
at each landmark time, to check whether the late prediction bias (mean π well
below 0.50) shrinks as more longitudinal information becomes available.

In [ ]:
# ── 12.3  CP on Landmark Model, Per-Landmark Coverage & Bias ────────────────
lm_surv_cal_matrices = {}
lm_results_by_landmark = {}

for tau in LANDMARK_TIMES:
    cal_tau  = lm_cal[lm_cal['landmark_time'] == tau]
    eval_tau = lm_eval[lm_eval['landmark_time'] == tau]
    if len(cal_tau) < 5 or len(eval_tau) < 5:
        print(f'tau={tau}: insufficient cal/eval rows ({len(cal_tau)}/{len(eval_tau)}), skipping.')
        continue

    # Predicted median residual time via survival function inversion at p=0.5
    surv_cal_tau  = cox_lm.predict_survival_function(cal_tau[core_features + ['landmark_time']])
    surv_eval_tau = cox_lm.predict_survival_function(eval_tau[core_features + ['landmark_time']])
    t_hat_cal_tau  = _invert_surv(surv_cal_tau)
    t_hat_eval_tau = _invert_surv(surv_eval_tau)

    dur_cal_tau  = cal_tau['duration_resid'].values
    dur_eval_tau = eval_tau['duration_resid'].values
    ev_cal_tau   = cal_tau['event'].values
    ev_eval_tau  = eval_tau['event'].values

    obs_mask = ev_cal_tau == 1
    if obs_mask.sum() < 3:
        print(f'tau={tau}: fewer than 3 observed events in cal, skipping.')
        continue
    scores_tau = t_hat_cal_tau[obs_mask] - dur_cal_tau[obs_mask]
    q_hat_tau  = _conformal_quantile(scores_tau, 0.20, obs_mask.sum())  # alpha=0.20, matches Section 9

    lo = np.clip(t_hat_eval_tau - q_hat_tau, 0, None)
    hi = t_hat_eval_tau + q_hat_tau
    obs_eval = ev_eval_tau == 1
    n_obs = obs_eval.sum()
    if n_obs == 0:
        print(f'tau={tau}: no observed events in eval, skipping bias check.')
        continue
    covered = ((dur_eval_tau[obs_eval] >= lo[obs_eval]) & (dur_eval_tau[obs_eval] <= hi[obs_eval])).sum()
    coverage = covered / n_obs

    width = hi[obs_eval] - lo[obs_eval]
    valid = width > 0
    pct = (dur_eval_tau[obs_eval][valid] - lo[obs_eval][valid]) / width[valid]
    pct = np.clip(pct, 0, 1)

    lm_results_by_landmark[tau] = dict(coverage=coverage, n_obs=n_obs,
                                        mean_pct=pct.mean(), median_pct=np.median(pct))
    print(f'tau={tau:.1f}yr | n_eval_events={n_obs} | coverage={coverage:.1%} '
          f'| mean_pi={pct.mean():.3f} (baseline Cox Section 9.5: mean_pi=0.259 at alpha=0.20)')

# ── Comparison plot: mean pi by landmark time vs baseline (tau=0 equivalent) ──
if lm_results_by_landmark:
    taus = sorted(lm_results_by_landmark.keys())
    means = [lm_results_by_landmark[t]['mean_pct'] for t in taus]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(taus, means, 'o-', color='#2980b9', lw=2, markersize=8, label='Landmark Cox: mean π')
    ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (uniform, π=0.50)')
    ax.axhline(0.259, color='#95a5a6', ls=':', lw=1.5, label='Baseline-only Cox (Section 9.5, α=0.20)')
    ax.set(xlabel='Landmark time τ (years since baseline)',
           ylabel='Mean within-interval percentile (π)',
           title='Does More Longitudinal Information Reduce the Late-Prediction Bias?\n'
                 '(Landmark Cox PH, MCI → AD, α=0.20)')
    ax.set_ylim(0, 0.6)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'landmark_bias_by_tau.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No landmark had enough data to evaluate. Check LANDMARK_TIMES and cohort size.')


### 12.4 Sensitivity to Visit Count / Feature Completeness

Tests whether the landmark bias reduction result in 12.3 holds when restricting
to subjects with more visits or more complete longitudinal data, vs. allowing
sparse/incomplete history. Gated behind a flag since it re-runs 12.1–12.3
multiple times.

In [ ]:
# ── 12.4  Visit-Count / Completeness Sensitivity ─────────────────────────────
RUN_VISIT_SENSITIVITY = True  # set True to run (re-fits the landmark model per threshold)

MIN_VISIT_THRESHOLDS = [1, 2, 3, 4]

def run_landmark_pipeline_for_min_visits(min_visits):
    """Restricts landmark_stack to (subject, landmark) rows with at least
    min_visits visits used in the snapshot, then refits + re-evaluates."""
    stack = landmark_stack[landmark_stack['n_visits_used'] >= min_visits].copy()
    if stack['RID'].nunique() < 20:
        return None  # too few subjects left to be meaningful

    rids = stack['RID'].unique()
    events = surv_mci_labels.loc[rids, 'event']
    train_rids, caleval_rids = train_test_split(
        rids, test_size=0.30, random_state=RANDOM_SEED, stratify=events)
    cal_rids, eval_rids = train_test_split(
        caleval_rids, test_size=0.333, random_state=RANDOM_SEED,
        stratify=surv_mci_labels.loc[caleval_rids, 'event'])

    train = stack[stack['RID'].isin(train_rids)].copy()
    cal   = stack[stack['RID'].isin(cal_rids)].copy()
    ev_df = stack[stack['RID'].isin(eval_rids)].copy()
    for col in core_features:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        cal[col]   = cal[col].fillna(med)
        ev_df[col] = ev_df[col].fillna(med)

    model = CoxPHFitter(penalizer=0.1)
    try:
        model.fit(train[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']],
                  duration_col='duration_resid', event_col='event', cluster_col='RID')
    except TypeError:
        model.fit(train[core_features + ['duration_resid', 'event', 'landmark_time']],
                  duration_col='duration_resid', event_col='event')

    obs_cal = cal['event'].values == 1
    if obs_cal.sum() < 3:
        return None
    t_hat_cal = _invert_surv(model.predict_survival_function(cal[core_features + ['landmark_time']]))
    t_hat_eval = _invert_surv(model.predict_survival_function(ev_df[core_features + ['landmark_time']]))
    scores = t_hat_cal[obs_cal] - cal['duration_resid'].values[obs_cal]
    q_hat = _conformal_quantile(scores, 0.20, obs_cal.sum())

    lo = np.clip(t_hat_eval - q_hat, 0, None)
    hi = t_hat_eval + q_hat
    obs_eval = ev_df['event'].values == 1
    if obs_eval.sum() == 0:
        return None
    width = hi[obs_eval] - lo[obs_eval]
    valid = width > 0
    true_t = ev_df['duration_resid'].values[obs_eval][valid]
    pct = np.clip((true_t - lo[obs_eval][valid]) / width[valid], 0, 1)

    return dict(min_visits=min_visits, n_subjects=stack['RID'].nunique(),
                n_rows=len(stack), n_eval_events=obs_eval.sum(),
                mean_pi=pct.mean(), median_pi=np.median(pct))

if RUN_VISIT_SENSITIVITY:
    visit_sensitivity_results = [
        r for r in (run_landmark_pipeline_for_min_visits(m) for m in MIN_VISIT_THRESHOLDS) if r
    ]
    visit_sensitivity_df = pd.DataFrame(visit_sensitivity_results)
    print(visit_sensitivity_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(visit_sensitivity_df['min_visits'], visit_sensitivity_df['mean_pi'],
            'o-', color='#8e44ad', lw=2, markersize=8)
    ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
    ax.set(xlabel='Minimum visits required per subject',
           ylabel='Mean within-interval percentile (π)',
           title='Sensitivity of Late-Prediction Bias to Minimum Visit Count')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'landmark_visit_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('RUN_VISIT_SENSITIVITY is False, set to True to run this (re-fits 4x).')


### 12.5 K-Fold Pooling + Bootstrap CIs (Addressing Small Eval-Set Sizes)

Sections 12.3/12.4 evaluate on a single fixed eval split, which thins fast at
later landmarks (down to 17 events at τ=2.0) and higher visit count thresholds
(down to 40 events at min_visits=4), too few to trust a precise trend.

This section runs **5-fold subject level stratified CV**: every subject is
held out exactly once across the 5 folds, and each fold's held out predictions
are pooled together per landmark time. This uses effectively all 958 subjects'
information once, instead of only the ~96 who happened to land in one fixed
eval split, roughly a 5x increase in usable eval event count per landmark.

Each pooled landmark's mean π also gets a **bootstrap 95% CI** (1000
resamples), so the τ-vs-bias trend can be read with actual uncertainty bounds
instead of implying more precision than the sample supports.

In [ ]:
# ── 12.5  K-Fold Pooling + Bootstrap CIs ──────────────────────────────────────
RUN_KFOLD_BOOTSTRAP = True  # landmark CIs reported in the paper (refits Cox 5x, fast)

N_FOLDS = 5
BOOTSTRAP_ITERS = 1000

def bootstrap_ci(values, n_iter=BOOTSTRAP_ITERS, seed=RANDOM_SEED):
    values = np.array(values)
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_iter)
    ])
    return values.mean(), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

if RUN_KFOLD_BOOTSTRAP:
    from sklearn.model_selection import StratifiedKFold, train_test_split as _kf_tts

    unique_rids_arr = landmark_stack['RID'].unique()
    fold_events = surv_mci_labels.loc[unique_rids_arr, 'event'].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    pooled_pct_by_tau = {tau: [] for tau in LANDMARK_TIMES}
    fold_coverage_records = []

    for fold_idx, (trainval_idx, eval_idx) in enumerate(skf.split(unique_rids_arr, fold_events)):
        trainval_rids = unique_rids_arr[trainval_idx]
        eval_rids     = unique_rids_arr[eval_idx]

        trainval_events = surv_mci_labels.loc[trainval_rids, 'event']
        train_rids, cal_rids = _kf_tts(
            trainval_rids, test_size=0.20, random_state=RANDOM_SEED + fold_idx,
            stratify=trainval_events)

        train_fold = landmark_stack[landmark_stack['RID'].isin(train_rids)].copy()
        cal_fold   = landmark_stack[landmark_stack['RID'].isin(cal_rids)].copy()
        eval_fold  = landmark_stack[landmark_stack['RID'].isin(eval_rids)].copy()

        for col in core_features:
            med = train_fold[col].median()
            train_fold[col] = train_fold[col].fillna(med)
            cal_fold[col]   = cal_fold[col].fillna(med)
            eval_fold[col]  = eval_fold[col].fillna(med)

        model = CoxPHFitter(penalizer=0.1)
        try:
            model.fit(train_fold[core_features + ['duration_resid', 'event', 'landmark_time', 'RID']],
                       duration_col='duration_resid', event_col='event', cluster_col='RID')
        except TypeError:
            model.fit(train_fold[core_features + ['duration_resid', 'event', 'landmark_time']],
                       duration_col='duration_resid', event_col='event')

        for tau in LANDMARK_TIMES:
            cal_tau  = cal_fold[cal_fold['landmark_time'] == tau]
            eval_tau = eval_fold[eval_fold['landmark_time'] == tau]
            if len(cal_tau) < 5 or len(eval_tau) < 5:
                continue

            obs_cal = cal_tau['event'].values == 1
            if obs_cal.sum() < 3:
                continue

            t_hat_cal  = _invert_surv(model.predict_survival_function(cal_tau[core_features + ['landmark_time']]))
            t_hat_eval = _invert_surv(model.predict_survival_function(eval_tau[core_features + ['landmark_time']]))
            dur_cal  = cal_tau['duration_resid'].values
            dur_eval = eval_tau['duration_resid'].values

            scores = t_hat_cal[obs_cal] - dur_cal[obs_cal]
            q_hat = _conformal_quantile(scores, 0.20, obs_cal.sum())

            lo = np.clip(t_hat_eval - q_hat, 0, None)
            hi = t_hat_eval + q_hat
            obs_eval = eval_tau['event'].values == 1
            if obs_eval.sum() == 0:
                continue

            covered = ((dur_eval[obs_eval] >= lo[obs_eval]) & (dur_eval[obs_eval] <= hi[obs_eval])).sum()
            width = hi[obs_eval] - lo[obs_eval]
            valid = width > 0
            pct = np.clip((dur_eval[obs_eval][valid] - lo[obs_eval][valid]) / width[valid], 0, 1)

            pooled_pct_by_tau[tau].extend(pct.tolist())
            fold_coverage_records.append(dict(fold=fold_idx, tau=tau,
                                               n_eval_events=obs_eval.sum(),
                                               coverage=covered / obs_eval.sum()))
        print(f'Fold {fold_idx + 1}/{N_FOLDS} done.')

    fold_df = pd.DataFrame(fold_coverage_records)
    print('\nPooled eval events per landmark (sum across all 5 folds):')
    print(fold_df.groupby('tau')['n_eval_events'].sum())

    # ── Bootstrap CI per landmark ──────────────────────────────────────────────
    bootstrap_results = {}
    for tau in LANDMARK_TIMES:
        vals = pooled_pct_by_tau[tau]
        if len(vals) < 5:
            print(f'tau={tau}: insufficient pooled events ({len(vals)}), skipping.')
            continue
        mean_pi, ci_lo, ci_hi = bootstrap_ci(vals)
        bootstrap_results[tau] = dict(n=len(vals), mean_pi=mean_pi, ci_lo=ci_lo, ci_hi=ci_hi)
        print(f'tau={tau:.1f}yr | n_pooled_events={len(vals)} | mean_pi={mean_pi:.3f} '
              f'| 95% CI=[{ci_lo:.3f}, {ci_hi:.3f}]')

    # ── Plot with error bars ────────────────────────────────────────────────────
    if bootstrap_results:
        taus_sorted = sorted(bootstrap_results.keys())
        means = [bootstrap_results[t]['mean_pi'] for t in taus_sorted]
        err_lo = [bootstrap_results[t]['mean_pi'] - bootstrap_results[t]['ci_lo'] for t in taus_sorted]
        err_hi = [bootstrap_results[t]['ci_hi'] - bootstrap_results[t]['mean_pi'] for t in taus_sorted]

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.errorbar(taus_sorted, means, yerr=[err_lo, err_hi], fmt='o-', color='#2980b9',
                    lw=2, markersize=8, capsize=5,
                    label=f'{N_FOLDS}-fold pooled mean π (95% bootstrap CI)')
        ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
        ax.axhline(0.259, color='#95a5a6', ls=':', lw=1.5,
                   label='Baseline-only Cox (Section 9.5, α=0.20)')
        ax.set(xlabel='Landmark time τ (years since baseline)',
               ylabel='Mean within-interval percentile (π)',
               title=f'{N_FOLDS}-Fold Pooled Landmark Bias, with Bootstrap 95% CIs\n(Cox PH, MCI → AD, α=0.20)')
        ax.set_ylim(0, 0.6)
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'landmark_kfold_bootstrap.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('No landmark had enough pooled data for a bootstrap CI.')
else:
    print('RUN_KFOLD_BOOTSTRAP is False. Set to True to run this (refits Cox 5x).')


### 12.6 Landmark Table: Mean pi by Landmark Time


In [ ]:
# ── 12.6  Landmark table: pooled mean pi by landmark time with bootstrap 95% CIs ─
# Uses bootstrap_results from 12.5. Prints LaTeX rows for the landmark table (Section 5.4).
lm = pd.DataFrame([dict(tau=t, n_events=int(v['n']), mean_pi=float(v['mean_pi']),
                        ci_lo=float(v['ci_lo']), ci_hi=float(v['ci_hi']))
                   for t, v in sorted(bootstrap_results.items())])
lm.to_csv(OUT_DIR / 'landmark_pi_by_tau.csv', index=False)
print(lm.round(3).to_string(index=False))
print('\n=== Landmark table rows (paste into LaTeX) ===')
for r in lm.itertuples():
    print(f'{r.tau:.1f} & {r.n_events} & {r.mean_pi:.3f} & [{r.ci_lo:.3f}, {r.ci_hi:.3f}] \\\\')


In [ ]:
# ── Landmark Model: Fixed-Width CP vs Adaptive CSD (same style as Section 10.6) ─
# Generates the same two-panel figure style used for Cox/AFT/GBSA/DeepSurv in
# Section 10.6, but for the landmark Cox supermodel from Section 12. Reuses
# plot_intervals() from 10.6 directly rather than reimplementing it, so the
# visual style (markers, legend, coverage text) matches exactly.
#
# Uses landmark time tau=0 for the comparison — the baseline landmark, which
# is the fairest apples-to-apples point against the other four models' own
# baseline-only predictions. Change LANDMARK_TAU_FOR_PLOT to compare at a
# different landmark time instead.

LANDMARK_TAU_FOR_PLOT = 0.0

cal_tau0  = lm_cal[lm_cal['landmark_time'] == LANDMARK_TAU_FOR_PLOT]
eval_tau0 = lm_eval[lm_eval['landmark_time'] == LANDMARK_TAU_FOR_PLOT]

surv_lm_cal  = cox_lm.predict_survival_function(cal_tau0[core_features + ['landmark_time']])
surv_lm_eval = cox_lm.predict_survival_function(eval_tau0[core_features + ['landmark_time']])

dur_lm_cal  = cal_tau0['duration_resid'].values
ev_lm_cal   = cal_tau0['event'].values
dur_lm_eval = eval_tau0['duration_resid'].values
ev_lm_eval  = eval_tau0['event'].values

print(f'Landmark (tau={LANDMARK_TAU_FOR_PLOT}) cal  : n={len(dur_lm_cal)}, '
      f'events={int((ev_lm_cal==1).sum())}')
print(f'Landmark (tau={LANDMARK_TAU_FOR_PLOT}) eval : n={len(dur_lm_eval)}, '
      f'events={int((ev_lm_eval==1).sum())}')

# ── Fixed-width CP (same procedure as Section 9.3/12.3) ──────────────────────
t_hat_cal_lm  = _invert_surv(surv_lm_cal)
t_hat_eval_lm = _invert_surv(surv_lm_eval)

obs_cal_lm = ev_lm_cal == 1
scores_lm  = t_hat_cal_lm[obs_cal_lm] - dur_lm_cal[obs_cal_lm]
q_hat_lm   = _conformal_quantile(scores_lm, 0.20, obs_cal_lm.sum())

lo_fw_lm = np.clip(t_hat_eval_lm - q_hat_lm, 0, None)
hi_fw_lm = t_hat_eval_lm + q_hat_lm

# ── CSD adaptive intervals (same procedure as Section 10.1-10.3) ────────────
pcts_cal_lm   = compute_pcts(surv_lm_cal, percentile_levels)
pcts_eval_lm  = compute_pcts(surv_lm_eval, percentile_levels)
q_hats_csd_lm = csd_calibrate(pcts_cal_lm, dur_lm_cal, ev_lm_cal, percentile_levels)
pcts_adj_lm   = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(pcts_eval_lm, q_hats_csd_lm), 0, None))

lo_csd_lm  = pcts_adj_lm[:, -1]   # rho=0.9 -> 10th pct of event time (early)
hi_csd_lm  = pcts_adj_lm[:, 0]    # rho=0.1 -> 90th pct of event time (late)
mid_csd_lm = pcts_adj_lm[:, 4]    # rho=0.5 -> predicted median

# ── Plot — reuses plot_intervals() from Section 10.6 unchanged ──────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_intervals(axes[0], lo_fw_lm, hi_fw_lm, (lo_fw_lm + hi_fw_lm) / 2, dur_lm_eval, ev_lm_eval,
               f'Landmark Cox (τ={LANDMARK_TAU_FOR_PLOT:.1f}): Fixed Width (80% target)')
plot_intervals(axes[1], lo_csd_lm, hi_csd_lm, mid_csd_lm, dur_lm_eval, ev_lm_eval,
               f'Landmark Cox (τ={LANDMARK_TAU_FOR_PLOT:.1f}): Adaptive CSD (80% central)')

axes[0].set_ylim(0, 60)   # matches the y-cap used on the other four models' grid
for ax in axes:
    ax.set_xlabel('Subjects (sorted)')
axes[0].set_ylabel('Years from baseline')

plt.tight_layout()
plt.savefig(FIG_DIR / 'landmark_fixed_vs_csd.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. IPCWWeighted Calibration (Testing the Censoring Hypothesis Directly)

Sections 9.5 and 12 established that the late prediction bias (mean π well
below 0.50) is consistent across all four models and does not improve with
longitudinal covariates, ruling out model choice and baseline-only-covariates
as explanations. Censoring remains the leading hypothesis, but has not yet
been tested directly.

**Mechanism being tested:** observed-events-only calibration draws its
calibration sample disproportionately from *earlier* converters, since later
converters are more likely to still be censored at any given follow-up cutoff.
IPCW (inverse probability of censoring weighting) corrects this by upweighting
observed-event subjects whose event time made them *unlikely* to have been
observed (such as subjects with later event times, who represent an
underrepresented part of the true converter population).

**Version 1 (this section):** marginal censoring curve, one KM fit, no
covariates. Appropriate if censoring is non-informative (doesn't depend on
patient characteristics). 13.1 includes a diagnostic check for this
assumption before proceeding.

**Scope:** Cox PH baseline (Section 9.3) only, as a first test, not the
landmark model, to isolate the censoring-weighting effect from the
longitudinal-covariate question already answered in Section 12.

In [ ]:
# ── 13.1  Marginal Censoring Curve + Informativeness Diagnostic ─────────────
from lifelines import KaplanMeierFitter, CoxPHFitter as _CoxPHFitter_diag

# ── Marginal censoring survival curve G(t) = P(remain uncensored past t) ────
# Flip the event indicator: "event" here means CENSORED, not converted.
kmf_censoring = KaplanMeierFitter()
_cens_train = surv_mci_labels.loc[surv_mci_labels.index.isin(MCI_SPLIT['train'])]  # training subjects only
print(f'Censoring curve fit on {len(_cens_train)} training subjects')
kmf_censoring.fit(_cens_train['duration'],
                   event_observed=(1 - _cens_train['event']),
                   label='Censoring survival function G(t)')

fig, ax = plt.subplots(figsize=(8, 5))
kmf_censoring.plot_survival_function(ax=ax, color='#8e44ad', ci_show=True, ci_alpha=0.15)
ax.set(xlabel='Years from baseline', ylabel='P(remain uncensored past t)',
       title='Marginal Censoring Survival Function: G(t)\n(MCI → AD cohort)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'ipcw_censoring_curve.png', dpi=150, bbox_inches='tight')
plt.show()

def G_hat(t, kmf=kmf_censoring):
    """Evaluate the censoring survival function at time(s) t."""
    t = np.atleast_1d(t)
    return np.array([
        kmf.survival_function_at_times(tt).values[0] if tt <= kmf.timeline.max()
        else kmf.survival_function_.iloc[-1, 0]
        for tt in t
    ])

# ── Informativeness diagnostic: is censoring predictable from covariates? ───
# Fits Cox with CENSORING as the event, against the same core_features used
# everywhere else. If nothing comes out significant, the marginal (Version 1)
# curve above is defensible. If several features are significant, censoring is
# likely informative and Version 2 (covariate-dependent G(t|x)) is needed.
baseline_snap = build_landmark_snapshot(df, surv_mci_labels, 0.0)
diag_df = baseline_snap[core_features].copy()
for col in core_features:
    diag_df[col] = diag_df[col].fillna(diag_df[col].median())
diag_df['duration'] = surv_mci_labels.loc[baseline_snap.index, 'duration']
diag_df['censored'] = 1 - surv_mci_labels.loc[baseline_snap.index, 'event']

cox_diag = _CoxPHFitter_diag(penalizer=0.1)
cox_diag.fit(diag_df, duration_col='duration', event_col='censored')
diag_summary = cox_diag.summary[['coef', 'exp(coef)', 'p']].sort_values('p')
n_significant_uncorrected = (diag_summary['p'] < 0.05).sum()

# ── Multiple-testing correction (Benjamini-Hochberg) ──────────────────────────
# 35 uncorrected tests at alpha=0.05 would produce ~1.75 false positives by
# chance alone, so a small number of "significant" hits (like 3/35) is close
# to what pure noise would produce. BH-FDR correction settles this properly
# rather than relying on an arbitrary uncorrected count.
from statsmodels.stats.multitest import multipletests
bh_reject, bh_pvals_corrected, _, _ = multipletests(diag_summary['p'].values, alpha=0.05, method='fdr_bh')
diag_summary['p_bh_corrected'] = bh_pvals_corrected
diag_summary['significant_bh'] = bh_reject
n_significant_bh = bh_reject.sum()

print(f'Censoring-informativeness diagnostic: {n_significant_uncorrected} / {len(diag_summary)} '
      f'features significant at uncorrected p<0.05.')
print(f'After Benjamini-Hochberg FDR correction: {n_significant_bh} / {len(diag_summary)} remain significant.')
print(diag_summary.head(10).to_string())
if n_significant_bh == 0:
    print('\n→ No features survive multiple-testing correction, no real evidence of informative '
          'censoring. Marginal G(t) (Version 1) is well justified, not just a convenient default.')
else:
    print(f'\n→ {n_significant_bh} feature(s) survive BH correction, some real evidence of informative '
          'censoring; Version 2 (covariate-dependent G(t|x)) would be more appropriate for these features.')


In [ ]:
# ── 13.2  IPCW Weighted Conformal Calibration ────────────────────────────────
def weighted_quantile(values, weights, quantile):
    """Weighted empirical quantile. NOTE: this is a straightforward weighted
    percentile, not the finite-sample weighted-exchangeability correction from
    Tibshirani et al. (2019) adequate for a first mechanism test, but a
    refinement worth adding if this direction proves out."""
    values = np.asarray(values)
    weights = np.asarray(weights)
    sorter = np.argsort(values)
    values, weights = values[sorter], weights[sorter]
    cum_weights = np.cumsum(weights) - 0.5 * weights
    cum_weights /= np.sum(weights)
    return np.interp(quantile, cum_weights, values)

# Rebuild Cox's cal/eval predictions FRESH here, rather than relying on Section
# 9.3's derived globals (t_hat_eval, dur_eval_cox, ev_eval_cox, etc.) 
_, surv_cox_cal_fresh  = calc_cox_ph_c(cox_model_mci, cox_scaler_mci, X_mci_cal_cox,
                                        y_ev_mci_cal,  y_dur_mci_cal)
_, surv_cox_eval_fresh = calc_cox_ph_c(cox_model_mci, cox_scaler_mci, X_mci_eval_cox,
                                        y_ev_mci_eval, y_dur_mci_eval)

t_hat_cal_fresh  = _invert_surv(surv_cox_cal_fresh)
t_hat_eval_fresh = _invert_surv(surv_cox_eval_fresh)

dur_cal_fresh  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_fresh   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_fresh = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_fresh  = np.array(y_ev_mci_eval,  dtype=np.int32)

obs_cal_fresh    = ev_cal_fresh == 1
scores_cal_fresh = t_hat_cal_fresh[obs_cal_fresh] - dur_cal_fresh[obs_cal_fresh]
n_cal_fresh      = obs_cal_fresh.sum()

print(f'Rebuilt Cox cal/eval: cal n={len(dur_cal_fresh)} (events={obs_cal_fresh.sum()}), '
      f'eval n={len(dur_eval_fresh)} (events={ev_eval_fresh.sum()})')

G_at_events = G_hat(dur_cal_fresh[obs_cal_fresh])
G_at_events = np.clip(G_at_events, 0.05, 1.0)  # avoid exploding weights near end of follow-up
ipcw_weights = 1.0 / G_at_events

print(f'IPCW weight range: min={ipcw_weights.min():.2f}, max={ipcw_weights.max():.2f}, '
      f'mean={ipcw_weights.mean():.2f}')
print('(Weight > 1 means that subject is upweighted to compensate for '
      'under-representation of later-converting subjects in the observed-events sample.)')

results_ipcw_cox = {}
for alpha in [0.10, 0.20]:
    q_hat_unweighted = _conformal_quantile(scores_cal_fresh, alpha, n_cal_fresh)
    q_hat_ipcw = weighted_quantile(scores_cal_fresh, ipcw_weights, 1 - alpha)

    for label, q_hat in [('unweighted', q_hat_unweighted), ('IPCW', q_hat_ipcw)]:
        lo = np.clip(t_hat_eval_fresh - q_hat, 0, None)
        hi = t_hat_eval_fresh + q_hat
        obs = ev_eval_fresh == 1
        n_obs = obs.sum()
        covered = ((dur_eval_fresh[obs] >= lo[obs]) & (dur_eval_fresh[obs] <= hi[obs])).sum()
        coverage = covered / n_obs

        width = hi[obs] - lo[obs]
        valid = width > 0
        pct = np.clip((dur_eval_fresh[obs][valid] - lo[obs][valid]) / width[valid], 0, 1)

        results_ipcw_cox[(alpha, label)] = dict(q_hat=q_hat, coverage=coverage,
                                                  mean_pi=pct.mean(), n_obs=n_obs)
        print(f'alpha={alpha:.2f} [{label:>11}] q_hat={q_hat:.4f}yr coverage={coverage:.1%} '
              f'mean_pi={pct.mean():.3f}')
    print()

### 13.3 Comparison: Does IPCW Weighting Reduce the Late-Prediction Bias?

In [ ]:
# ── 13.3  IPCW vs Unweighted: Visual Comparison ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_idx, alpha in enumerate([0.10, 0.20]):
    ax = axes[ax_idx]
    labels = ['Unweighted\n(Section 9.5)', 'IPCW-weighted']
    means = [results_ipcw_cox[(alpha, 'unweighted')]['mean_pi'],
             results_ipcw_cox[(alpha, 'IPCW')]['mean_pi']]
    colors = ['#95a5a6', '#8e44ad']
    ax.bar(labels, means, color=colors)
    ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
    ax.set(ylabel='Mean within-interval percentile (π)',
           title=f'Cox PH: alpha={alpha} (target {(1-alpha)*100:.0f}%)')
    ax.set_ylim(0, 0.6)
    ax.legend()
plt.suptitle('Does IPCW-Weighted Calibration Reduce the Late-Prediction Bias? (MCI → AD)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'ipcw_vs_unweighted_bias.png', dpi=150, bbox_inches='tight')
plt.show()

print('Summary:')
for alpha in [0.10, 0.20]:
    unw = results_ipcw_cox[(alpha, 'unweighted')]['mean_pi']
    ipcw = results_ipcw_cox[(alpha, 'IPCW')]['mean_pi']
    shift = ipcw - unw
    direction = 'toward 0.50 (improvement)' if abs(ipcw - 0.5) < abs(unw - 0.5) else 'away from 0.50 (worse)'
    print(f'alpha={alpha:.2f}: unweighted={unw:.3f} -> IPCW={ipcw:.3f}  (shift={shift:+.3f}, {direction})')


In [ ]:
# ── 13.4  CSD Within Interval Percentile: Interval-Shape Check ─────────────
# Rebuilt fresh from surv_cox_mci / surv_cox_mci_eval
pcts_cal_check  = compute_pcts(surv_cox_mci,      percentile_levels)
pcts_eval_check = compute_pcts(surv_cox_mci_eval, percentile_levels)

dur_cal_check  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_check   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_check = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_check  = np.array(y_ev_mci_eval,  dtype=np.int32)

q_hats_check   = csd_calibrate(pcts_cal_check, dur_cal_check, ev_cal_check, percentile_levels)
pcts_adj_check = csd_adjust_pcts(pcts_eval_check, q_hats_check)
pcts_adj_check = np.clip(pcts_adj_check, 0, None)
pcts_adj_check = enforce_monotonic_pcts(pcts_adj_check)

# 80% central interval: lo = rho=0.9 col (10th pct of event time, index -1),
# hi = rho=0.1 col (90th pct of event time, index 0) same convention as 10.6.
lo_csd = pcts_adj_check[:, -1]
hi_csd = pcts_adj_check[:, 0]

obs_eval_check = ev_eval_check == 1
width_csd = hi_csd[obs_eval_check] - lo_csd[obs_eval_check]
valid_csd = width_csd > 0
pct_csd = np.clip(
    (dur_eval_check[obs_eval_check][valid_csd] - lo_csd[obs_eval_check][valid_csd])
    / width_csd[valid_csd], 0, 1)

mean_pi_fixed_unweighted = results_ipcw_cox[(0.20, 'unweighted')]['mean_pi']
mean_pi_fixed_ipcw       = results_ipcw_cox[(0.20, 'IPCW')]['mean_pi']
mean_pi_csd              = pct_csd.mean()

print('CSD adaptive interval (Cox PH) within-interval percentile:')
print(f'  n_obs={valid_csd.sum()}  mean_pi={mean_pi_csd:.3f}  median_pi={np.median(pct_csd):.3f}')
print()
print('Comparison across interval constructions (Cox PH, alpha=0.20, MCI -> AD):')
print(f'  Fixed-width, symmetric,  unweighted (9.5) : mean_pi={mean_pi_fixed_unweighted:.3f}')
print(f'  Fixed-width, symmetric,  IPCW-weighted     : mean_pi={mean_pi_fixed_ipcw:.3f}')
print(f'  Adaptive (CSD), percentile-based           : mean_pi={mean_pi_csd:.3f}')

# ── Bar chart comparison ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
labels = ['Fixed-width\n(symmetric,\nunweighted)', 'Fixed-width\n(symmetric,\nIPCW)',
          'CSD Adaptive\n(percentile-based)']
means = [mean_pi_fixed_unweighted, mean_pi_fixed_ipcw, mean_pi_csd]
colors = ['#95a5a6', '#8e44ad', '#2980b9']
ax.bar(labels, means, color=colors)
ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
ax.set(ylabel='Mean within-interval percentile (π)',
       title='Does Interval Construction Explain the Late-Prediction Bias?\n(Cox PH, MCI → AD, α=0.20)')
ax.set_ylim(0, 0.6)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'interval_shape_bias_check.png', dpi=150, bbox_inches='tight')
plt.show()

if abs(mean_pi_csd - 0.5) < abs(mean_pi_fixed_unweighted - 0.5) - 0.05:
    print('\n→ CSD meaningfully closer to 0.50- some support for interval-shape/skewness '
          'as a contributing factor.')
elif abs(mean_pi_csd - 0.5) > abs(mean_pi_fixed_unweighted - 0.5) + 0.05:
    print('\n→ CSD is NOT closer to 0.50 (possibly further)- argues against interval shape '
          'as the explanation; bias likely lies elsewhere in the calibration mechanism.')
else:
    print('\n→ CSD is roughly comparable to the fixed-width result- interval shape does not '
          'appear to meaningfully explain the bias.')

### 13.5 Interval-Shape Check on All 4 Models

Extends 13.4's fixed width vs. CSD comparison to GBSA, AFT, and DeepSurv, to
check whether the interval shape effect found for Cox PH generalizes. All
survival matrices are rebuilt fresh here

In [ ]:
# ── 13.5  Interval-Shape Check: All 4 Models ────────────────────────────────
def compute_both_mean_pi(surv_cal, surv_eval, dur_cal, ev_cal, dur_eval, ev_eval, alpha=0.20):
    """Returns fixed-width (symmetric) and CSD (percentile-based) mean within-
    interval percentile for one model, given fresh cal/eval survival matrices."""
    t_hat_cal  = _invert_surv(surv_cal)
    t_hat_eval = _invert_surv(surv_eval)
    obs_cal = ev_cal == 1
    scores = t_hat_cal[obs_cal] - dur_cal[obs_cal]
    q_hat = _conformal_quantile(scores, alpha, obs_cal.sum())
    lo_fw = np.clip(t_hat_eval - q_hat, 0, None)
    hi_fw = t_hat_eval + q_hat
    obs_eval = ev_eval == 1
    width_fw = hi_fw[obs_eval] - lo_fw[obs_eval]
    valid_fw = width_fw > 0
    pct_fw = np.clip((dur_eval[obs_eval][valid_fw] - lo_fw[obs_eval][valid_fw])
                      / width_fw[valid_fw], 0, 1)

    pcts_cal  = compute_pcts(surv_cal, percentile_levels)
    pcts_eval = compute_pcts(surv_eval, percentile_levels)
    q_hats_csd = csd_calibrate(pcts_cal, dur_cal, ev_cal, percentile_levels)
    pcts_adj = csd_adjust_pcts(pcts_eval, q_hats_csd)
    pcts_adj = np.clip(pcts_adj, 0, None)
    pcts_adj = enforce_monotonic_pcts(pcts_adj)
    lo_csd, hi_csd = pcts_adj[:, -1], pcts_adj[:, 0]
    width_csd = hi_csd[obs_eval] - lo_csd[obs_eval]
    valid_csd = width_csd > 0
    pct_csd = np.clip((dur_eval[obs_eval][valid_csd] - lo_csd[obs_eval][valid_csd])
                       / width_csd[valid_csd], 0, 1)

    return dict(fixed_mean_pi=pct_fw.mean(), fixed_n=int(valid_fw.sum()),
                csd_mean_pi=pct_csd.mean(), csd_n=int(valid_csd.sum()))

dur_cal_all  = np.array(y_dur_mci_cal,  dtype=np.float64)
ev_cal_all   = np.array(y_ev_mci_cal,   dtype=np.int32)
dur_eval_all = np.array(y_dur_mci_eval, dtype=np.float64)
ev_eval_all  = np.array(y_ev_mci_eval,  dtype=np.int32)

interval_shape_results = {}

# Cox: reuse already computed 13.2/13.4 results (no need to recompute)
interval_shape_results['Cox PH'] = dict(
    fixed_mean_pi=results_ipcw_cox[(0.20, 'unweighted')]['mean_pi'], fixed_n=39,
    csd_mean_pi=mean_pi_csd, csd_n=int(valid_csd.sum()))

# GBSA:rebuild fresh
_, surv_gbsa_cal_fresh  = calc_gbsa_c(gbsa_model_mci, X_mci_cal,  y_ev_mci_cal,  y_dur_mci_cal)
_, surv_gbsa_eval_fresh = calc_gbsa_c(gbsa_model_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)
interval_shape_results['GBSA'] = compute_both_mean_pi(
    surv_gbsa_cal_fresh, surv_gbsa_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all)

# AFT: direct predict_survival_function (no scaler/inversion helper needed)
aft_surv_cal_fresh  = aft_model_mci.predict_survival_function(X_mci_cal)
aft_surv_eval_fresh = aft_model_mci.predict_survival_function(X_mci_eval)
interval_shape_results['AFT'] = compute_both_mean_pi(
    aft_surv_cal_fresh, aft_surv_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all)

# DeepSurv: rebuild fresh
_, surv_ds_cal_fresh  = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_cal,  y_ev_mci_cal,  y_dur_mci_cal)
_, surv_ds_eval_fresh = calc_deepsurv_c(ds_model_mci, ds_scaler_mci, X_mci_eval, y_ev_mci_eval, y_dur_mci_eval)
interval_shape_results['DeepSurv'] = compute_both_mean_pi(
    surv_ds_cal_fresh, surv_ds_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all)

print('Interval-shape check: all 4 models (alpha=0.20, MCI -> AD):')
print(f'{"Model":<10} {"Fixed mean_pi":<15} {"CSD mean_pi":<15} {"Shift toward 0.50"}')
print('-' * 60)
for model, r in interval_shape_results.items():
    shift = abs(r['csd_mean_pi'] - 0.5) < abs(r['fixed_mean_pi'] - 0.5)
    print(f'{model:<10} {r["fixed_mean_pi"]:<15.3f} {r["csd_mean_pi"]:<15.3f} '
          f'{"YES" if shift else "no"}')

fig, ax = plt.subplots(figsize=(11, 6))
models = list(interval_shape_results.keys())
x = np.arange(len(models))
width = 0.35
fixed_vals = [interval_shape_results[m]['fixed_mean_pi'] for m in models]
csd_vals   = [interval_shape_results[m]['csd_mean_pi'] for m in models]
ax.bar(x - width/2, fixed_vals, width, label='Fixed-width (symmetric)', color='#95a5a6')
ax.bar(x + width/2, csd_vals,   width, label='CSD Adaptive (percentile-based)', color='#2980b9')
ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
ax.set(xticks=x, xticklabels=models, ylabel='Mean within-interval percentile (π)',
       title='Interval Shape vs. Late-Prediction Bias: All 4 Models (α=0.20)')
ax.set_ylim(0, 0.6)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'interval_shape_bias_all4.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 13.5 (extended): Interval-Shape Check: All 3 Methods, All 4 Models ─────
# Builds on Section 13.5's two-method result (fixed-width vs. CSD) by adding
# the IPCW-weighted fixed-width method for all four models. Section 13.2 only
# ever computed IPCW for Cox PH; this generalizes that same procedure
# (G_hat is cohort-wide, not model-specific, so it transfers directly) to
# GBSA, AFT, and DeepSurv using the "fresh" cal/eval survival matrices
# Section 13.5 already built for each model.

def compute_ipcw_mean_pi(surv_cal, surv_eval, dur_cal, ev_cal, dur_eval, ev_eval, alpha=0.20):
    """Mirrors Section 13.2's IPCW procedure for Cox, generalized to any model's
    cal/eval survival matrices. Returns mean within-interval percentile under
    IPCW-weighted fixed-width calibration."""
    t_hat_cal = _invert_surv(surv_cal)
    t_hat_eval = _invert_surv(surv_eval)

    obs_cal = np.asarray(ev_cal) == 1
    dur_cal_arr = np.asarray(dur_cal, dtype=np.float64)
    scores = t_hat_cal[obs_cal] - dur_cal_arr[obs_cal]

    G_at_events = np.clip(G_hat(dur_cal_arr[obs_cal]), 0.05, 1.0)
    weights = 1.0 / G_at_events
    q_hat_ipcw = weighted_quantile(scores, weights, 1 - alpha)

    lo = np.clip(t_hat_eval - q_hat_ipcw, 0, None)
    hi = t_hat_eval + q_hat_ipcw

    dur_eval_arr = np.asarray(dur_eval, dtype=np.float64)
    obs_eval = np.asarray(ev_eval) == 1
    width = hi[obs_eval] - lo[obs_eval]
    valid = width > 0
    pct = np.clip((dur_eval_arr[obs_eval][valid] - lo[obs_eval][valid]) / width[valid], 0, 1)
    return pct.mean()

# ── Compute the missing IPCW numbers for GBSA, AFT, DeepSurv ────────────────
# Cox already has this from Section 13.2 (results_ipcw_cox). Reuse it rather
# than recomputing, so the Cox bar matches the earlier Image 1 exactly.
ipcw_mean_pi_by_model = {
    'Cox PH': results_ipcw_cox[(0.20, 'IPCW')]['mean_pi'],
    'GBSA': compute_ipcw_mean_pi(
        surv_gbsa_cal_fresh, surv_gbsa_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all),
    'AFT': compute_ipcw_mean_pi(
        aft_surv_cal_fresh, aft_surv_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all),
    'DeepSurv': compute_ipcw_mean_pi(
        surv_ds_cal_fresh, surv_ds_eval_fresh, dur_cal_all, ev_cal_all, dur_eval_all, ev_eval_all),
}

print('IPCW-weighted mean π by model (alpha=0.20):')
for m, v in ipcw_mean_pi_by_model.items():
    print(f'  {m:<10} {v:.3f}')

# ── Three-method grouped bar chart, all four models ─────────────────────────
models = list(interval_shape_results.keys())
fixed_vals = [interval_shape_results[m]['fixed_mean_pi'] for m in models]
ipcw_vals  = [ipcw_mean_pi_by_model[m] for m in models]
csd_vals   = [interval_shape_results[m]['csd_mean_pi'] for m in models]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(models))
width = 0.26

ax.bar(x - width, fixed_vals, width, label='Fixed-width\n(symmetric, unweighted)', color='#95a5a6')
ax.bar(x,          ipcw_vals,  width, label='Fixed-width\n(symmetric, IPCW)',       color='#8e44ad')
ax.bar(x + width,  csd_vals,   width, label='CSD Adaptive\n(percentile-based)',      color='#2980b9')

ax.axhline(0.50, color='#e74c3c', ls='--', lw=1.5, label='Ideal (π=0.50)')
ax.set(xticks=x, xticklabels=models, ylabel='Mean within-interval percentile (π)',
       title='Does Interval Construction Explain the Late-Prediction Bias? (α=0.20)')
ax.set_ylim(0, 0.6)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'interval_shape_three_methods_all4.png', dpi=150, bbox_inches='tight')
plt.show()

# Also append the IPCW numbers into interval_shape_results so any downstream
# cell (e.g. the 13.6 k-fold/bootstrap extension) can reuse them consistently.
for m in models:
    interval_shape_results[m]['ipcw_mean_pi'] = ipcw_mean_pi_by_model[m]

### 13.6 K-Fold Pooling + Bootstrap CIs for the Interval-Shape Effect on All 4 Models

This extends the 5-fold pooling + bootstrap CI approach (originally built for
Cox alone) to GBSA, AFT, and DeepSurv as well, using each model's own fitting
procedure (`run_cox_ph`, `gbsa_survival_cv`, `_fit_aft_with_cv`, `run_deepsurv`)
within each fold. HPO trial counts are kept low (`n_trials=10`) since this is a
robustness check, not a final model, not meant to reproduce each model's
original tuned performance exactly.

In [ ]:
# ── 13.6  K-Fold + Bootstrap: Fixed width vs CSD vs CiPOT, all 4 models ──────
# REPLACES the existing 13.6 cell. Same folds, seeds, and model fitting as before;
# the only change is that CiPOT is computed on the same fold predictions, so the
# fixed-width and CSD columns should match the old Table 3 up to run-to-run noise.
# Slow (refits 4 models x 5 folds). Results are saved to disk as soon as each
# model finishes, so a Kaggle timeout doesn't lose finished models.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_INTERVAL_SHAPE_KFOLD = True
MODEL_TYPES = ['Cox PH', 'GBSA', 'AFT', 'DeepSurv']
KFOLD_N_TRIALS = 10
KFOLD_OUT = OUT_DIR / 'kfold_three_methods'
KFOLD_OUT.mkdir(parents=True, exist_ok=True)

from preprocessing import mice_impute

# ── Self-contained prelude: defines what 13.6 borrowed from Sections 9.3, 12.1, 12.5,
#    so Sections 11, 12 and 13.1-13.5 can be skipped. Same definitions as those cells.
if 'surv_mci_labels' not in globals():
    from preprocessing import build_survival_labels
    surv_mci_labels = build_survival_labels(df, df_bl, 'MCI', 'AD', exclusion_rids=REVERSION_EXCLUSIONS)
if 'bootstrap_ci' not in globals():
    BOOTSTRAP_ITERS = 1000
    def bootstrap_ci(values, n_iter=BOOTSTRAP_ITERS, seed=RANDOM_SEED):
        values = np.array(values)
        rng = np.random.default_rng(seed)
        boot = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_iter)])
        return values.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5)
if '_conformal_quantile' not in globals():
    def _conformal_quantile(scores, alpha, n_cal):
        level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
        return float(np.quantile(scores, level))
if '_invert_surv' not in globals():
    def _invert_surv(surv_matrix, p=0.5):
        times = surv_matrix.index.values.astype(np.float64)
        medians = np.full(surv_matrix.shape[1], times[-1])
        for j in range(surv_matrix.shape[1]):
            below = np.where(surv_matrix.iloc[:, j].values <= p)[0]
            if len(below) > 0:
                medians[j] = times[below[0]]
        return medians


def _surv_at_kf(surv_matrix, times_obs):
    """Predicted S_i(T_i) for each subject at its own observed time (step function)."""
    grid = surv_matrix.index.values.astype(float)
    vals = surv_matrix.values
    idx = np.searchsorted(grid, np.asarray(times_obs, dtype=float), side='right') - 1
    out = np.ones(len(times_obs))
    has = idx >= 0
    out[has] = vals[idx[has], np.arange(len(times_obs))[has]]
    return out


def _pi(lo, hi, dur, ev):
    o = ev == 1
    w = hi[o] - lo[o]
    v = w > 0
    return np.clip((dur[o][v] - lo[o][v]) / w[v], 0, 1)


def compute_three_pi_arrays(surv_cal, surv_eval, dur_cal, ev_cal, dur_eval, ev_eval, alpha=0.20):
    """Per-subject pi for fixed width, CSD, and CiPOT at alpha = 0.20.
    Uses percentile_levels (0.1..0.9) from 10.1: interval = [rho 0.9, rho 0.1]."""
    dur_cal, dur_eval = np.asarray(dur_cal, float), np.asarray(dur_eval, float)
    ev_cal, ev_eval = np.asarray(ev_cal, int), np.asarray(ev_eval, int)

    # Fixed width (unchanged from the original 13.6)
    t_hat_cal = _invert_surv(surv_cal)
    t_hat_eval = _invert_surv(surv_eval)
    obs_cal = ev_cal == 1
    scores = t_hat_cal[obs_cal] - dur_cal[obs_cal]
    q_hat = _conformal_quantile(scores, alpha, obs_cal.sum())
    pi_fw = _pi(np.clip(t_hat_eval - q_hat, 0, None), t_hat_eval + q_hat, dur_eval, ev_eval)

    # CSD (unchanged)
    pcts_cal = compute_pcts(surv_cal, percentile_levels)
    pcts_eval = compute_pcts(surv_eval, percentile_levels)
    q_hats_csd = csd_calibrate(pcts_cal, dur_cal, ev_cal, percentile_levels)
    pcts_adj = enforce_monotonic_pcts(np.clip(csd_adjust_pcts(pcts_eval, q_hats_csd), 0, None))
    pi_csd = _pi(pcts_adj[:, -1], pcts_adj[:, 0], dur_eval, ev_eval)

    # CiPOT (simplified, observed events only): same grid, recalibrated probability levels
    ipot = _surv_at_kf(surv_cal, dur_cal)[obs_cal]
    n = len(ipot)
    p_levels = np.array([np.quantile(ipot, min(np.ceil((n + 1) * r) / n, 1.0)) for r in percentile_levels])
    pcts_cp = enforce_monotonic_pcts(compute_pcts(surv_eval, p_levels))
    pi_cp = _pi(pcts_cp[:, -1], pcts_cp[:, 0], dur_eval, ev_eval)

    return pi_fw, pi_csd, pi_cp


def fit_model_fold(model_type, X_train, ev_train, dur_train):
    if model_type == 'Cox PH':
        model, scaler = run_cox_ph(X_train, ev_train, dur_train,
                                   label=f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[2:4]
        return 'cox', model, scaler
    if model_type == 'GBSA':
        model = gbsa_survival_cv(X_train, ev_train, dur_train, feature_names=list(X_train.columns),
                                 label=f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[3]
        return 'gbsa', model, None
    if model_type == 'AFT':
        return 'aft', _fit_aft_with_cv(X_train, ev_train, dur_train, f'{model_type}Fold'), None
    if model_type == 'DeepSurv':
        model, scaler = run_deepsurv(X_train, ev_train, dur_train,
                                     f'{model_type}Fold', n_trials=KFOLD_N_TRIALS)[2:4]
        return 'deepsurv', model, scaler
    raise ValueError(model_type)


def predict_surv_fold(kind, model, scaler, X, ev, dur):
    if kind == 'cox':
        return calc_cox_ph_c(model, scaler, X, ev, dur)[1]
    if kind == 'gbsa':
        return calc_gbsa_c(model, X, ev, dur)[1]
    if kind == 'aft':
        d = X.copy()
        d['duration'] = np.array(dur, dtype=np.float64)
        d['event'] = np.array(ev, dtype=np.int32)
        return model.predict_survival_function(d)
    if kind == 'deepsurv':
        return calc_deepsurv_c(model, scaler, X, ev, dur)[1]
    raise ValueError(kind)


if RUN_INTERVAL_SHAPE_KFOLD:
    from sklearn.model_selection import StratifiedKFold, train_test_split as _is_tts

    unique_rids_full = surv_mci_labels.index.values
    fold_events_full = surv_mci_labels['event'].values
    skf_is = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    kfold_bootstrap_results = {}

    for model_type in MODEL_TYPES:
        cache = KFOLD_OUT / f'{model_type.replace(" ", "_")}.npz'
        if cache.exists():   # already finished in an earlier run of this session
            z = np.load(cache)
            pooled = {'fixed': z['fixed'], 'csd': z['csd'], 'cipot': z['cipot']}
            print(f'{model_type}: loaded from {cache}')
        else:
            feature_cols = X_mci_train_cox.columns if model_type == 'Cox PH' else X_mci_train.columns
            pooled = {'fixed': [], 'csd': [], 'cipot': []}
            for fold_idx, (trainval_idx, eval_idx) in enumerate(skf_is.split(unique_rids_full, fold_events_full)):
                trainval_rids = unique_rids_full[trainval_idx]
                eval_rids = unique_rids_full[eval_idx]
                train_rids, cal_rids = _is_tts(
                    trainval_rids, test_size=0.20, random_state=RANDOM_SEED + fold_idx,
                    stratify=surv_mci_labels.loc[trainval_rids, 'event'])

                _raw = df_bl.set_index('RID')
                if 'mice_fit_transform' in globals():   # version 2: MICE fit on the fold's training rows only
                    X_tr, (X_ca, X_ev) = mice_fit_transform(_raw.loc[train_rids, feature_cols],
                                                            [_raw.loc[cal_rids, feature_cols],
                                                             _raw.loc[eval_rids, feature_cols]])
                else:                                   # version 1 behaviour
                    X_tr = mice_impute(_raw.loc[train_rids, feature_cols])
                    X_ca = mice_impute(_raw.loc[cal_rids, feature_cols])
                    X_ev = mice_impute(_raw.loc[eval_rids, feature_cols])
                lab = lambda rids, c: surv_mci_labels.loc[rids, c].values

                kind, model, scaler = fit_model_fold(model_type, X_tr, lab(train_rids, 'event'), lab(train_rids, 'duration'))
                s_ca = predict_surv_fold(kind, model, scaler, X_ca, lab(cal_rids, 'event'), lab(cal_rids, 'duration'))
                s_ev = predict_surv_fold(kind, model, scaler, X_ev, lab(eval_rids, 'event'), lab(eval_rids, 'duration'))

                fw, csd, cp = compute_three_pi_arrays(s_ca, s_ev, lab(cal_rids, 'duration'), lab(cal_rids, 'event'),
                                                      lab(eval_rids, 'duration'), lab(eval_rids, 'event'))
                pooled['fixed'].extend(fw); pooled['csd'].extend(csd); pooled['cipot'].extend(cp)
                print(f'{model_type}: fold {fold_idx + 1}/5 done (n fixed={len(fw)}, csd={len(csd)}, cipot={len(cp)})')
            pooled = {k: np.asarray(v) for k, v in pooled.items()}
            np.savez(cache, **pooled)
            print(f'{model_type}: saved {cache}')

        res = {k: (*bootstrap_ci(list(v)), len(v)) for k, v in pooled.items()}  # (mean, lo, hi, n)
        kfold_bootstrap_results[model_type] = res
        for k in ['fixed', 'csd', 'cipot']:
            m, lo, hi, n = res[k]
            print(f'   {k:<6} mean_pi={m:.3f} 95%CI=[{lo:.3f}, {hi:.3f}] (n={n})')
        for k in ['csd', 'cipot']:
            overlap = not (res['fixed'][2] < res[k][1] or res[k][2] < res['fixed'][1])
            print(f'   fixed vs {k}: CIs overlap = {overlap}')
        print()

    # ── Save + LaTeX rows for Table 3 ────────────────────────────────────────
    tbl = pd.DataFrame({m: {k: '%.3f [%.3f, %.3f]' % kfold_bootstrap_results[m][k][:3]
                            for k in ['fixed', 'csd', 'cipot']} for m in MODEL_TYPES}).T
    tbl.to_csv(OUT_DIR / 'table3_kfold_three_methods.csv')
    with open(OUT_DIR / 'table3_kfold_three_methods.json', 'w') as f:
        json.dump({m: {k: list(map(float, v[:3])) + [int(v[3])] for k, v in r.items()}
                   for m, r in kfold_bootstrap_results.items()}, f, indent=2)
    print('Table 3 rows (single-column layout, paste into LaTeX):')
    for m in ['Cox PH', 'AFT', 'GBSA', 'DeepSurv']:
        r = kfold_bootstrap_results[m]
        cells = [r'\makecell{%.3f\\ {\scriptsize [%.3f, %.3f]}}' % r[k][:3] for k in ['fixed', 'csd', 'cipot']]
        print(f'{m} & ' + ' & '.join(cells) + r' \\')
    print('pooled observed events per model:', {m: kfold_bootstrap_results[m]['fixed'][3] for m in MODEL_TYPES})

    # ── Error-bar figure (optional, not in the paper) ────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(MODEL_TYPES))
    for off, k, col, lab in [(-0.12, 'fixed', '#95a5a6', 'Fixed width'),
                             (0.0, 'csd', '#2980b9', 'CSD'),
                             (0.12, 'cipot', '#16a085', 'CiPOT')]:
        mu = np.array([kfold_bootstrap_results[m][k][0] for m in MODEL_TYPES])
        lo = np.array([kfold_bootstrap_results[m][k][1] for m in MODEL_TYPES])
        hi = np.array([kfold_bootstrap_results[m][k][2] for m in MODEL_TYPES])
        ax.errorbar(x + off, mu, yerr=[mu - lo, hi - mu], fmt='o', color=col, capsize=5, ms=8, label=lab)
    ax.axhline(0.5, color='#e74c3c', ls='--', label='Ideal (0.50)')
    ax.set_xticks(x); ax.set_xticklabels(MODEL_TYPES); ax.set_ylim(0, 0.6)
    ax.set_ylabel('Mean within-interval percentile (π)'); ax.legend()
    ax.set_title('5-fold pooled mean π with bootstrap 95% CIs (α = 0.20)')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'interval_shape_kfold_bootstrap_three_methods.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('RUN_INTERVAL_SHAPE_KFOLD is False.')


## 14. Paper Outputs

Everything the manuscript reports, in one place: the numbers (14.1), the final figures at 300 dpi (14.2), and a zip of the outputs to download (14.3). Every block is wrapped so that one failure cannot abort the run.

In [ ]:
# ── 14.1  Paper numbers: everything the manuscript reports, in one place ──────
import json, traceback

paper_numbers = {}

def _run(title, fn):
    print('\n' + '=' * 78)
    print(title)
    print('=' * 78)
    try:
        fn()
    except Exception as e:
        print(f'[SKIPPED, {type(e).__name__}: {e}]')
        traceback.print_exc(limit=2)

def _events(ev):
    return np.asarray(ev) == 1

# 1. Cohort and split sizes ---------------------------------------------------
def _cohort():
    d = dict(n_cohort=int(len(surv_mci_labels)), n_events=int(surv_mci_labels['event'].sum()),
             event_rate_pct=round(float(surv_mci_labels['event'].mean() * 100), 1))
    parts = {'train': (X_mci_train, y_ev_mci_train),
             'cal': (X_mci_cal, y_ev_mci_cal),
             'eval': (X_mci_eval, y_ev_mci_eval)}
    for k, (X, ev) in parts.items():
        d[f'n_{k}'] = int(len(X))
        d[f'events_{k}'] = int(_events(ev).sum())
    tot = d['n_train'] + d['n_cal'] + d['n_eval']
    d['split_pct_train_cal_eval'] = [round(100 * d[f'n_{k}'] / tot, 1) for k in ('train', 'cal', 'eval')]
    d['n_features_main'] = int(X_mci_train.shape[1])
    d['n_features_cox'] = int(X_mci_train_cox.shape[1])
    print(json.dumps(d, indent=2))
    print('\n-> The paper currently says 80/10/10. Update Section 4 and 5.1 to match split_pct above.')
    paper_numbers['cohort_and_split'] = d
_run('1. Cohort and split sizes', _cohort)

# 2. Table 1 ------------------------------------------------------------------
def _table1():
    print(results_df.to_string(index=False))
    print('\n-> Confirm which C-td variant Table 1 reports (this table is IPCW Antolini with bootstrap CIs).')
    paper_numbers['table1_results_df'] = results_df.to_dict('records')
_run('2. Model comparison table (C-td)', _table1)

# 3. Fixed-width and CSD coverage and mean pi ----------------------------------
def _cp_tables():
    def cov(lo, hi, dur, ev):
        o = _events(ev)
        d = np.asarray(dur)[o]
        return float(((d >= np.asarray(lo)[o]) & (d <= np.asarray(hi)[o])).mean())
    out = {}
    for name, lo_f, hi_f, dur_f, ev_f, lo_a, hi_a, mid_a, dur_a, ev_a in model_configs:
        out[name] = dict(n_obs=int(_events(ev_f).sum()),
                         fixed_cov_a20=round(cov(lo_f, hi_f, dur_f, ev_f), 4),
                         csd_cov_a20=round(cov(lo_a, hi_a, dur_a, ev_a), 4))
    res = {'AFT (Weibull)': (results_cp_aft, ev_eval_aft, dur_eval_aft),
           'Cox PH': (results_cp_cox, ev_eval_cox, dur_eval_cox),
           'GBSA': (results_cp_gbsa, ev_eval_gbsa, dur_eval_gbsa),
           'DeepSurv': (results_cp_deepsurv, ev_eval_ds, dur_eval_ds)}
    for name, (R, ev, dur) in res.items():
        for a in (0.10, 0.20):
            r = R[a]
            o = _events(ev)
            lo, hi, t = np.asarray(r['lo'])[o], np.asarray(r['hi'])[o], np.asarray(dur)[o]
            w = hi - lo
            ok = w > 0
            pi = np.clip((t[ok] - lo[ok]) / w[ok], 0, 1)
            tag = int(round(a * 100))
            out[name][f'fixed_mean_pi_a{tag}'] = round(float(pi.mean()), 3)
            out[name][f'fixed_cov_a{tag}'] = round(float(r['coverage']), 4)
            out[name][f'q_hat_a{tag}_yr'] = round(float(r['q_hat']), 3)
    print(pd.DataFrame(out).T.to_string())
    print('\n-> Section 5.3 uses fixed_mean_pi_a10 / a20. Section 5.6 coverage paragraph uses fixed_cov_a20 and csd_cov_a20.')
    paper_numbers['cp_by_model'] = out
_run('3. Fixed-width intervals and CSD coverage, mean pi', _cp_tables)

# 4. Interval-shape table (Table 2) --------------------------------------------
def _shape():
    d = {}
    for m, v in interval_shape_results.items():
        d[m] = {k: round(float(v[k]), 3) for k in ('fixed_mean_pi', 'csd_mean_pi', 'ipcw_mean_pi') if k in v}
    print(pd.DataFrame(d).T.to_string())
    paper_numbers['table2_interval_shape'] = d
_run('4. Table 2: mean pi by interval construction (alpha = 0.20)', _shape)

# 5. IPCW -----------------------------------------------------------------------
def _ipcw():
    d = dict(weight_min=round(float(ipcw_weights.min()), 2),
             weight_max=round(float(ipcw_weights.max()), 2),
             weight_mean=round(float(ipcw_weights.mean()), 2),
             cox_mean_pi_unweighted_a20=round(float(results_ipcw_cox[(0.20, 'unweighted')]['mean_pi']), 3),
             cox_mean_pi_ipcw_a20=round(float(results_ipcw_cox[(0.20, 'IPCW')]['mean_pi']), 3))
    print(json.dumps(d, indent=2))
    paper_numbers['ipcw'] = d
_run('5. IPCW weights and Cox PH effect', _ipcw)

# 6. Feature importance ---------------------------------------------------------
def _fi():
    print('Family share of SHAP mass (%):')
    print(family_pct.round(1).to_string())
    print('\nSpearman rank correlation of SHAP feature rankings:')
    print(corr_matrix.round(3).to_string())
    key = ['AV45', 'CDRSB', 'LDELTOTAL', 'Fusiform_ICV', 'mPACCtrailsB', 'APOE4']
    print('\nRanks of features the text names (1 = most important):')
    print(shap_rank.reindex(key).to_string())
    paper_numbers['family_pct'] = family_pct.round(1).to_dict()
    paper_numbers['shap_spearman'] = corr_matrix.round(3).to_dict()
    paper_numbers['shap_ranks_named'] = shap_rank.reindex(key).to_dict()
_run('6. Feature importance', _fi)

# 7. Cross-model time consistency ----------------------------------------------
def _consistency():
    print(pair_df.round(3).to_string(index=False))
    print(f"\nSpearman range: {pair_df['spearman_rho'].min():.3f} to {pair_df['spearman_rho'].max():.3f}")
    print(f"Mean abs diff range (yr): {pair_df['mean_abs_diff_yr'].min():.2f} to {pair_df['mean_abs_diff_yr'].max():.2f}")
    paper_numbers['time_consistency'] = pair_df.round(3).to_dict('records')
_run('7. Cross-model predicted-time consistency', _consistency)

# 8. Landmark ---------------------------------------------------------------------
def _landmark():
    d = {}
    if 'lm_results_by_landmark' in globals():
        d['mean_pi_by_tau'] = {str(t): round(float(v['mean_pct']), 3) for t, v in sorted(lm_results_by_landmark.items())}
    if 'visit_sensitivity_df' in globals():
        print(visit_sensitivity_df.round(3).to_string(index=False))
        d['visit_sensitivity'] = visit_sensitivity_df.round(3).to_dict('records')
    if 'bootstrap_results' in globals():
        d['kfold_bootstrap_by_tau'] = {str(t): dict(n=int(v['n']), mean_pi=round(float(v['mean_pi']), 3),
                                                    ci=[round(float(v['ci_lo']), 3), round(float(v['ci_hi']), 3)])
                                       for t, v in sorted(bootstrap_results.items())}
    print(json.dumps(d, indent=2))
    paper_numbers['landmark'] = d
_run('8. Landmark analysis', _landmark)

# 9. Table 3: pooled k-fold validation -----------------------------------------
def _val():
    d = {}
    for m, v in kfold_bootstrap_results.items():
        d[m] = dict(fixed=[round(float(x), 3) for x in v['fixed'][:3]] + [int(v['fixed'][3])],
                    csd=[round(float(x), 3) for x in v['csd'][:3]] + [int(v['csd'][3])])
    for m, v in d.items():
        print(f"{m:<9} fixed {v['fixed'][0]:.3f} [{v['fixed'][1]:.3f}, {v['fixed'][2]:.3f}]   "
              f"CSD {v['csd'][0]:.3f} [{v['csd'][1]:.3f}, {v['csd'][2]:.3f}]   n={v['fixed'][3]}")
    paper_numbers['table3_pooled_validation'] = d
_run('9. Table 3: 5-fold pooled mean pi with bootstrap CIs', _val)

# 10. Cross-model bootstrap (only if it was run in this commit) ------------------
def _boot():
    p = OUT_DIR / 'model_comparison_pairwise_ci.csv'
    if p.exists():
        t = pd.read_csv(p)
        print(t.round(4).to_string(index=False))
        paper_numbers['model_bootstrap_pairs'] = t.round(4).to_dict('records')
    else:
        print('Not run in this commit (RUN_MODEL_COMPARISON_BOOTSTRAP = False). Keep the earlier results.')
_run('10. Cross-model bootstrap, pairwise C-td differences', _boot)

def _default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

with open(OUT_DIR / 'paper_numbers.json', 'w') as f:
    json.dump(paper_numbers, f, indent=2, default=_default)
print('\nSaved:', OUT_DIR / 'paper_numbers.json')


In [ ]:
# ── 14.1b  New paper numbers (Nov 2026 additions) ─────────────────────────────
# Collects the CSVs written by 4.4, 10.8, 10.10, 12.6, and 13.6 into paper_numbers.json
# and prints the values the manuscript's FILL placeholders need.
import json
_new = {}
def _load(name):
    p = OUT_DIR / name
    return pd.read_csv(p) if p.exists() else None

rf = _load('review_followups.csv')            # 10.8: Table 2 (pi, coverage, misses, widths)
rc = _load('reviewer_checks.csv')             # 10.10: re-centered, IPCW pi, full CiPOT
gm = _load('grid_max_check.csv')              # 10.10: grid-maximum medians
lmk = _load('landmark_pi_by_tau.csv')         # 12.6
t3 = _load('table3_kfold_three_methods.csv')  # 13.6

if rf is not None:
    print('Misses pooled across models (Section 5.6):')
    print(rf.groupby(['alpha', 'method'])[['miss_below', 'miss_above']].sum().to_string())
    _new['table2'] = rf.to_dict('records')
if rc is not None:
    a20 = rc[rc.alpha == 0.20]
    def rng(method, col):
        v = a20[a20.method == method][col]
        return f'{v.min():.3f}-{v.max():.3f}'
    print('\nalpha = 0.20 ranges across models:')
    for m in ['Fixed width (Table 2)', 'Fixed width, re-centered', 'CSD', 'CiPOT (simplified)',
              'CiPOT (full, with censored)', 'Fixed width, excl. grid-max subjects']:
        print(f'  {m:<38} mean pi {rng(m, "mean_pi")} | IPCW pi {rng(m, "ipcw_mean_pi")} | '
              f'median width {rng(m, "median_width_yr")} yr')
    _new['reviewer_checks'] = rc.to_dict('records')
if gm is not None:
    print('\nGrid-maximum medians (Section 5.3):')
    print(gm.to_string(index=False))
    _new['grid_max'] = gm.to_dict('records')
if lmk is not None:
    _new['landmark_table'] = lmk.to_dict('records')
if t3 is not None:
    print('\nTable 3:')
    print(t3.to_string())
    _new['table3_three_methods'] = t3.to_dict('index')

paper_numbers['nov2026_additions'] = _new
with open(OUT_DIR / 'paper_numbers.json', 'w') as f:
    json.dump(paper_numbers, f, indent=2, default=str)
print('\nUpdated', OUT_DIR / 'paper_numbers.json')


In [ ]:
# ── 14.2  Final paper figures at 300 dpi (compact sizes) ─────────────────────
import traceback
from lifelines import KaplanMeierFitter

PAPER_DPI = 300
PAPER_FIG_DIR = FIG_DIR / 'paper'
PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
PAPER_YMAX = 25   # shared y-limit (years) for the interval figures; raise it to show more of the long AFT tails
MODEL_COLORS = {'AFT (Weibull)': '#3498db', 'Cox PH': '#e74c3c', 'GBSA': '#2ecc71', 'DeepSurv': '#9b59b6'}
RC = {'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5,
      'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.7}

def _save(fig, name):
    path = PAPER_FIG_DIR / name
    fig.savefig(path, dpi=PAPER_DPI, bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.show()
    plt.close(fig)
    print('saved', path)

def _try(label, fn):
    try:
        fn()
    except Exception as e:
        print(f'[{label} FAILED: {type(e).__name__}: {e}]')
        traceback.print_exc(limit=2)

# Figure 2: Kaplan-Meier, MCI to AD only, single column -------------------------
def fig_km():
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(3.4, 2.4))
        kmf = KaplanMeierFitter()
        kmf.fit(surv_mci_labels['duration'], event_observed=surv_mci_labels['event'], label='MCI to AD')
        kmf.plot_survival_function(ax=ax, color='#c0392b', ci_show=True, ci_alpha=0.15, lw=1.2)
        ax.axhline(0.5, color='gray', ls=':', lw=0.8)
        ax.set(xlabel='Years from baseline', ylabel='Survival probability')
        ax.legend(loc='upper right', frameon=False)
        _save(fig, 'KM_overall.png')
_try('Figure 2 (KM)', fig_km)

# Figure 3: individual survival curves, one subject, all models ------------------
def fig_survival(i=56):
    curves = [('GBSA', surv_gbsa_mci.iloc[:, i]), ('AFT (Weibull)', aft_surv_mci.iloc[:, i]),
              ('DeepSurv', surv_mci.iloc[:, i]), ('Cox PH', surv_cox_mci.iloc[:, i])]
    dur = float(y_dur_mci_test.iloc[i] if hasattr(y_dur_mci_test, 'iloc') else y_dur_mci_test[i])
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(5.2, 2.5))
        for name, s in curves:
            t = np.asarray(s.index, dtype=float)
            v = np.asarray(s.values, dtype=float)
            ax.step(t, v, where='post', color=MODEL_COLORS[name], lw=1.3,
                    label=f'{name}, S({dur:.2f}) = {v[min(np.searchsorted(t, dur), len(v) - 1)]:.3f}')
            ax.plot([dur], [v[min(np.searchsorted(t, dur), len(v) - 1)]], 'D', color=MODEL_COLORS[name],
                    ms=4, mec='black', mew=0.5, zorder=5)
        ax.axvline(dur, color='gray', ls='--', lw=0.8)
        ax.set(xlabel='Time (years)', ylabel='Survival probability S(t)', ylim=(0, 1.03))
        ax.legend(loc='upper right', frameon=False)
        _save(fig, 'All_model_survival_curves.png')
_try('Figure 3 (survival curves)', fig_survival)

# Figure 4: within-interval percentile, all models at alpha = 0.20, one 2x2 figure -
def fig_within():
    entries = [('Cox PH', results_cp_cox, ev_eval_cox, dur_eval_cox),
               ('AFT (Weibull)', results_cp_aft, ev_eval_aft, dur_eval_aft),
               ('GBSA', results_cp_gbsa, ev_eval_gbsa, dur_eval_gbsa),
               ('DeepSurv', results_cp_deepsurv, ev_eval_ds, dur_eval_ds)]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(2, 2, figsize=(7.0, 3.6), sharex=True, sharey=True)
        for ax, (name, R, ev, dur) in zip(axes.flat, entries):
            r = R[0.20]
            o = np.asarray(ev) == 1
            lo, hi, t = np.asarray(r['lo'])[o], np.asarray(r['hi'])[o], np.asarray(dur)[o]
            w = hi - lo
            ok = w > 0
            pi = np.clip((t[ok] - lo[ok]) / w[ok], 0, 1)
            ax.hist(pi, bins=10, range=(0, 1), density=True, color='#2980b9', alpha=0.8, edgecolor='white')
            ax.axhline(1.0, color='#e74c3c', ls='--', lw=1.2, label='Uniform (ideal)')
            ax.set_title(f'{name} (mean $\\pi$ = {pi.mean():.2f}, n = {ok.sum()})')
            ax.set_xlim(0, 1)
        axes[0, 0].legend(loc='upper right', frameon=False)
        for ax in axes[1, :]:
            ax.set_xlabel('Within-interval percentile (0 = lower bound, 1 = upper bound)')
        for ax in axes[:, 0]:
            ax.set_ylabel('Density')
        fig.tight_layout()
        _save(fig, 'within_interval_dist.png')
_try('Figure 4 (within-interval)', fig_within)

# Figure 5: feature importance by family ------------------------------------------
def fig_family():
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(6.0, 2.7))
        x = np.arange(len(family_pct.index))
        for k, model in enumerate(family_pct.columns):
            ax.bar(x + (k - 1.5) * 0.2, family_pct[model], 0.2, label=model, color=MODEL_COLORS.get(model, None))
        ax.set_xticks(x)
        ax.set_xticklabels(family_pct.index, rotation=15, ha='right')
        ax.set_ylabel("% of model's total SHAP mass")
        ax.legend(frameon=False)
        fig.tight_layout()
        _save(fig, 'feat_fam_import.png')
_try('Figure 5 (feature families)', fig_family)

# Figures 6 and 7: fixed width vs CSD intervals, compact --------------------------
def interval_fig(names, fname, ymax=PAPER_YMAX):
    cfgs = [c for c in model_configs if c[0] in names]
    with plt.rc_context(RC):
        fig, axes = plt.subplots(len(cfgs), 2, figsize=(7.0, 1.75 * len(cfgs) + 0.45),
                                 sharex=True, sharey=True, squeeze=False)
        for r, (name, lo_f, hi_f, dur_f, ev_f, lo_a, hi_a, mid_a, dur_a, ev_a) in enumerate(cfgs):
            for c, (lo, hi, key, dur, ev, label) in enumerate([
                    (lo_f, hi_f, (np.asarray(lo_f) + np.asarray(hi_f)) / 2, dur_f, ev_f, 'fixed width'),
                    (lo_a, hi_a, mid_a, dur_a, ev_a, 'CSD adaptive')]):
                ax = axes[r, c]
                lo, hi, dur, ev = map(np.asarray, (lo, hi, dur, ev))
                order = np.argsort(key)
                xs = np.arange(len(order))
                obs = ev[order] == 1
                cov_all = (dur[order] >= lo[order]) & (dur[order] <= hi[order])
                ax.vlines(xs[~obs], lo[order][~obs], hi[order][~obs], color='lightgray', lw=0.8,
                          label='Censored', zorder=1)
                ci = obs & cov_all
                ax.vlines(xs[ci], lo[order][ci], hi[order][ci], color='#2980b9', lw=1.0, zorder=2)
                ax.scatter(xs[ci], dur[order][ci], color='#2980b9', marker='D', s=9, label='Covered', zorder=3)
                nc = obs & ~cov_all
                ax.vlines(xs[nc], lo[order][nc], hi[order][nc], color='#c0392b', lw=1.0, zorder=2)
                ax.scatter(xs[nc], dur[order][nc], color='#c0392b', marker='x', s=12, label='Not covered', zorder=3)
                ax.set_title(f'{name}: {label} ({ci.sum() / obs.sum():.1%}, {ci.sum()}/{obs.sum()})')
                ax.set_ylim(0, ymax)
                n_clip = int(((hi[order][obs]) > ymax).sum())
                if n_clip:
                    print(f'  {name} {label}: {n_clip} observed-event upper bounds above {ymax} yr are clipped in the plot')
        axes[0, 0].legend(loc='upper left', frameon=False, ncol=1)
        for ax in axes[-1, :]:
            ax.set_xlabel('Subjects (sorted)')
        for ax in axes[:, 0]:
            ax.set_ylabel('Years from baseline')
        fig.tight_layout()
        _save(fig, fname)

_try('Figure 6', lambda: interval_fig(['AFT (Weibull)', 'Cox PH'], 'AFT_COX_CP_CSD_curves.png'))
_try('Figure 7', lambda: interval_fig(['GBSA', 'DeepSurv'], 'GBSA_DS_CP_CSD_curves.png'))
# One-model version in case AIIM has a main-text page limit:
_try('One-model figure', lambda: interval_fig(['Cox PH'], 'COX_CP_CSD_curves.png'))


In [ ]:
# ── 14.3  Zip the outputs so they are easy to download from the commit ────────
import zipfile

zip_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(OUT_DIR).parent
zip_path = zip_dir / 'paper_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (FIG_DIR, OUT_DIR):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(Path(folder).parent))
print('Wrote', zip_path, f'({zip_path.stat().st_size / 1e6:.1f} MB)')
print('Final figures are in', PAPER_FIG_DIR if 'PAPER_FIG_DIR' in globals() else FIG_DIR)
